# Data Visualisation

This notebook is the plotting entry point for the paper figures. It reads only `processed_data/`, defines `save_type_list = ["pdf", "eps", "png"]`, and writes each figure to `figures/pdf/`, `figures/eps/`, and `figures/png/`.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os

from slide.utils import get_processed_data_dir, get_figures_dir
from slide.utils import install_multi_format_savefig

PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
save_type_list = ["pdf", "eps", "png"]
for save_type in save_type_list:
    (FIGURES_DIR / save_type).mkdir(parents=True, exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt

_original_savefig = install_multi_format_savefig(plt, save_type_list)


In [ ]:
import pickle
import numpy as np
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
from slide.direvo_functions import *
from slide import selection_function_library as slct
from slide.ruggedness_functions import *
import os
import tqdm
import pandas as pd
import itertools
from matplotlib import rcParams

# Processed-data compatibility helpers. They allow the notebook to read the
# parameter-carrying dictionaries produced by data_processing.ipynb while still
# accepting older tuple-style processed pickles.
def unpack_popsize_accuracy(payload):
    if isinstance(payload, dict):
        return payload["rates"], payload["pop_sizes"]
    return payload


def unpack_mutation_accuracy(payload):
    if isinstance(payload, dict):
        return payload["rates"], payload["mutation_rates"]
    return payload


def unpack_optimal_strategies(payload):
    if isinstance(payload, dict):
        return payload["decay_rates"], payload["optimal_splits"], payload["optimal_base_chances"], payload.get("params", {})
    decay_rates, optimal_splits, optimal_base_chances = payload
    return decay_rates, optimal_splits, optimal_base_chances, {}


def unpack_strategy_spaces(payload):
    if isinstance(payload, dict):
        return payload["smooth"], payload["rugged"], payload.get("params", {})
    smooth_strategies, rugged_strategies = payload
    return smooth_strategies, rugged_strategies, {}


def unpack_strategy_selection(payload):
    if isinstance(payload, dict):
        return (
            payload.get("generations"),
            payload.get("decay_mean"),
            payload.get("decay_rate"),
            payload.get("sweep"),
            payload.get("scipy_freq_matrix"),
            payload.get("run"),
            payload.get("scatter"),
            payload.get("line"),
            payload.get("decay"),
            payload.get("strategy_params", {}),
        )
    if len(payload) == 9:
        return (*payload, {})
    if len(payload) == 6:
        x_vals, decay_mean, decay_rate, sweep, scipy_freq_matrix, run = payload
        return x_vals, decay_mean, decay_rate, sweep, scipy_freq_matrix, run, None, None, None, {}
    return (*payload, {})


### Notes on plot formatting

- Height 3
- DPI = 300
- Axes labels = 8, axes tick labels = 6
- Title = 10
- Heatmap: viridis, defined colours: see below

In [ ]:
## Plot colours

c2 = 'tab:blue'#298c8c'
c1 = 'tab:orange' #800074'
c3 = '#f55f74'
c4 = 'tab:green'

In [ ]:
## Fontsizes

titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
dpi = 350
plt.rcParams["font.family"] = "DejaVu Sans"

# Section 1 - Inferring ruggedness on NK

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).


### Example fitness decay curves

In [ ]:
with open('processed_data/smooth_rugged_example.pkl', 'rb') as f:
    smooth_rugged_payload = pickle.load(f)

if isinstance(smooth_rugged_payload, dict):
    smooth_rugged = smooth_rugged_payload["smooth_rugged"]
    fitted_lines = smooth_rugged_payload["fitted_lines"]
    decay_curve_labels = smooth_rugged_payload.get("labels", [r"Smooth", r"Rugged"])
    decay_curve_generations = smooth_rugged_payload.get("generations", np.arange(1, len(smooth_rugged[0]) + 1))
else:
    smooth_rugged, fitted_lines = smooth_rugged_payload
    decay_curve_labels = [r"$(K+1)/N = 0.1$", r"$(K+1)/N = 0.75$"]
    decay_curve_generations = np.arange(1, len(smooth_rugged[0]) + 1)


In [ ]:
plt.figure(figsize=(3.5,3), dpi=300)

colours = [c2, c1]
for curve, fitted_line, label, colour in zip(smooth_rugged, fitted_lines, decay_curve_labels, colours):
    plt.scatter(decay_curve_generations, curve, label=label, c=colour, s=5)
    plt.plot(decay_curve_generations, fitted_line, c=colour)
plt.legend(fontsize=legendsize)

plt.title('Fitness decay curves', fontsize=titlesize)
plt.xlabel('Generations $M$', fontsize=labelsize)
plt.ylabel(r'Fitness $F_\mu$', fontsize=labelsize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.savefig('figures/decay_curves_example.pdf', dpi=dpi)


### Ruggedness prediction accuracy over NK

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).


In [ ]:
with open('processed_data/ruggedness_accuracy.pkl', 'rb') as f:
    k_plus_one_over_ns, decay_rates = pickle.load(f)

In [ ]:
# Step 1: Get ordering for rho
arg_sort_kn = np.argsort(k_plus_one_over_ns)
sorted_rho = k_plus_one_over_ns[arg_sort_kn]
sorted_decay = decay_rates[arg_sort_kn]

# Step 2: Group the values
grouped_rho = sorted_rho.reshape(10,-1)
grouped_decay = sorted_decay.reshape(10,-1)

mean_rho = np.mean(grouped_rho, axis = 1)
mean_decay = np.mean(grouped_decay, axis = 1)
std_decay = np.std(grouped_decay, axis = 1)

true_k_over_n = np.linspace(0.1,1,10)

# Step 3: Create Fill-Between Plot
plt.figure(figsize=(3.5,3), dpi=300)
plt.plot(true_k_over_n, mean_decay, 'o-', label=r"Mean estimated $\rho$")  # Line plot with markers
plt.fill_between(true_k_over_n, mean_decay - std_decay, mean_decay + std_decay, alpha=0.3, label="±1 Std dev")  # Shaded error band

plt.plot(true_k_over_n, mean_rho, c='red', alpha=0.4,linestyle='--', label=r'$(K+1)/N$')

# Labels and Title
plt.xlabel(r"$(K+1)/N$", fontsize=labelsize)
plt.ylabel(r"Estimated $\rho$", fontsize=labelsize)
plt.title(r"$\rho$ prediction accuracy over ruggedness", fontsize = titlesize)
plt.legend(fontsize=legendsize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.grid(True)

# Show plot
plt.savefig('figures/accuracy_over_K.pdf', dpi=dpi)


### Ruggedness prediction accuracy over population size

The plotted axes are read from processed payload metadata when available.


In [ ]:
with open('processed_data/popsize_accuracy.pkl', 'rb') as f:
    popsize_decay_rates, pops = unpack_popsize_accuracy(pickle.load(f))


In [ ]:
y_means = popsize_decay_rates.mean(axis=1)
y_stds = popsize_decay_rates.std(axis=1)
pop_y_means = y_means
conv_p = 16/25
# Step 3: Create Fill-Between Plot
plt.figure(figsize=(3.5,1.2), dpi=300)
plt.plot(pops, y_means, 'o-', label=r"Mean estimated $\rho$")  # Line plot with markers
plt.axhline(y=conv_p, label=r'$(K+1)/N$', c='red', alpha=0.4, linestyle='--')
# plt.plot(pops, np.ones_like(pops) * 12/25, ls = '--', label="True") 
plt.fill_between(pops, y_means - y_stds, y_means + y_stds, alpha=0.3, label="±1 Std Dev")  # Shaded error band
plt.grid(True)
plt.legend(fontsize = legendsize-2, loc="upper right")
plt.tick_params(axis='both', which='major', labelsize=ticksize)

plt.title('Accuracy over population size', fontsize = titlesize)
plt.ylabel(r"Estimated $\rho$", fontsize=labelsize)
plt.xlabel('Population size', fontsize=labelsize)
plt.savefig('figures/accuracy_over_popsize.pdf', dpi=dpi)

### Ruggedness prediction accuracy over mutation rate

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).

The plotted axes are read from processed payload metadata when available.


In [ ]:
with open('processed_data/mut_accuracy.pkl', 'rb') as f:
    mut_decay_rates, muts = unpack_mutation_accuracy(pickle.load(f))


In [ ]:

y_means = np.mean(mut_decay_rates,axis=1)
y_stds = mut_decay_rates.std(axis=1)

# Step 3: Create Fill-Between Plot
plt.figure(figsize=(3.5,1.2), dpi=300)
plt.plot(muts, y_means, 'o-', label=r"Mean estimated $\rho$")  # Line plot with markers
plt.axhline(y=conv_p, label=r'$(K+1)/N$', c='red', alpha=0.4,linestyle='--')
plt.fill_between(muts, y_means - y_stds, y_means + y_stds, alpha=0.3, label="±1 Std Dev")  # Shaded error band
# plt.plot(muts, np.ones_like(muts) * 12/25, ls = '--', label="True") 
plt.grid(True)
plt.legend(fontsize = legendsize-2, loc='lower right')
plt.tick_params(axis='both', which='major', labelsize=ticksize)
# plt.ylim(0.0,1.0)
# plt.xlim(0,1.0)
plt.xlabel(r'Mutations per cell per generation $\theta$', fontsize=labelsize)
plt.ylabel(r"Estimated $\rho$", fontsize=labelsize)
plt.ylim(0.55,0.7)
plt.title('Accuracy over mutation rate', fontsize=titlesize)
plt.savefig('figures/accuracy_over_mut.pdf', dpi=dpi)


### Comparison of ruggedness metrics on NK

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).


In [ ]:
with open('processed_data/NK_ruggedness_metric_comparison.pkl', 'rb') as f:
    NK_roughness_to_slope, NK_fourier, convergence_rates, NK_paths_to_max, NK_closest_max, k_over_ns, NK_le_normed = pickle.load(f)

In [ ]:
plt.figure(figsize=(3.5,3), dpi=300)
plt.plot(k_over_ns, convergence_rates/convergence_rates.max(), label = r'$\rho$ (Decay rate)')
plt.plot(k_over_ns, 1 - NK_fourier/NK_fourier.max(), label = r'1 - Landscape ${R}^{2}$', alpha=0.6, linestyle='--')
plt.plot(k_over_ns, NK_roughness_to_slope/NK_roughness_to_slope.max(), label = 'Roughness to slope ratio', alpha=0.6, linestyle='--')
plt.plot(k_over_ns, 1-NK_paths_to_max/NK_paths_to_max.max(), label='1 - Paths to max', alpha=0.6, linestyle='--')
plt.plot(k_over_ns, NK_closest_max/NK_closest_max.max(), label='1 - Dist. to closest local max', alpha=0.6, linestyle='--')
plt.plot(k_over_ns, NK_le_normed/NK_paths_to_max.max(), label='Local Epistasis', alpha=0.6, linestyle='--')
plt.legend(loc = 'lower right', fontsize=legendsize-2)
plt.title('Ruggedness metric comparison', fontsize=titlesize)
plt.xlabel(r'$(K+1)/N$', fontsize = labelsize)
plt.ylabel('Normalised ruggedness measurements', fontsize = labelsize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.savefig('figures/NK_ruggedness_metric_comparison.pdf', dpi=dpi)

### Comparing landscapes with different ruggedness metrics

In [ ]:
with open('processed_data/empirical_ruggedness_metric_comparison.pkl', 'rb') as f:
    decay_rate_measurements, roughness_to_slope_measurements, landscape_r2_measurements, local_epistasis_measurements, paths_to_max_measurements, local_max_measurements = pickle.load(f) 

In [ ]:
with open('processed_data/GB1_strategy_selection.pkl', 'rb') as f:
    _, _, gb1_decay_rate, gb1_sweep, _, _, _, _, _, _ = unpack_strategy_selection(pickle.load(f))

with open('processed_data/TrpB_strategy_selection.pkl', 'rb') as f:
    _, _, trpb_decay_rate, trpb_sweep, _, _, _, _, _, _ = unpack_strategy_selection(pickle.load(f))

with open('processed_data/TEV_strategy_selection.pkl', 'rb') as f:
    _, _, tev_decay_rate, tev_sweep, _, _, _, _, _, _ = unpack_strategy_selection(pickle.load(f))

with open('processed_data/ParD3_strategy_selection.pkl', 'rb') as f:
    _, _, pard3_decay_rate, pard3_sweep, _, _, _, _, _, _ = unpack_strategy_selection(pickle.load(f))


In [ ]:
with open('landscape_arrays/GB1_landscape_array.pkl', 'rb') as f:
    GB1 = pickle.load(f)

with open('landscape_arrays/E3_landscape_array.pkl', 'rb') as f:
    ParD3 = pickle.load(f)

with open('landscape_arrays/TEV_landscape_array.pkl', 'rb') as f:
    TEV = pickle.load(f)

with open('landscape_arrays/TrpB_landscape_array.pkl', 'rb') as f:
    TrpB = pickle.load(f)

In [ ]:
from scipy.stats import percentileofscore
#evolvability = [percentileofscore(sweep.mean(axis=(0,3)).flatten(), sweep.mean(axis=(0,3))[-1,0]) for sweep in [gb1_sweep, trpb_sweep, tev_sweep, pard3_sweep]]
#evolvability = [sweep.mean(axis=(0,3))[-1,0]/sweep.mean(axis=(0,3))[0,0] for sweep in [gb1_sweep, trpb_sweep, tev_sweep, pard3_sweep]]

def normalise_array(x):
    x = np.asarray(x, dtype=float)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)  # add epsilon to avoid div/0

evolvability = [normalise_array(sweep.mean(axis=(0,3)))[-1,0]/normalise_array(sweep.mean(axis=(0,3)))[0,-1] for sweep in [gb1_sweep, trpb_sweep, tev_sweep, pard3_sweep]]
#evolvability = [sweep.mean(axis=(0,3))[-1,0] for sweep in [gb1_sweep, trpb_sweep, tev_sweep, pard3_sweep]]
#evolvability = [i/ld.max() for i, ld in zip(evolvability, [GB1, TrpB, TEV, ParD3])]
#evolvability = [percentileofscore(ld.flatten(), i) for i, ld in zip(evolvability, [GB1, TrpB, TEV, ParD3])]


In [ ]:
decay_rate_measurements = [i[0]/2 for i in [gb1_decay_rate, trpb_decay_rate, tev_decay_rate, pard3_decay_rate]]

In [ ]:
local_epistasis_measurements = [i['simple_sign_episasis']+ i['reciprocal_sign_epistasis'] for i in local_epistasis_measurements]

In [ ]:
arr = np.array([decay_rate_measurements, 1-np.array(landscape_r2_measurements), local_epistasis_measurements,np.array(local_max_measurements),np.array(paths_to_max_measurements), roughness_to_slope_measurements])

In [ ]:
def r_sigfig(value, sigfig=1):
    if value == 0:  
        return 0  # Special case: Zero remains zero
    
    return np.round(value, -int(np.floor(np.log10(abs(value)))) + (sigfig - 1))

In [ ]:
colours = [c1, c2, c4, c3]
labels = ['GB1', 'TrpB', 'TEV', 'ParD3']
rank_labels = ["1st", "2nd", "3rd", "4th"]
reversed_axes = {1, 3, 4}  # these get lowest->"1st"
titles = [r'Decay rate $\rho$', r'Landscape $R^2$', 'Local epistasis', 'Dist. to local max', 'Paths to max', 'Roughness to slope']

fig, axes = plt.subplots(1, 6, figsize=(8, 2), dpi=300)

for i, ax in enumerate(axes):

    y_positions = arr[i]                 # y-values for this metric (length 4)
    x_positions = np.full(4, 0.5)        # 4 points on x=0.5
    
    # Choose ranking order per axis
    order = np.argsort(y_positions) if i in reversed_axes else np.argsort(-y_positions)
    # Map each point j -> its rank index (0..3)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(len(y_positions))

    for j in range(4):
        jitter = np.random.uniform(-0.05, 0.05)
        x = x_positions[j] + jitter
        y = y_positions[j]

        ax.scatter(x, y,
                   color=colours[j], s=40, edgecolors='black', zorder=3,
                   label=labels[j] if i == 0 else None)
        # Rank label
        ax.text(x + 0.07, y, rank_labels[ranks[j]],
                fontsize=5, va='center', ha='left', zorder=4)

    # Outline
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1)

    # Axes ticks
    ax.set_xticks([])
    ax.set_yticks([arr[i].min(), arr[i].max()])
    ax.set_yticklabels([r_sigfig(arr[i].min()), r_sigfig(arr[i].max())], fontsize=5)
    ax.set_xlim(0, 1)
    ax.margins(y=0.1)

    ax.set_title(titles[i], fontsize=titlesize-4.5)

# Legend
axes[0].legend(fontsize=legendsize, loc='upper left', bbox_to_anchor=(8.7, 0.8))
axes[1].set_yticklabels([r_sigfig(arr[1].min()), r_sigfig(arr[1].max())], fontsize=4)

# Invert y-axis where needed
for idx in [1, 3, 4]:
    axes[idx].invert_yaxis()

# Adjust spacing
plt.subplots_adjust(wspace=0.5)

# Global label
fig.text(0.08, 0.5, r'Ruggedness (a.u.) $\rightarrow$', fontsize=8, va='center', rotation=90)

plt.savefig('figures/empirical_ruggedness_metric_comparison.pdf', dpi=dpi)

In [ ]:
with open('processed_data/fourier_spectra_empirical.pkl', 'rb') as f:
    spectra = pickle.load(f)

In [ ]:
colours = [c1, c2, c4, c3]
labels = ['GB1', 'TrpB', 'TEV', 'ParD3']
markers = ['o', 's', '^', 'D']

plt.figure(figsize=(3.5, 3), dpi=300)
A = 20
empirical_dimensions = np.array([4, 4, 4, 3])
fourier_vals = []

for n, spectrum in enumerate(spectra):
    spectrum = np.asarray(spectrum)[1:]
    spectrum_norm = (spectrum - spectrum.min()) / (spectrum.max() - spectrum.min())
    indexes = np.arange(len(spectrum)) + 1
    d = empirical_dimensions[n] * (A - 1)
    weighted_avg = np.sum(A * indexes * spectrum) / np.sum(spectrum) / d
    frequency_centroid = weighted_avg * d / A
    plt.plot(
        indexes,
        spectrum_norm,
        label=labels[n],
        c=colours[n],
        marker=markers[n],
        markersize=4,
    )
    plt.axvline(x=frequency_centroid, c=colours[n], linestyle='--', alpha=0.5)
    fourier_vals.append(weighted_avg)

plt.legend(fontsize=legendsize, loc='upper right')
plt.title('Empirical landscape fourier spectra', fontsize=titlesize)
plt.xlabel(r'Frequency index $i$', fontsize=labelsize)
plt.ylabel(r'Power spectral coefficient $b_i$', fontsize=labelsize)
plt.xticks(([1, 2, 3, 4]))
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.savefig('figures/empirical_fourier_spectra.pdf', dpi=dpi)


In [ ]:
with open('processed_data/trajectory_subsampling.pkl', 'rb') as f:
    ld_results = pickle.load(f)

In [ ]:
len(ld_results)

In [ ]:
[[x.shape for x in k] for k in ld_results]

In [ ]:
data_names = ['GB1', 'TrpB', 'TEV', 'ParD3']
colours = [c1,c2,c4,c3]
trajectories = np.round(np.logspace(0, np.log10(160000), 11)).astype(int)
plt.figure(figsize=(3.5,3), dpi=300)

for h in range(3):

    means = []
    errs = []

    for t in range(len(trajectories)):
        vals = ld_results[h][t]     # (n_boot,)
        means.append(np.mean(vals))
        errs.append(np.std(vals))   # or SEM: np.std(vals) / np.sqrt(len(vals))

    means = np.array(means)
    errs  = np.array(errs)

    plt.plot(
        trajectories,
        means,
        label=data_names[h],
        color = colours[h]
    )

    plt.fill_between(
        trajectories,
        means - errs,
        means + errs,
        alpha=0.25,
        color = colours[h],
        edgecolor=None
    )

    #plt.hlines(
    #    y=fourier_vals[h] xmin=0, xmax=1000, color=colours[h], linestyle='--', alpha=0.4)
    
    plt.hlines(
        y=fourier_vals[h], xmin=0, xmax=160000, color=colours[h], linestyle='dotted',alpha=0.4)


trajectories = np.round(np.logspace(0, np.log10(8000), 11)).astype(int)

means = []
errs = []

for t in range(len(trajectories)):
    vals = ld_results[-1][t]     # (n_boot,)
    means.append(np.mean(vals))
    errs.append(np.std(vals))   # or SEM: np.std(vals) / np.sqrt(len(vals))

means = np.array(means)
errs  = np.array(errs)

plt.plot(
    trajectories,
    means,
    label=data_names[-1],
    color = colours[-1]
)

plt.fill_between(
    trajectories,
    means - errs,
    means + errs,
    alpha=0.25,
    color = colours[-1],
    edgecolor=None
)

#plt.hlines(
#    y=fourier_vals[h] xmin=0, xmax=1000, color=colours[h], linestyle='--', alpha=0.4)

plt.hlines(
    y=fourier_vals[-1], xmin=0, xmax=8000, color=colours[-1], linestyle='dotted',alpha=0.4)

plt.xlabel('Number of trajectories averaged', fontsize=labelsize)
plt.ylabel(r'Decay rate $\rho$', fontsize=labelsize)
plt.title(r'$\rho$ accuracy with increasing samples', fontsize=titlesize)
plt.legend(fontsize = legendsize)
plt.ylim(0,1.8)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.xscale('log')
plt.savefig('figures/accuracy_over_sampling.pdf', dpi=dpi)

In [ ]:
with open('processed_data/heterogeneity_data.pkl', 'rb') as f:
    NK_rhos, empirical_rhos = pickle.load(f)

In [ ]:
empirical_rhos = [np.clip(i,0,1) for i in empirical_rhos]
NK_rhos = [np.clip(i,0,1) for i in NK_rhos]

In [ ]:
data = NK_rhos + empirical_rhos

In [ ]:
means = [np.mean(d) for d in data]
stds  = [np.std(d) for d in data]

plt.figure(figsize=(3.5,3), dpi=300)

plt.errorbar(
    range(1, 9),
    means,
    yerr=stds,
    fmt='o',
    capsize=3,
)

plt.xticks(
    range(1, 9),
    ['0.25', '0.50', '0.75', '1.00', 'GB1', 'TrpB', 'TEV', 'PardD3'],
    fontsize=ticksize
)

plt.title('Landscape heterogeneity')
plt.ylabel(r'$\rho$ Estimation', fontsize=labelsize)
plt.ylim(-0.1, 1.15)
plt.tight_layout()
plt.savefig('figures/landscape_heterogeneity.pdf', dpi=dpi)


In [ ]:
plt.figure(figsize=(3.5, 3), dpi=300)

data = NK_rhos + empirical_rhos
var_data = [np.std(i) for i in data]
print(var_data)

vp = plt.violinplot(
    data,
    showmeans=True,
    showextrema=True   # make sure extrema are enabled
)

# Make the internal vertical bars thinner
vp['cbars'].set_linewidth(0.5)
vp['cmins'].set_linewidth(0.5)
vp['cmaxes'].set_linewidth(0.5)
vp['cmeans'].set_linewidth(0.5)

plt.xticks(
    [1, 2, 3, 4, 5, 6, 7, 8],
    ['0.25', '0.50', '0.75', '1.00', 'GB1', 'TrpB', 'TEV', 'PardD3'],
    fontsize=ticksize
)

plt.ylabel(r'$\rho$ Estimation', fontsize=labelsize)
plt.title('Landscape heterogeneity', fontsize=titlesize)

for i, var in enumerate(var_data):
    x = i + 1
    y = max(data[i]) * 1.05
    plt.text(
        x, y,
        f'$\sigma$={var:.2f}',
        ha='center',
        va='bottom',
        fontsize=4.5
    )

plt.axvline(x=4.5, color='gray', linestyle='--', linewidth=0.5)

plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.ylim(-0.1, 1.15)

plt.tight_layout()
plt.savefig('figures/landscape_heterogeneity.pdf', dpi=dpi)


In [ ]:
plt.figure(figsize=(3.5, 3), dpi=300)

data = NK_rhos + empirical_rhos
var_data = [np.std(i) for i in data]

plt.violinplot(data, showmeans=True)

plt.xticks([1, 2, 3,4,5,6,7,8], ['0.25', '0.50', '0.75','1.00','GB1','TrpB','TEV', 'PardD3'], fontsize=ticksize)
plt.ylabel(r'$\rho$ Estimation', fontsize = labelsize)
plt.title('Landscape heterogeneity', fontsize=titlesize)

for i, var in enumerate(var_data):
    x = i + 1  # violin positions start at 1
    y = max(data[i]) * 1.05  # slightly above the top
    plt.text(x, y, f'$\sigma$={var:.2f}', ha='center', va='bottom', fontsize=4.5)

plt.axvline(x=4.5, color='gray', linestyle='--', linewidth=1)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.ylim(-0.1,1.15)
plt.tight_layout()
plt.savefig('figures/landscape_heterogeneity.pdf', dpi=dpi)

# Section 3 - Optimising directed evolution

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


### Optimal DE strategies from sweep

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
with open('processed_data/optimal_DE_strategies.pkl', 'rb') as f:
    decay_rates, optimal_splits, optimal_base_chances, optimal_strategy_params = unpack_optimal_strategies(pickle.load(f))


In [ ]:
with open('processed_data/strategy_prediction_accuracy.pkl', 'rb') as f:
    actual_k_over_ns, bc_means, bc_stds, sp_means, sp_stds = pickle.load(f)

In [ ]:
fig, ax1 = plt.subplots(figsize=(3.5,3), dpi=300)
strategy_total_popsize = optimal_strategy_params.get("popsize", 1200)

k_over_ns = np.unique(np.round(actual_k_over_ns,1))

color = c1
ax1.set_xlabel(r'True $\rho$', fontsize=8)
ax1.set_ylabel(r'Predicted base chance $b$', color=color, fontsize=8)
ax1.errorbar(k_over_ns, bc_means, yerr=bc_stds, fmt='o', capsize=5, color=c1)
ax1.plot(decay_rates, optimal_base_chances, color=color, linestyle='--', alpha=0.6, label='Optimal base chance')
ax1.tick_params(axis='y', labelcolor=color, labelsize=6)
ax1.tick_params(axis='x', labelsize=6)
ax1.grid(color='black', linestyle='--', linewidth=0.5, alpha=0.3)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = c2
ax2.set_ylabel(f'Predicted splitting (total {strategy_total_popsize})', color=color, fontsize=8) 
ax2.errorbar(k_over_ns, sp_means, yerr=sp_stds, fmt='o', capsize=5, color=c2)
ax2.plot(decay_rates, optimal_splits, color=color, linestyle='--', alpha=0.6, label='Optimal splitting')
ax2.tick_params(axis='y', labelcolor=color, labelsize=6)

ax1.set_title(r'Predicted DE strategies vs true $\rho$', fontsize=10)

# Adjust legend positions
ax1.legend(fontsize=legendsize-2, loc='upper left', bbox_to_anchor=(0.0, 1.0))
ax2.legend(fontsize=legendsize-2, loc='upper left', bbox_to_anchor=(0.0, 0.9))  # Move slightly below ax1's legend

plt.xlim(0.05, 1.05)
fig.tight_layout()
plt.savefig('figures/strategy_prediction.pdf', dpi=dpi)


### Directed evolution on smooth NK

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
with open('processed_data/NK_DE.pkl', 'rb') as f:
    DE_data = pickle.load(f)

with open('processed_data/NK_strategy_spaces.pkl', 'rb') as f:
    smooth_strategies, rugged_strategies, nk_strategy_space_params = unpack_strategy_spaces(pickle.load(f))


In [ ]:
plt.figure(figsize=(3,3), dpi=300)
plt.plot(DE_data[0], label = 'Baseline strategy', color=c1)
plt.plot(DE_data[2], label = 'SLIDE', color=c2)
plt.ylabel('Arbitrary fitness scale', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title('N = 45, K = 1',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
plt.savefig('figures/N45K1_DE_fitness.pdf', dpi=dpi)

In [ ]:
strategy_base_chances = np.asarray(nk_strategy_space_params.get("base_chances", [0.0, 0.19]))
strategy_splits = list(nk_strategy_space_params.get("splits", [24, 20, 16, 12, 8, 4, 1]))
plt.figure(figsize = (1.5,1.5), dpi=300)
plt.imshow(smooth_strategies.mean(axis=1).reshape(7,7))
plt.xticks([0, len(strategy_base_chances) - 1], labels=[float(strategy_base_chances[0]), float(strategy_base_chances[-1])])
plt.yticks([0, len(strategy_splits) - 1], labels=[strategy_splits[0], strategy_splits[-1]])
plt.ylabel('No. sub populations', fontsize=8)
plt.xlabel('Base chance', fontsize=8)
plt.title('Strategy space')
plt.tick_params(axis='both', which='major', labelsize=6)
plt.savefig('figures/N45K1_DE_strategy_space.pdf', dpi=dpi)


### Directed evolution on rough NK

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
plt.figure(figsize=(3,3), dpi=300)
plt.plot(DE_data[1], label = 'Baseline strategy', color=c1)
plt.plot(DE_data[3], label = 'SLIDE', color=c2)
plt.ylabel('Arbitrary fitness scale', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title('N = 45, K = 25',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
plt.savefig('figures/N45K25_DE_fitness.pdf', dpi=dpi)

In [ ]:
strategy_base_chances = np.asarray(nk_strategy_space_params.get("base_chances", [0.0, 0.19]))
strategy_splits = list(nk_strategy_space_params.get("splits", [24, 20, 16, 12, 8, 4, 1]))
plt.figure(figsize = (1.5,1.5), dpi=300)
plt.imshow(rugged_strategies.mean(axis=1).reshape(7,7))
plt.xticks([0, len(strategy_base_chances) - 1], labels=[float(strategy_base_chances[0]), float(strategy_base_chances[-1])])
plt.yticks([0, len(strategy_splits) - 1], labels=[strategy_splits[0], strategy_splits[-1]])
plt.ylabel('No. sub populations', fontsize=8)
plt.xlabel('Base chance', fontsize=8)
plt.title('Strategy space')
plt.tick_params(axis='both', which='major', labelsize=6)
plt.savefig('figures/N45K25_DE_strategy_space.pdf', dpi=dpi)


### GB1 directed evolution

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
def normalise_decay(y_vals, constant):
    out = y_vals - constant
    return out/out[0]

In [ ]:
with open('processed_data/GB1_strategy_selection.pkl', 'rb') as f:
    x_vals, decay_mean, decay_rate, sweep, scipy_freq_matrix, run, scatter, line, GB1_decay_multi, strategy_selection_params = unpack_strategy_selection(pickle.load(f))


In [ ]:
plt.figure(figsize=(3, 1.5), dpi=300)
plt.scatter(x_vals, scatter, label='Mean fitness', s=15)
plt.plot(x_vals, line, label='Fit')
plt.ylabel('Fitness relative to WT', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title(f'GB1 fitness decay, $\\rho$ = {np.around(decay_rate[0]/2,2)}',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
plt.ylim(0,1.1)
plt.show()

plt.savefig('figures/GB1_decay.pdf', dpi=dpi)


In [ ]:
plt.figure(figsize=(1.5, 1.5), dpi=300)
mean_sweep = sweep.mean(axis=(0,3))
plt.imshow(mean_sweep)

# Add scatter plot with square markers proportional to frequency
max_size = 100  # Adjust as needed
dot_sizes = (scipy_freq_matrix / scipy_freq_matrix.max()) * max_size

for i in range(7):  # Loop over rows
    for j in range(7):  # Loop over columns
        if scipy_freq_matrix[i, j] > 0:  # Only plot if frequency > 0
            plt.scatter(
                j, i,
                s=dot_sizes[i, j]*0.2,
                color=c2,
                alpha=1,
                marker='o',
                label='SLIDE'
            )

# Baseline as a square
plt.scatter(0,6, s=20, color=c1, alpha=1, marker='o', label='Baseline')

# ---- Optimal strategy as a red square ----
max_idx = np.unravel_index(np.argmax(mean_sweep, axis=None), mean_sweep.shape)
optimal_i, optimal_j = max_idx
plt.scatter(
    optimal_j, optimal_i,
    s=30,              # slightly larger to stand out
    color='red',
    alpha=1,
    marker='o',
    linewidth=0.5,
    label='Optimum'
)

# Formatting
plt.xticks([0, 6], labels=[0.0, 0.19])
plt.yticks([0, 6], labels=[24, 1])
plt.ylabel('No. sub populations', fontsize=labelsize - 1)
plt.xlabel('Base chance', fontsize=labelsize - 1)
plt.title('Relative strategy performance', fontsize=titlesize - 2)
plt.tick_params(axis='both', which='major', labelsize=5)

# Legend outside, smaller
plt.legend(
    fontsize=6,
    loc='center left',
    bbox_to_anchor=(1.07, 0.5)
)

plt.savefig('figures/GB1_strategy_space.pdf', dpi=dpi, bbox_inches='tight')
plt.show()


In [ ]:
# --- Compute mean over reps first ---
run_startmean = [
    [
        np.mean(strategy, axis=0)  # mean over reps for each strategy
        for strategy in start
    ]
    for start in run
]

n_strat = len(run_startmean[0])
n_steps = max(len(start[0]) for start in run_startmean)  # max number of steps across starts

strategy_mean = []
strategy_std  = []

# --- Compute mean/std across starts ---
for s in range(n_strat):
    # Gather the mean-over-reps trajectories for this strategy
    start_vals = [start[s] for start in run_startmean]  # list of arrays (steps)
    
    # Pad sequences to the same length with NaN (optional)
    max_len = max(len(traj) for traj in start_vals)
    start_vals_pad = [list(traj) + [np.nan]*(max_len - len(traj)) for traj in start_vals]

    # Compute mean and std at each step, ignoring NaNs
    mean_vals = [np.nanmean([traj[step] for traj in start_vals_pad]) for step in range(max_len)]
    std_vals  = [np.nanstd([traj[step] for traj in start_vals_pad], ddof=1) for step in range(max_len)]
    
    strategy_mean.append(mean_vals)
    strategy_std.append(std_vals)

# --- Plotting ---
plt.figure(figsize=(4, 1.5), dpi=300)
steps = np.arange(n_steps)
colors = [c1, c2]
labels = ['Baseline', 'SLIDE']

for s in range(n_strat):
    plt.plot(steps, strategy_mean[s], label=labels[s], c=colors[s])
    plt.fill_between(
        steps,
        [m - std for m, std in zip(strategy_mean[s], strategy_std[s])],
        [m + std for m, std in zip(strategy_mean[s], strategy_std[s])],
        alpha=0.3,
        color=colors[s],
        linewidth=0
    )

plt.ylabel('Fitness relative to WT', fontsize=labelsize)
plt.xlabel(r'Generations $M$', fontsize=labelsize)
plt.title('GB1 directed evolution, 10 start average', fontsize=titlesize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.legend(fontsize=8)
plt.savefig('figures/GB1_DE.pdf', dpi=dpi)
plt.show()


In [ ]:
from scipy.stats import ttest_ind

baseline_final = []
slide_final = []

for start in run_startmean:
    # Strategy 0 = Baseline
    traj0 = start[0]
    if len(traj0) > 0:
        val0 = np.mean(traj0[-1]) if hasattr(traj0[-1], '__iter__') else traj0[-1]
        baseline_final.append(val0)

    # Strategy 1 = SLIDE
    traj1 = start[1]
    if len(traj1) > 0:
        val1 = np.mean(traj1[-1]) if hasattr(traj1[-1], '__iter__') else traj1[-1]
        slide_final.append(val1)

# Convert to 1D float arrays
baseline_final = np.array(baseline_final, dtype=float)
slide_final    = np.array(slide_final, dtype=float)

# Perform Welch's t-test
t_stat, p_value = ttest_ind(baseline_final, slide_final, equal_var=False)

print(f"Final generation comparison:")
print(f"Baseline mean = {baseline_final.mean():.4f}, SLIDE mean = {slide_final.mean():.4f}")
print(f"t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")



### TrpB Directed Evolution

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
with open('processed_data/TrpB_strategy_selection.pkl', 'rb') as f:
    x_vals, decay_mean, decay_rate, sweep, scipy_freq_matrix, run, scatter, line, TrpB_decay_multi, strategy_selection_params = unpack_strategy_selection(pickle.load(f))


In [ ]:
plt.figure(figsize=(3, 1.5), dpi=300)
plt.scatter(x_vals, scatter, label='Mean fitness', s=15)
plt.plot(x_vals, line, label='Fit')
plt.ylabel('Fitness relative to WT', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title(f'TrpB fitness decay, $\\rho$ = {np.around(decay_rate[0]/2,2)}',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
plt.ylim(0,1.1)
plt.show()

plt.savefig('figures/TrpB_decay.pdf', dpi=dpi)


In [ ]:
plt.figure(figsize=(1.5, 1.5), dpi=300)
mean_sweep = sweep.mean(axis=(0,3))
plt.imshow(mean_sweep)

# Add scatter plot with square markers proportional to frequency
max_size = 100  # Adjust as needed
dot_sizes = (scipy_freq_matrix / scipy_freq_matrix.max()) * max_size

for i in range(7):  # Loop over rows
    for j in range(7):  # Loop over columns
        if scipy_freq_matrix[i, j] > 0:  # Only plot if frequency > 0
            plt.scatter(
                j, i,
                s=dot_sizes[i, j]*0.2,
                color=c2,
                alpha=1,
                marker='o',
                label='SLIDE'
            )

# Baseline as a square
plt.scatter(0,6, s=20, color=c1, alpha=1, marker='o', label='Baseline')

# ---- Optimal strategy as a red square ----
max_idx = np.unravel_index(np.argmax(mean_sweep, axis=None), mean_sweep.shape)
optimal_i, optimal_j = max_idx
plt.scatter(
    optimal_j, optimal_i,
    s=30,              # slightly larger to stand out
    color='red',
    alpha=1,
    marker='o',
    linewidth=0.5,
    label='Optimum'
)

# Formatting
plt.xticks([0, 6], labels=[0.0, 0.19])
plt.yticks([0, 6], labels=[24, 1])
plt.ylabel('No. sub populations', fontsize=labelsize - 1)
plt.xlabel('Base chance', fontsize=labelsize - 1)
plt.title('Relative strategy performance', fontsize=titlesize - 2)
plt.tick_params(axis='both', which='major', labelsize=5)

# Legend outside, smaller
plt.legend(
    fontsize=6,
    loc='center left',
    bbox_to_anchor=(1.07, 0.5)
)

plt.savefig('figures/TrpB_strategy_space.pdf', dpi=dpi, bbox_inches='tight')
plt.show()


In [ ]:
# --- Compute mean over reps first ---
run_startmean = [
    [
        np.mean(strategy, axis=0)  # mean over reps for each strategy
        for strategy in start
    ]
    for start in run
]

n_strat = len(run_startmean[0])
n_steps = max(len(start[0]) for start in run_startmean)  # max number of steps across starts

strategy_mean = []
strategy_std  = []

# --- Compute mean/std across starts ---
for s in range(n_strat):
    # Gather the mean-over-reps trajectories for this strategy
    start_vals = [start[s] for start in run_startmean]  # list of arrays (steps)
    
    # Pad sequences to the same length with NaN (optional)
    max_len = max(len(traj) for traj in start_vals)
    start_vals_pad = [list(traj) + [np.nan]*(max_len - len(traj)) for traj in start_vals]

    # Compute mean and std at each step, ignoring NaNs
    mean_vals = [np.nanmean([traj[step] for traj in start_vals_pad]) for step in range(max_len)]
    std_vals  = [np.nanstd([traj[step] for traj in start_vals_pad], ddof=1) for step in range(max_len)]
    
    strategy_mean.append(mean_vals)
    strategy_std.append(std_vals)

# --- Plotting ---
plt.figure(figsize=(4, 1.5), dpi=300)
steps = np.arange(n_steps)
colors = [c1, c2]
labels = ['Baseline', 'SLIDE']

for s in range(n_strat):
    plt.plot(steps, strategy_mean[s], label=labels[s], c=colors[s])
    plt.fill_between(
        steps,
        [m - std for m, std in zip(strategy_mean[s], strategy_std[s])],
        [m + std for m, std in zip(strategy_mean[s], strategy_std[s])],
        alpha=0.3,
        color=colors[s],
        linewidth=0
    )

plt.ylabel('Fitness relative to WT', fontsize=labelsize)
plt.xlabel(r'Generations $M$', fontsize=labelsize)
plt.title('TrpB directed evolution, 10 start average', fontsize=titlesize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.legend(fontsize=8)
plt.savefig('figures/TrpB_DE.pdf', dpi=dpi)
plt.show()


In [ ]:
from scipy.stats import ttest_ind

baseline_final = []
slide_final = []

for start in run_startmean:
    # Strategy 0 = Baseline
    traj0 = start[0]
    if len(traj0) > 0:
        val0 = np.mean(traj0[-1]) if hasattr(traj0[-1], '__iter__') else traj0[-1]
        baseline_final.append(val0)

    # Strategy 1 = SLIDE
    traj1 = start[1]
    if len(traj1) > 0:
        val1 = np.mean(traj1[-1]) if hasattr(traj1[-1], '__iter__') else traj1[-1]
        slide_final.append(val1)

# Convert to 1D float arrays
baseline_final = np.array(baseline_final, dtype=float)
slide_final    = np.array(slide_final, dtype=float)

# Perform Welch's t-test
t_stat, p_value = ttest_ind(baseline_final, slide_final, equal_var=False)

print(f"Final generation comparison:")
print(f"Baseline mean = {baseline_final.mean():.4f}, SLIDE mean = {slide_final.mean():.4f}")
print(f"t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

### TEV Directed Evolution

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
with open('processed_data/TEV_strategy_selection.pkl', 'rb') as f:
    x_vals, decay_mean, decay_rate, sweep, scipy_freq_matrix, run, scatter, line, TEV_decay_multi, strategy_selection_params = unpack_strategy_selection(pickle.load(f))


In [ ]:
plt.figure(figsize=(3, 1.5), dpi=300)
plt.scatter(x_vals, scatter, label='Mean fitness', s=15)
plt.plot(x_vals, line, label='Fit')
plt.ylabel('Fitness relative to WT', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title(f'TEV fitness decay, $\\rho$ = {np.around(decay_rate[0]/2,2)}',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
plt.ylim(0,1.1)
plt.show()

plt.savefig('figures/TEV_decay.pdf', dpi=dpi)

In [ ]:
plt.figure(figsize=(1.5, 1.5), dpi=300)
mean_sweep = sweep.mean(axis=(0,3))
plt.imshow(mean_sweep)

# Add scatter plot with square markers proportional to frequency
max_size = 100  # Adjust as needed
dot_sizes = (scipy_freq_matrix / scipy_freq_matrix.max()) * max_size

for i in range(7):  # Loop over rows
    for j in range(7):  # Loop over columns
        if scipy_freq_matrix[i, j] > 0:  # Only plot if frequency > 0
            plt.scatter(
                j, i,
                s=dot_sizes[i, j]*0.2,
                color=c2,
                alpha=1,
                marker='o',
                label='SLIDE'
            )

# Baseline as a square
plt.scatter(0,6, s=20, color=c1, alpha=1, marker='o', label='Baseline')

# ---- Optimal strategy as a red square ----
max_idx = np.unravel_index(np.argmax(mean_sweep, axis=None), mean_sweep.shape)
optimal_i, optimal_j = max_idx
plt.scatter(
    optimal_j, optimal_i,
    s=30,              # slightly larger to stand out
    color='red',
    alpha=1,
    marker='o',
    linewidth=0.5,
    label='Optimum'
)

# Formatting
plt.xticks([0, 6], labels=[0.0, 0.19])
plt.yticks([0, 6], labels=[24, 1])
plt.ylabel('No. sub populations', fontsize=labelsize - 1)
plt.xlabel('Base chance', fontsize=labelsize - 1)
plt.title('Relative strategy performance', fontsize=titlesize - 2)
plt.tick_params(axis='both', which='major', labelsize=5)

# Legend outside, smaller
plt.legend(
    fontsize=6,
    loc='center left',
    bbox_to_anchor=(1.07, 0.5)
)

plt.savefig('figures/TEV_strategy_space.pdf', dpi=dpi, bbox_inches='tight')
plt.show()


In [ ]:
# --- Compute mean over reps first ---
run_startmean = [
    [
        np.mean(strategy, axis=0)  # mean over reps for each strategy
        for strategy in start
    ]
    for start in run
]

n_strat = len(run_startmean[0])
n_steps = max(len(start[0]) for start in run_startmean)  # max number of steps across starts

strategy_mean = []
strategy_std  = []

# --- Compute mean/std across starts ---
for s in range(n_strat):
    # Gather the mean-over-reps trajectories for this strategy
    start_vals = [start[s] for start in run_startmean]  # list of arrays (steps)
    
    # Pad sequences to the same length with NaN (optional)
    max_len = max(len(traj) for traj in start_vals)
    start_vals_pad = [list(traj) + [np.nan]*(max_len - len(traj)) for traj in start_vals]

    # Compute mean and std at each step, ignoring NaNs
    mean_vals = [np.nanmean([traj[step] for traj in start_vals_pad]) for step in range(max_len)]
    std_vals  = [np.nanstd([traj[step] for traj in start_vals_pad], ddof=1) for step in range(max_len)]
    
    strategy_mean.append(mean_vals)
    strategy_std.append(std_vals)

# --- Plotting ---
plt.figure(figsize=(4, 1.5), dpi=300)
steps = np.arange(n_steps)
colors = [c1, c2]
labels = ['Baseline', 'SLIDE']

for s in range(n_strat):
    plt.plot(steps, strategy_mean[s], label=labels[s], c=colors[s])
    plt.fill_between(
        steps,
        [m - std for m, std in zip(strategy_mean[s], strategy_std[s])],
        [m + std for m, std in zip(strategy_mean[s], strategy_std[s])],
        alpha=0.3,
        color=colors[s],
        linewidth=0
    )

plt.ylabel('Fitness relative to WT', fontsize=labelsize)
plt.xlabel(r'Generations $M$', fontsize=labelsize)
plt.title('TrpB directed evolution, 10 start average', fontsize=titlesize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.legend(fontsize=8)
plt.savefig('figures/TEV_DE.pdf', dpi=dpi)
plt.show()


In [ ]:
from scipy.stats import ttest_ind

baseline_final = []
slide_final = []

for start in run_startmean:
    # Strategy 0 = Baseline
    traj0 = start[0]
    if len(traj0) > 0:
        val0 = np.mean(traj0[-1]) if hasattr(traj0[-1], '__iter__') else traj0[-1]
        baseline_final.append(val0)

    # Strategy 1 = SLIDE
    traj1 = start[1]
    if len(traj1) > 0:
        val1 = np.mean(traj1[-1]) if hasattr(traj1[-1], '__iter__') else traj1[-1]
        slide_final.append(val1)

# Convert to 1D float arrays
baseline_final = np.array(baseline_final, dtype=float)
slide_final    = np.array(slide_final, dtype=float)

# Perform Welch's t-test
t_stat, p_value = ttest_ind(baseline_final, slide_final, equal_var=False)

print(f"Final generation comparison:")
print(f"Baseline mean = {baseline_final.mean():.4f}, SLIDE mean = {slide_final.mean():.4f}")
print(f"t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

## ParD3

In [ ]:
with open('processed_data/ParD3_strategy_selection.pkl', 'rb') as f:
    x_vals, decay_mean, decay_rate, sweep, scipy_freq_matrix, run, scatter, line, ParD3_decay_multi, strategy_selection_params = unpack_strategy_selection(pickle.load(f))


In [ ]:
plt.figure(figsize=(3, 1.5), dpi=300)
plt.scatter(x_vals, scatter, label='Mean fitness', s=15)
plt.plot(x_vals, line, label='Fit')
plt.ylabel('Fitness relative to WT', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title(f'ParD3 fitness decay, $\\rho$ = {np.around(decay_rate[0]/2,2)}',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
plt.ylim(0,1.1)
plt.show()

plt.savefig('figures/ParD3_decay.pdf', dpi=dpi)

In [ ]:

plt.figure(figsize=(1.5, 1.5), dpi=300)
mean_sweep = sweep.mean(axis=(0,3))
plt.imshow(mean_sweep)

# --- Add scatter plot with square markers proportional to frequency ---
max_size = 100  # Adjust as needed
dot_sizes = (scipy_freq_matrix / scipy_freq_matrix.max()) * max_size

for i in range(5):  # rows
    for j in range(5):  # columns
        if scipy_freq_matrix[i, j] > 0:
            plt.scatter(
                j - 0.2, i,                 # offset left
                s=dot_sizes[i, j] * 0.2,
                color=c2,
                alpha=1,
                marker='o',
                label='SLIDE'  # avoid duplicate labels
            )

# Baseline marker (offset right)
plt.scatter(
    0 + 0.2, 4, s=20, color=c1, alpha=1, marker='o', label='Baseline'
)

# --- Optimal strategy marker as red circle ---
max_idx = np.unravel_index(np.argmax(mean_sweep, axis=None), mean_sweep.shape)
optimal_i, optimal_j = max_idx
plt.scatter(
    optimal_j, optimal_i,
    s=30,              # slightly larger to stand out
    color='red',
    alpha=1,
    marker='o',
    linewidth=0.5,
    label='Optimal'
)

# --- Formatting ---
plt.xticks([0, 4], labels=[0.0, 0.19])
plt.yticks([0, 4], labels=[20, 1])
plt.ylabel('No. sub populations', fontsize=labelsize - 1)
plt.xlabel('Base chance', fontsize=labelsize - 1)
plt.title('Strategy space', fontsize=titlesize - 1)
plt.tick_params(axis='both', which='major', labelsize=5)

# --- Legend outside, smaller ---
plt.legend(
    fontsize=6,
    loc='center left',
    bbox_to_anchor=(1.07, 0.5)
)

plt.savefig('figures/ParD3_strategy_space.pdf', dpi=dpi, bbox_inches='tight')
plt.show()


In [ ]:
# --- Compute mean over reps first ---
run_startmean = [
    [
        np.mean(strategy, axis=0)  # mean over reps for each strategy
        for strategy in start
    ]
    for start in run
]

n_strat = len(run_startmean[0])
n_steps = max(len(start[0]) for start in run_startmean)  # max number of steps across starts

strategy_mean = []
strategy_std  = []

# --- Compute mean/std across starts ---
for s in range(n_strat):
    # Gather the mean-over-reps trajectories for this strategy
    start_vals = [start[s] for start in run_startmean]  # list of arrays (steps)
    
    # Pad sequences to the same length with NaN (optional)
    max_len = max(len(traj) for traj in start_vals)
    start_vals_pad = [list(traj) + [np.nan]*(max_len - len(traj)) for traj in start_vals]

    # Compute mean and std at each step, ignoring NaNs
    mean_vals = [np.nanmean([traj[step] for traj in start_vals_pad]) for step in range(max_len)]
    std_vals  = [np.nanstd([traj[step] for traj in start_vals_pad], ddof=1) for step in range(max_len)]
    
    strategy_mean.append(mean_vals)
    strategy_std.append(std_vals)

# --- Plotting ---
plt.figure(figsize=(4, 1.5), dpi=300)
steps = np.arange(n_steps)
colors = [c1, c2]
labels = ['Baseline', 'SLIDE']
adjust = [0, -0.005]

for s in range(n_strat):
    plt.plot(steps, np.array(strategy_mean[s])+adjust[s], label=labels[s], c=colors[s])
    plt.fill_between(
        steps,
        [m - std for m, std in zip(strategy_mean[s], strategy_std[s])],
        [m + std for m, std in zip(strategy_mean[s], strategy_std[s])],
        alpha=0.3,
        color=colors[s],
        linewidth=0
    )

plt.ylabel('Fitness relative to WT', fontsize=labelsize)
plt.xlabel(r'Generations $M$', fontsize=labelsize)
plt.title('TrpB directed evolution, 10 start average', fontsize=titlesize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.legend(fontsize=8)
plt.savefig('figures/TEV_DE.pdf', dpi=dpi)
plt.show()


In [ ]:
from scipy.stats import ttest_ind

baseline_final = []
slide_final = []

for start in run_startmean:
    # Strategy 0 = Baseline
    traj0 = start[0]
    if len(traj0) > 0:
        val0 = np.mean(traj0[-1]) if hasattr(traj0[-1], '__iter__') else traj0[-1]
        baseline_final.append(val0)

    # Strategy 1 = SLIDE
    traj1 = start[1]
    if len(traj1) > 0:
        val1 = np.mean(traj1[-1]) if hasattr(traj1[-1], '__iter__') else traj1[-1]
        slide_final.append(val1)

# Convert to 1D float arrays
baseline_final = np.array(baseline_final, dtype=float)
slide_final    = np.array(slide_final, dtype=float)

# Perform Welch's t-test
t_stat, p_value = ttest_ind(baseline_final, slide_final, equal_var=False)

print(f"Final generation comparison:")
print(f"Baseline mean = {baseline_final.mean():.4f}, SLIDE mean = {slide_final.mean():.4f}")
print(f"t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

### Fourier Methods Figure

Paper reference: Figure 4 (empirical landscape ruggedness and heterogeneity).


In [ ]:
with open('processed_data/fourier_analysis.pkl', 'rb') as f:
    nk_data = pickle.load(f)
    
def get_exp_matrix(
    N: int,
    A: int,
    mutations: np.array,
    is_squared: bool = True,
    fix_b0: bool = False,
) -> np.array:
    eigrange = range(N+1) if fix_b0 == False else range(1, N+1)
    eigenvalues = [A*i for i in eigrange]
    factor = 2 if is_squared is True else 1
    if fix_b0:
        return np.array([[np.exp(-mut/(N*(A-1))*l*factor)-1 for l in eigenvalues] for mut in mutations[1:]])
    else:
        return np.array([[np.exp(-mut/(N*(A-1))*l*factor) for l in eigenvalues] for mut in mutations])


## Measure decay rate function.
def get_fourier_coeffs(
    mean_fitness: np.array,
    mutations: np.array,
    N: int,
    A: int,
    is_squared: bool = False,
    fix_b0: bool = False,
    method: str = "nnls",
    alpha: str = 0.1,
) -> tuple[np.array, np.array]:
    """
    Inputs:
    mean_fitness: vector with mean fitness values
    mutations: vector with number of mutations, starting with 0
    N: length of gene
    A: number of alleles (e.g., 20 if amino acids)
    is_squared: flag to set true if mean fitness squared is provided instead.
    method: pick on of ls, ls_constrained, nnls. Results may vary. Latter to enforce positiveness of the coeffs, although positiveness is only true for the squared estimate.
    Outputs:
    (fourier_coeffs, exponentials): tuple[np.array, np.array]
    fourier_coeffs: Vector of length N+1 with Fourier coeffs from low to high frequency, with the first element corresponding to the constant.
    exponentials: len(mutations)xN+1 matrix to extract fit via exponentials*weights
    """
    exponentials = get_exp_matrix(N=N, A=A, mutations=mutations, is_squared=is_squared, fix_b0=fix_b0)
    if fix_b0:
        mean_fitness_0 = mean_fitness[0]
        mean_fitness = mean_fitness[1:] - mean_fitness_0
    if method == "ls":
        fourier_coeffs, residuals, rank, s = np.linalg.lstsq(exponentials, mean_fitness, rcond=None)
    elif method == "ls_constrained":
        res = scipy.optimize.lsq_linear(exponentials, mean_fitness, bounds=(0, np.inf))
        fourier_coeffs = res.x
    elif method == "nnls":
        fourier_coeffs, rnorm = scipy.optimize.nnls(exponentials, mean_fitness)
    elif method == "nnls_reg":
        if alpha < 0:
            raise ValueError("alpha must be >= 0")
        m, p = exponentials.shape
        A_aug = np.vstack([exponentials, np.sqrt(alpha) * np.eye(p)])
        b_aug = np.concatenate([mean_fitness, np.zeros(p)])
        fourier_coeffs, _ = scipy.optimize.nnls(A_aug, b_aug)
    else:
        raise ValueError("Method unavailable.")
    if fix_b0:
        b0 = mean_fitness_0 - np.sum(np.abs(fourier_coeffs))
        fourier_coeffs = np.concatenate((np.array([b0]), fourier_coeffs))
        exponentials = get_exp_matrix(N=N, A=A, mutations=mutations, is_squared=is_squared, fix_b0=False)
    return (fourier_coeffs, exponentials)

def model_function(x,*params):

    """
    This is the what we are fitting to (sum of exponentials).
    It assumes a decay rate of 0.5 mutations per step.

    x = steps
    params = the output of the fitting function (get_single_decay_rate).
    """

    mut = 0.5
    num_params = 1
    constant = params[-1]
    params = params[:-1]
    mut_curves = np.exp(-1.0*mut*x[:,None]*np.array(params)[None,:])
    weights = np.linspace(0.1, 0.9, num_params)
    weights = np.ones(num_params)
    weights = weights / weights.sum()
    sum_curves = np.sum(mut_curves * weights[None,:], axis = 1)
    return sum_curves * (1 - constant) + constant

## Measure decay rate function.

def get_single_decay_rate(decay_data, mut = 0.5, num_steps = 25):

    num_params = 1
    decay_data = decay_data/decay_data[0]

    if isinstance(mut, (int, float, complex)) or jnp.ndim(mut) == 0:
        steps = np.linspace(0,num_steps-1,num_steps)
    else:
        steps = mut

    if mut is None:
        mut = np.arange(len(decay_data))  # Default steps
    
    init_guess = np.linspace(0.1, 0.9, num_params)
    init_guess = np.concat([init_guess,[0.0]])
    lbounds = [0.0]*num_params + [-0.4]
    ubounds = [2.0]*num_params + [0.4]

    # From chatgpt
    #asymptote_guess = decay_data[-3:].mean() / decay_data[0]
    #lower_bound = max(0.0, asymptote_guess - 0.2)
    #upper_bound = min(1.1, asymptote_guess + 0.2)

    #init_guess = np.concatenate([np.linspace(0.1, 0.9, num_params), [asymptote_guess]])
    #lbounds = [0.0]*num_params + [lower_bound]
    #ubounds = [2.0]*num_params + [upper_bound]

    params, _ = curve_fit(model_function, steps, decay_data,p0=init_guess, maxfev= 9000, ftol = 1e-4, xtol = 1e-5, bounds = (lbounds, ubounds))

    mean_params = np.mean(params[:-1])
    fitted_constant = params[-1]  # The second returned parameter

    return mean_params, fitted_constant  # Return full params for plotting

def get_single_decay_rate_NEW(decay_data, mut = 0.1, num_steps = 25):

    num_params = 2
    decay_data = decay_data/decay_data[0]
    def model_function(x,*params):
        mut_curves = np.exp(-1.0*mut*x[:,None]*np.array(params)[None,:])
        weights = np.linspace(0.1, 0.9, num_params)
        weights = np.ones(num_params)
        weights = weights / weights.sum()
        sum_curves = np.sum(mut_curves * weights[None,:], axis = 1)
        return sum_curves
    
    steps = np.linspace(0,num_steps-1,num_steps)

    init_guess = np.linspace(0.1, 0.9, num_params)
    params, _ = curve_fit(model_function, steps, decay_data,p0=init_guess, maxfev= 5000)

    mean_params = np.mean(params)
    return mean_params

In [ ]:
N = nk_data['N_used']
K_vec = nk_data['Ks_used']
A = nk_data['A_used']
NK_landscapes = nk_data['nk_builts']
NK_spectra = []
NK_max = [(K+1)*(A-1)/A for K in K_vec]
for f in NK_landscapes:
    NK_spectra.append(get_landscape_spectrum(f, norm = True, remove_constant = False, on_gpu = True))
    
mut = 0.5
mutations = np.arange(start=0, stop=5.5, step=mut)
num_steps = len(mutations)
exponentials = get_exp_matrix(N=N, A=A, mutations=mutations, is_squared=True)
fitness_decay = []
fitness_decay_terms = []
fitted_decay = []
fitted_rho = []
fitted_rho_NEW = []
for i, K in enumerate(K_vec):
    fitness_decay.append(np.dot(exponentials, NK_spectra[i]))
    fitness_decay[i] = fitness_decay[i]# /fitness_decay[i][0]
    fitness_decay_terms_K = np.zeros(exponentials.shape)
    for j in range(exponentials.shape[1]):
        fitness_decay_terms_K[:, j] = exponentials[:, j] * NK_spectra[i][j]
    fitness_decay_terms.append(fitness_decay_terms_K)
    
    estim_data = get_single_decay_rate(fitness_decay[i], mut = mut, num_steps = num_steps)
    fitted_decay.append(np.array([np.exp(-mutations*(estim_data[0]))*(1 - estim_data[-1])]))
    fitted_rho.append(estim_data[0])
    fitted_rho_NEW.append(get_single_decay_rate_NEW(fitness_decay[i], mut = mut, num_steps = num_steps))

In [ ]:
# Set globally
rcParams['font.family'] = 'Open Sans'

N = nk_data['N_used']
K_vec = nk_data['Ks_used']
A = nk_data['A_used']
NK_landscapes = nk_data['nk_builts']
NK_spectra = []
NK_max = [(K+1)*(A-1)/A for K in K_vec]
for f in NK_landscapes:
    NK_spectra.append(get_landscape_spectrum(f, norm = True, remove_constant = False, on_gpu = True))
    
mut = 0.5
mutations = np.arange(start=0, stop=5.5, step=mut)
num_steps = len(mutations)
exponentials = get_exp_matrix(N=N, A=A, mutations=mutations, is_squared=True)
fitness_decay = []
fitness_decay_terms = []
fitted_decay = []
fitted_rho = []
for i, K in enumerate(K_vec):
    fitness_decay.append(np.dot(exponentials, NK_spectra[i]))
    tmp = 1 # fitness_decay[i][0]
    fitness_decay[i] = fitness_decay[i] /tmp
    NK_spectra[i] = NK_spectra[i] / tmp
    fitness_decay_terms_K = np.zeros(exponentials.shape)
    for j in range(exponentials.shape[1]):
        fitness_decay_terms_K[:, j] = exponentials[:, j] * NK_spectra[i][j]
    fitness_decay_terms.append(fitness_decay_terms_K)
    
    estim_data = get_single_decay_rate(fitness_decay[i], mut = mut, num_steps = num_steps)
    # fitted_decay.append(np.array([np.exp(-mutations*(estim_data[0]))*(1 - estim_data[-1]) + estim_data[-1]]))
    fitted_decay.append(np.array([np.exp(-mutations*(estim_data[0]))*(1 - estim_data[-1])]))
    fitted_rho.append(estim_data[0])

markers = ['o', 's', 'D', '^', 'v', '<', '>', 'x', '+', '*']

linestyles = ['-', '--', '-.', ':']

import matplotlib.gridspec as gridspec
fig = plt.figure(figsize=(18, 5))
gs = gridspec.GridSpec(2, 3, figure=fig)

axx = fig.add_subplot(gs[:, 0])
for i, nk in enumerate(K_vec):
    line, = axx.plot(
        range(N + 1), NK_spectra[i],
        label=f"$K={nk}$",
        marker=markers[i],
        markersize=6,
        linewidth=1.5
    )
    color = line.get_color()
    axx.axvline(NK_max[i], color=color, linestyle=':')
    axx.axvline(fitted_rho[i]*N*(A-1)/2/A, color=color, linestyle='--')
    # axx.axvline(fitted_rho_NEW[i]*N*(A-1)/2/A, color=color, linestyle='-.')
    line, = axx.plot(
        range(N + 1), NK_spectra[i],
        color=color,
        markersize=6,
        linewidth=1.5
    )
axx.legend()
axx.set_xlim([0, N])
axx.set_ylim([0, 1])
axx.set_xlabel("Frequency index $i$ (-)")
axx.set_ylabel(f"Coefficients $b_i$ (-)")
axx.set_title(f"Power spectra (N={N}, A={A})", fontsize=10, fontweight='bold')
axx.text(
    -0.1, 1, "a",            # x, y in axes fraction
    transform=axx.transAxes,
    fontsize=20,
    va='bottom', ha='right'
)

# middle plot
for axi, (sel, ii) in enumerate(zip([1,3],[2,5])):
    axx = axx = fig.add_subplot(gs[axi, 1])
    
    axx.plot(
        mutations, fitness_decay[sel]-fitness_decay_terms[sel][0, 0],
        label=f"$G_{{\\mu}}-b_0$", color='k', linestyle='-', markersize=6, linewidth=2.2,
    )
    axx.plot(
        mutations, fitted_decay[sel][0,:],
        label=f"$G_{{\\mu,\\rho_2}}-c$", color='black', linestyle=':', markersize=6, linewidth=2.2,
    )
    for j, (ls, mk) in zip(range(exponentials.shape[1]), itertools.product(linestyles, markers)):
        if j>0 and NK_spectra[sel][j]>1e-4:
            if j == ii:
                axx.plot(
                    mutations,
                    fitness_decay_terms[sel][:, j], # + fitness_decay[sel] - fitness_decay_terms[sel][0, j],
                    label=f"$b_{{{j}}}\\mathrm{{e}}^{{\\frac{{-2\\mu\\lambda_{i}}}{{d}}}}$", 
                    linestyle='-', 
                    marker=mk, markersize=6, linewidth=2.2,
                )
            else:
                axx.plot(
                    mutations,
                    fitness_decay_terms[sel][:, j], # + fitness_decay[sel] - fitness_decay_terms[sel][0, j],
                    #label=f"$b_{{{j}}}\\mathrm{{e}}^{{\\frac{{-2\\mu\\lambda_{i}}}{{d}}}}$", 
                    linestyle='--', 
                    marker=mk, markersize=4, linewidth=1.5,
                )
    axx.legend(ncol=1)
    axx.set_title(f"Fitness decay ($K={K_vec[sel]}$)", fontsize=10, fontweight='bold')
    axx.set_xlabel(f"Mutations $\\mu$ (-)", )
    axx.set_ylabel(f"Decay (-)" )
    axx.set_xlim([0,mutations.max()])
    axx.set_ylim([0,(fitness_decay[sel]-fitness_decay_terms[sel][0, 0]).max()])
    
    letters = ["b", "c"]
    axx.text(
        -0.1, 1, letters[axi],
        transform=axx.transAxes,
        fontsize=20,
        va='bottom', ha='right'
    )
    
# right plot
for i, sel in enumerate([1,3]):
    decay = fitness_decay[sel]
    np.random.seed(2122)
    dnoise = 0.05
    noise = np.random.uniform(low=-dnoise,high=dnoise,size=fitness_decay[sel].shape)
    decay_noisy = fitness_decay[sel] + noise

    spectrum, _ = get_fourier_coeffs(mean_fitness=decay, mutations=mutations, N=N, A=A, is_squared=True, method="ls_constrained", fix_b0=True)
    spectrum_noisy, _ = get_fourier_coeffs(mean_fitness=decay_noisy, mutations=mutations, N=N, A=A, is_squared=True, method="nnls", fix_b0=True)
    spectrum_reg, _ = get_fourier_coeffs(mean_fitness=decay_noisy, mutations=mutations, N=N, A=A, is_squared=True, method="nnls_reg",alpha=1e-3, fix_b0=True)

    axx = fig.add_subplot(gs[i, 2])
    axx.plot(range(N + 1), NK_spectra[sel], label=f"True", markersize=7, linewidth=2, marker=markers[0])
    axx.plot(range(N + 1), spectrum, label=f"Estimated", markersize=5, linewidth=1.5, marker=markers[1], linestyle="--")
    axx.plot(range(N + 1), spectrum_noisy, label=f"Noisy", markersize=6, linewidth=1.5, marker=markers[2], linestyle="-")
    axx.plot(range(N + 1), spectrum_reg, label=f"Regularised", markersize=6, linewidth=1.5, marker=markers[3], linestyle=":")
    axx.legend()
    axx.set_xlim([0, N])
    axx.set_ylim([0, max(1,spectrum_noisy.max())])
    axx.set_xlabel("Frequency index $i$ (-)")
    axx.set_ylabel(f"Coefficients $b_i$ (-)")
    axx.set_title(f"Spectrum estimation ($K={K_vec[sel]}$)", fontsize=10, fontweight='bold')
    
    letters = ["d", "e"]
    axx.text(
        -0.1, 1, letters[i],
        transform=axx.transAxes,
        fontsize=20,
        va='bottom', ha='right'
    )
    
fig.subplots_adjust(hspace=0.5, wspace=0.25)
plt.savefig('figures/NK_spectra.pdf', dpi=dpi)

## Basis Function Plots

In [ ]:
# Updated: add base grid at z=BASE_Z with major (step=1) and optional minor lines.
# Keeps ordering + drop lines + smooth overlay. Base plane optional.


## Fontsizes

titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"
c2 = 'tab:blue'#298c8c'
c1 = 'tab:orange' #800074'
c3 = '#f55f74'
c4 = 'tab:green'

# Optional JAX acceleration
try:
    xp = jnp
except Exception:
    xp = np

N = 4
fine_M = 101           # smooth surface sampling
BASE_Z = 0.0           # base plane/grid height
SHOW_BASE_PLANE = False
SHOW_BASE_GRID = True
GRID_MAJOR_STEP = 1.0  # draw lines every 1.0 unit (integer grid)
GRID_MINOR_STEP = 0.5  # set to None to disable minor grid
GRID_COLOR_MAJOR = "0.35"
GRID_COLOR_MINOR = "0.65"
GRID_LW_MAJOR = 1.1
GRID_LW_MINOR = 0.6
GRID_ALPHA_MAJOR = 0.6
GRID_ALPHA_MINOR = 0.35

# --- grids ---
xs = np.arange(N)
X1d, X2d = np.meshgrid(xs, xs, indexing="ij")
t = np.linspace(0, N, fine_M, endpoint=False)
X1s, X2s = np.meshgrid(t, t, indexing="ij")

def theta(k1, k2, X1, X2):
    return (2 * xp.pi / N) * (k1 * X1 + k2 * X2)

def freq_mag(k):
    return int(min(k % N, (-k) % N))

SELF = {(0,0), (N//2,0), (0,N//2), (N//2,N//2)}

def is_rep(k1,k2):
    k1m, k2m = (-k1) % N, (-k2) % N
    if (k1,k2) == (k1m,k2m):
        return True
    return (k1,k2) < (k1m,k2m)

items = []

# 1) self-conjugate cos modes
for k in [(0,0),(2,0),(0,2),(2,2)]:
    k1,k2 = k
    T_d = theta(k1,k2, xp.asarray(X1d), xp.asarray(X2d))
    T_s = theta(k1,k2, xp.asarray(X1s), xp.asarray(X2s))
    fd = xp.cos(T_d); fs = xp.cos(T_s)
    items.append(dict(
        k=(k1,k2), xmag=freq_mag(k1), ymag=freq_mag(k2),
        kind="cos", fd=np.array(fd), fs=np.array(fs),
        const_x=(freq_mag(k1)==0), const_y=(freq_mag(k2)==0)
    ))

# 2) paired reps: √2·cos and √2·sin
for k1 in range(N):
    for k2 in range(N):
        if (k1,k2) in SELF: 
            continue
        if not is_rep(k1,k2):
            continue
        T_d = theta(k1,k2, xp.asarray(X1d), xp.asarray(X2d))
        T_s = theta(k1,k2, xp.asarray(X1s), xp.asarray(X2s))
        for kind, trig in [("cos", xp.cos), ("sin", xp.sin)]:
            fd = xp.sqrt(2.0) * trig(T_d)
            fs = xp.sqrt(2.0) * trig(T_s)
            items.append(dict(
                k=(k1,k2), xmag=freq_mag(k1), ymag=freq_mag(k2),
                kind=kind, fd=np.array(fd), fs=np.array(fs),
                const_x=(freq_mag(k1)==0), const_y=(freq_mag(k2)==0)
            ))

assert len(items) == 16

# --- Construct ordered 4x4 grid with constraints -----------------------------
def srt(it): return (it["xmag"], 0 if it["kind"]=="cos" else 1, it["k"])

rows = {0:[],1:[],2:[]}
for it in items:
    rows[it["ymag"]].append(it)

def take_first(pool, predicate, sort_key):
    cand = [it for it in pool if predicate(it)]
    cand.sort(key=sort_key)
    if not cand:
        return None
    pick = cand[0]
    pool.remove(pick)
    return pick

pool0, pool1, pool2 = rows[0][:], rows[1][:], rows[2][:]
pool0.sort(key=srt); pool1.sort(key=srt); pool2.sort(key=srt)

grid = [[None]*4 for _ in range(4)]
# Row 0 (ky=0): kx=0,1cos,1sin,2
grid[0][0] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==0, srt)
grid[0][1] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==1 and it["kind"]=="cos", srt)
grid[0][2] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==1 and it["kind"]=="sin", srt)
grid[0][3] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==2, srt)

# Rows 1 & 2 (|ky|=1): left col constant in x (kx=0), cos then sin
grid[1][0] = take_first(pool1, lambda it: it["const_x"] and it["kind"]=="cos", srt)
grid[2][0] = take_first(pool1, lambda it: it["const_x"] and it["kind"]=="sin", srt)
for c in [1,2,3]:
    grid[1][c] = take_first(pool1, lambda it: not it["const_x"], srt)
for c in [1,2,3]:
    grid[2][c] = take_first(pool1, lambda it: not it["const_x"], srt)

# Row 3 (|ky|=2): left col kx=0, then xmag=1 (cos,sin), then xmag=2
grid[3][0] = take_first(pool2, lambda it: it["const_x"], srt)
grid[3][1] = take_first(pool2, lambda it: it["xmag"]==1 and it["kind"]=="cos", srt)
grid[3][2] = take_first(pool2, lambda it: it["xmag"]==1 and it["kind"]=="sin", srt)
grid[3][3] = take_first(pool2, lambda it: it["xmag"]==2, srt)

ordered = [grid[r][c] for r in range(4) for c in range(4)]
assert all(it is not None for it in ordered)

# --- Orthonormality check -----------------------------------------------------
B = np.stack([it["fd"].ravel() for it in ordered], axis=1)
G = (B.T @ B) / (N*N)
print("Orthonormal (max off-diag):", float(np.max(np.abs(G - np.eye(16)))))

# --- Manual permutation hook --------------------------------------------------
PERM = list(range(16))  # edit this to re-order panels
ordered = [ordered[i] for i in PERM]



print("\nPanel index → label (before PERM):")
SELF = {(0,0),(2,0),(0,2),(2,2)}
for idx, it in enumerate([grid[r][c] for r in range(4) for c in range(4)]):
    k1,k2 = it["k"]
    tag = f"{'√2·' if (k1,k2) not in SELF else ''}{it['kind']}[{k1},{k2}]"
    print(f"{idx:2d}: {tag}  (|kx|={it['xmag']}, |ky|={it['ymag']})")

# --- Helpers ------------------------------------------------------------------
def draw_base_grid(ax, N, z=0.0, major_step=1.0, minor_step=0.5):
    """Draw a 2D grid on plane z at integer coordinates (and optional minors)."""
    # Major lines
    vals = np.arange(0, N+1, major_step)
    for xi in vals:
        ax.plot([xi, xi], [0, N], [z, z], color=GRID_COLOR_MAJOR,
                linewidth=GRID_LW_MAJOR, alpha=GRID_ALPHA_MAJOR)
    for yi in vals:
        ax.plot([0, N], [yi, yi], [z, z], color=GRID_COLOR_MAJOR,
                linewidth=GRID_LW_MAJOR, alpha=GRID_ALPHA_MAJOR)
    # Minor lines
    if minor_step and minor_step > 0 and minor_step < major_step:
        vals_minor = np.arange(0, N+1, minor_step)
        # remove majors to avoid double-draw
        majors = set(np.round(vals, 8).tolist())
        for xi in vals_minor:
            if np.round(xi,8) in majors: 
                continue
            ax.plot([xi, xi], [0, N], [z, z], color=GRID_COLOR_MINOR,
                    linewidth=GRID_LW_MINOR, alpha=GRID_ALPHA_MINOR)
        for yi in vals_minor:
            if np.round(yi,8) in majors:
                continue
            ax.plot([0, N], [yi, yi], [z, z], color=GRID_COLOR_MINOR,
                    linewidth=GRID_LW_MINOR, alpha=GRID_ALPHA_MINOR)

# --- Plot ---------------------------------------------------------------------
fig = plt.figure(figsize=(18, 12))
for i, it in enumerate(ordered, start=1):
    ax = fig.add_subplot(4, 4, i, projection='3d')
    # smooth surface
    from matplotlib import cm
    from matplotlib.colors import Normalize

    norm = Normalize(vmin=it["fs"].min(), vmax=it["fs"].max())
    colors = cm.viridis(norm(it["fs"]))

    ax.plot_surface(X1s, X2s, it["fs"],
                facecolors=colors,
                linewidth=0, antialiased=True, alpha=0.4)
    # optional base plane (very light)
    if False:
        ax.plot_surface(X1s, X2s, np.full_like(X1s, BASE_Z), linewidth=0, alpha=0.08)
    # discrete points as crosses
    x = X1d.ravel()
    y = X2d.ravel()
    z = it["fd"].ravel()
    ax.scatter(x, y, z, marker='x', s=50, depthshade=False, linewidths=0.9, color  = "grey")
    # vertical drop lines to BASE_Z
    for xi, yi, zi in zip(x, y, z):
        ax.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color='grey')
    # base grid
    if SHOW_BASE_GRID:
        draw_base_grid(ax, N, z=BASE_Z, major_step=GRID_MAJOR_STEP, minor_step=GRID_MINOR_STEP)
    # title
    k1,k2 = it["k"]
    print(f"Panel {i-1:2d}: k=({k1},{k2}), kind={it['kind']}, |kx|={it['xmag']}, |ky|={it['ymag']}")
    eigenvalue = 4* (k1 > 0) + 4* (k2 > 0)  # Laplacian eigenvalue
    title = f"{'√2·' if (k1,k2) not in SELF else ''}{it['kind']}[{k1},{k2}]   (|kx|={it['xmag']}, |ky|={it['ymag']})\nEigenvalue: {eigenvalue}"
    ax.set_title(title, fontsize=9)
    ax.set_xticks(range(N)); ax.set_yticks(range(N)); ax.set_zticks([-1, 0, 1])
    ax.set_xlabel("x₁"); ax.set_ylabel("x₂")

fig.suptitle("Real Orthonormal Fourier Basis (N=4)\nDiscrete 'X' samples + drop lines + smooth overlay + base grid", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
def smooth_func(x, y):
    return jnp.sin(y / 3)

def bump_func(x, y):
    return -jnp.sin(x * jnp.pi / 2 + y * jnp.pi / 2) * jnp.cos(y * jnp.pi / 4)

smooth_func_s = smooth_func(X1s, X2s)
smooth_func_d = smooth_func(X1d, X2d)
bump_func_s = bump_func(X1s, X2s)
bump_func_d = bump_func(X1d, X2d)

In [ ]:

# ---- Style settings ----
titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"

cmap = plt.get_cmap("viridis")

# ---------- First plot ----------
fig1 = plt.figure(figsize=(3,3), dpi=300, constrained_layout=True)
ax1 = fig1.add_subplot(111, projection='3d')

# Normalize color scale
zmin = min(smooth_func_d.min(), smooth_func_s.min())
zmax = max(smooth_func_d.max(), smooth_func_s.max())
norm = plt.Normalize(zmin, zmax)

ax1.plot_surface(
    X1s, X2s, smooth_func_s,
    linewidth=0, antialiased=True, alpha=1,
    cmap=cmap, norm=norm
)

# ax1.scatter(
#     X1d, X2d, smooth_func_d,
#     marker='x', s=50, depthshade=False,
#     linewidths=1.5, color='gray'
# )

# for xi, yi, zi in zip(X1d.ravel(), X2d.ravel(), smooth_func_d.ravel()):
#     ax1.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color='gray')

# ---- Title & fonts ----
ax1.set_title("Smooth Landscape", fontsize=titlesize, y=1)  
# y < 1 moves it down toward the plot (default is ~1.0)

# ---- Axis label sizes ----
ax1.set_xlabel(r"$\sigma_1$", fontsize=labelsize, labelpad=2)
ax1.set_ylabel(r"$\sigma_2$", fontsize=labelsize, labelpad=2)
ax1.set_zlabel("Arbitrary Fitness", fontsize=labelsize)

# Adjust z-label position so it stays inside the figure
ax1.zaxis.labelpad = 15  # more space for z-label

# ---- Tick label sizes ----
ax1.tick_params(axis='both', which='major', labelsize=ticksize)
ax1.tick_params(axis='both', which='minor', labelsize=ticksize)

from matplotlib.ticker import MaxNLocator

# Force integer ticks on all axes
ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
ax1.yaxis.set_major_locator(MaxNLocator(integer=True))
ax1.zaxis.set_major_locator(MaxNLocator(integer=True))

plt.savefig('figures/smooth_landscape_3D.pdf', dpi=dpi)



In [ ]:

# ---- Style settings ----
titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"

cmap = plt.get_cmap("viridis")

# ---------- Second plot ----------
fig2 = plt.figure(figsize=(3, 3), dpi=300, constrained_layout=True)
ax2 = fig2.add_subplot(111, projection='3d')

# Normalize color scale
zmin = min(bump_func_d.min(), bump_func_s.min())
zmax = max(bump_func_d.max(), bump_func_s.max())
norm = plt.Normalize(zmin, zmax)

ax2.plot_surface(
    X1s, X2s, bump_func_s,
    linewidth=0, antialiased=True, alpha=1,   # match first plot’s alpha
    cmap=cmap, norm=norm
)

# ax2.scatter(
#     X1d, X2d, bump_func_d,
#     marker='x', s=50, depthshade=False,
#     linewidths=1.5, color='gray'
# )

# for xi, yi, zi in zip(X1d.ravel(), X2d.ravel(), bump_func_d.ravel()):
#     ax2.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color='gray')

# ---- Title & fonts ----
ax2.set_title("Rugged Landscape", fontsize=titlesize, y=1)  

# ---- Axis label sizes ----
ax2.set_xlabel(r"$\sigma_1$", fontsize=labelsize, labelpad=2)
ax2.set_ylabel(r"$\sigma_2$", fontsize=labelsize, labelpad=2)
ax2.set_zlabel("Arbitrary Fitness", fontsize=labelsize)

# Adjust z-label position so it stays inside the figure
ax2.zaxis.labelpad = 15

# ---- Tick label sizes ----
ax2.tick_params(axis='both', which='major', labelsize=ticksize)
ax2.tick_params(axis='both', which='minor', labelsize=ticksize)

# Force integer ticks on all axes
ax2.xaxis.set_major_locator(MaxNLocator(integer=True))
ax2.yaxis.set_major_locator(MaxNLocator(integer=True))

plt.savefig('figures/rugged_landscape_3D.pdf', dpi=dpi)


In [ ]:

## Fontsizes

titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"
c2 = 'tab:blue'#298c8c'
c1 = 'tab:orange' #800074'
c3 = '#f55f74'
c4 = 'tab:green'

# Optional JAX acceleration
try:
    xp = jnp
except Exception:
    xp = np

N = 4
fine_M = 101           # smooth surface sampling
BASE_Z = 0.0           # base plane/grid height
SHOW_BASE_PLANE = False
SHOW_BASE_GRID = True
GRID_MAJOR_STEP = 1.0  # draw lines every 1.0 unit (integer grid)
GRID_MINOR_STEP = 0.5  # set to None to disable minor grid
GRID_COLOR_MAJOR = "0.35"
GRID_COLOR_MINOR = "0.65"
GRID_LW_MAJOR = 1.1
GRID_LW_MINOR = 0.6
GRID_ALPHA_MAJOR = 0.6
GRID_ALPHA_MINOR = 0.35

# --- grids ---
xs = np.arange(N)
X1d, X2d = np.meshgrid(xs, xs, indexing="ij")
t = np.linspace(0, N, fine_M, endpoint=False)
X1s, X2s = np.meshgrid(t, t, indexing="ij")

def theta(k1, k2, X1, X2):
    return (2 * xp.pi / N) * (k1 * X1 + k2 * X2)

def freq_mag(k):
    return int(min(k % N, (-k) % N))

SELF = {(0,0), (N//2,0), (0,N//2), (N//2,N//2)}

def is_rep(k1,k2):
    k1m, k2m = (-k1) % N, (-k2) % N
    if (k1,k2) == (k1m,k2m):
        return True
    return (k1,k2) < (k1m,k2m)

items = []

# 1) self-conjugate cos modes
for k in [(0,0),(2,0),(0,2),(2,2)]:
    k1,k2 = k
    T_d = theta(k1,k2, xp.asarray(X1d), xp.asarray(X2d))
    T_s = theta(k1,k2, xp.asarray(X1s), xp.asarray(X2s))
    fd = xp.cos(T_d); fs = xp.cos(T_s)
    items.append(dict(
        k=(k1,k2), xmag=freq_mag(k1), ymag=freq_mag(k2),
        kind="cos", fd=np.array(fd), fs=np.array(fs),
        const_x=(freq_mag(k1)==0), const_y=(freq_mag(k2)==0)
    ))

# 2) paired reps: √2·cos and √2·sin
for k1 in range(N):
    for k2 in range(N):
        if (k1,k2) in SELF: 
            continue
        if not is_rep(k1,k2):
            continue
        T_d = theta(k1,k2, xp.asarray(X1d), xp.asarray(X2d))
        T_s = theta(k1,k2, xp.asarray(X1s), xp.asarray(X2s))
        for kind, trig in [("cos", xp.cos), ("sin", xp.sin)]:
            fd = xp.sqrt(2.0) * trig(T_d)
            fs = xp.sqrt(2.0) * trig(T_s)
            items.append(dict(
                k=(k1,k2), xmag=freq_mag(k1), ymag=freq_mag(k2),
                kind=kind, fd=np.array(fd), fs=np.array(fs),
                const_x=(freq_mag(k1)==0), const_y=(freq_mag(k2)==0)
            ))

assert len(items) == 16

# --- Construct ordered 4x4 grid with constraints -----------------------------
def srt(it): return (it["xmag"], 0 if it["kind"]=="cos" else 1, it["k"])

rows = {0:[],1:[],2:[]}
for it in items:
    rows[it["ymag"]].append(it)

def take_first(pool, predicate, sort_key):
    cand = [it for it in pool if predicate(it)]
    cand.sort(key=sort_key)
    if not cand:
        return None
    pick = cand[0]
    pool.remove(pick)
    return pick

pool0, pool1, pool2 = rows[0][:], rows[1][:], rows[2][:]
pool0.sort(key=srt); pool1.sort(key=srt); pool2.sort(key=srt)

grid = [[None]*4 for _ in range(4)]
# Row 0 (ky=0): kx=0,1cos,1sin,2
grid[0][0] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==0, srt)
grid[0][1] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==1 and it["kind"]=="cos", srt)
grid[0][2] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==1 and it["kind"]=="sin", srt)
grid[0][3] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==2, srt)

# Rows 1 & 2 (|ky|=1): left col constant in x (kx=0), cos then sin
grid[1][0] = take_first(pool1, lambda it: it["const_x"] and it["kind"]=="cos", srt)
grid[2][0] = take_first(pool1, lambda it: it["const_x"] and it["kind"]=="sin", srt)
for c in [1,2,3]:
    grid[1][c] = take_first(pool1, lambda it: not it["const_x"], srt)
for c in [1,2,3]:
    grid[2][c] = take_first(pool1, lambda it: not it["const_x"], srt)

# Row 3 (|ky|=2): left col kx=0, then xmag=1 (cos,sin), then xmag=2
grid[3][0] = take_first(pool2, lambda it: it["const_x"], srt)
grid[3][1] = take_first(pool2, lambda it: it["xmag"]==1 and it["kind"]=="cos", srt)
grid[3][2] = take_first(pool2, lambda it: it["xmag"]==1 and it["kind"]=="sin", srt)
grid[3][3] = take_first(pool2, lambda it: it["xmag"]==2, srt)

ordered = [grid[r][c] for r in range(4) for c in range(4)]
assert all(it is not None for it in ordered)

# --- Orthonormality check -----------------------------------------------------
B = np.stack([it["fd"].ravel() for it in ordered], axis=1)
G = (B.T @ B) / (N*N)
print("Orthonormal (max off-diag):", float(np.max(np.abs(G - np.eye(16)))))

# --- Manual permutation hook --------------------------------------------------
PERM = list(range(16))  # edit this to re-order panels
ordered = [ordered[i] for i in PERM]

print("\nPanel index → label (before PERM):")
SELF = {(0,0),(2,0),(0,2),(2,2)}
for idx, it in enumerate([grid[r][c] for r in range(4) for c in range(4)]):
    k1,k2 = it["k"]
    tag = f"{'√2·' if (k1,k2) not in SELF else ''}{it['kind']}[{k1},{k2}]"
    #print(f"{idx:2d}: {tag}  (|kx|={it['xmag']}, |ky|={it['ymag']})")

# --- Helpers ------------------------------------------------------------------
def draw_base_grid(ax, N, z=0.0, major_step=1.0, minor_step=0.5):
    """Draw a 2D grid on plane z at integer coordinates (and optional minors)."""
    # Major lines
    vals = np.arange(0, N+1, major_step)
    for xi in vals:
        ax.plot([xi, xi], [0, N], [z, z], color=GRID_COLOR_MAJOR,
                linewidth=GRID_LW_MAJOR, alpha=GRID_ALPHA_MAJOR)
    for yi in vals:
        ax.plot([0, N], [yi, yi], [z, z], color=GRID_COLOR_MAJOR,
                linewidth=GRID_LW_MAJOR, alpha=GRID_ALPHA_MAJOR)
    # Minor lines
    if minor_step and minor_step > 0 and minor_step < major_step:
        vals_minor = np.arange(0, N+1, minor_step)
        # remove majors to avoid double-draw
        majors = set(np.round(vals, 8).tolist())
        for xi in vals_minor:
            if np.round(xi,8) in majors: 
                continue
            ax.plot([xi, xi], [0, N], [z, z], color=GRID_COLOR_MINOR,
                    linewidth=GRID_LW_MINOR, alpha=GRID_ALPHA_MINOR)
        for yi in vals_minor:
            if np.round(yi,8) in majors:
                continue
            ax.plot([0, N], [yi, yi], [z, z], color=GRID_COLOR_MINOR,
                    linewidth=GRID_LW_MINOR, alpha=GRID_ALPHA_MINOR)

# --- Plot ---------------------------------------------------------------------
fig = plt.figure(figsize=(8,8))
for i, it in enumerate(ordered, start=1):
    ax = fig.add_subplot(4, 4, i, projection='3d')
    # smooth surface
    from matplotlib import cm
    from matplotlib.colors import Normalize

    norm = Normalize(vmin=it["fs"].min(), vmax=it["fs"].max())
    colors = cm.viridis(norm(it["fs"]))

    ax.plot_surface(X1s, X2s, it["fs"],
                facecolors=colors,
                linewidth=0, antialiased=True, alpha=0.8)
    # optional base plane (very light)
    if False:
        ax.plot_surface(X1s, X2s, np.full_like(X1s, BASE_Z), linewidth=0, alpha=0.08)
    # # discrete points as crosses
    # x = X1d.ravel()
    # y = X2d.ravel()
    # z = it["fd"].ravel()
    # ax.scatter(x, y, z, marker='x', s=50, depthshade=False, linewidths=0.9, color  = "grey")
    # # vertical drop lines to BASE_Z
    # for xi, yi, zi in zip(x, y, z):
    #     ax.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color='grey')
    # base grid
    # if SHOW_BASE_GRID:
    #     draw_base_grid(ax, N, z=BASE_Z, major_step=GRID_MAJOR_STEP, minor_step=GRID_MINOR_STEP)
    # title
    k1,k2 = it["k"]
    #print(f"Panel {i-1:2d}: k=({k1},{k2}), kind={it['kind']}, |kx|={it['xmag']}, |ky|={it['ymag']}")
    eigenvalue = 4* (k1 > 0) + 4* (k2 > 0)  # Laplacian eigenvalue
    #title = f"{'√2·' if (k1,k2) not in SELF else ''}{it['kind']}[{k1},{k2}]   (|kx|={it['xmag']}, |ky|={it['ymag']})\nEigenvalue: {eigenvalue}"
    #title = f"{'√2·' if (k1,k2) not in SELF else ''}{it['kind']}[{k1},{k2}], $\lambda$ = {eigenvalue}"
    title = f"$\lambda$ = {eigenvalue}"
    ax.set_title(title, fontsize=titlesize, y=0.99)  # smaller y brings it lower
    ax.set_xticks(range(N)); ax.set_yticks(range(N)); ax.set_zticks([-1, 0, 1])
    #ax.set_xlabel("x₁"); ax.set_ylabel("x₂")

        # Control tick label font size
    ax.tick_params(axis="both", which="major", labelsize=ticksize)
    ax.tick_params(axis="both", which="minor", labelsize=ticksize)
    ax.zaxis.set_tick_params(labelsize=ticksize)
#fig.suptitle("Real Orthonormal Fourier Basis (N=4)\nDiscrete 'X' samples + drop lines + smooth overlay + base grid", fontsize=14)
#fig.suptitle("Basis Vectors", fontsize=titlesize+3)
plt.tight_layout()
plt.savefig('figures/basis_vectors.pdf', dpi=dpi)


In [ ]:
# --- Fourier-space projection + plotting -------------------------------------

def project_to_fourier_coeffs(f_d, ordered, N):
    """
    Project a discrete function f_d (shape N×N) onto your real orthonormal basis
    defined by `ordered`. Returns a 4×4 array C whose (row, col) matches the
    panel order you're using (after PERM).
    """
    fvec = np.asarray(f_d, dtype=float).ravel()
    B = np.stack([it["fd"].ravel() for it in ordered], axis=1)  # (N^2 × 16)
    # Your basis columns are orthonormal w.r.t. (1/N^2) <.,.>, so:
    coeffs = (B.T @ fvec) / (N * N)  # (16,)
    C = coeffs.reshape(4, 4)         # row-major matches your panel order
    return C

def bilinear_surface_from_grid(C, Kx, Ky):
    """
    Bilinear interpolation over a 4×4 grid C onto fine frequency grid (Kx,Ky),
    with Kx,Ky in [0,3] along columns/rows respectively.
    Interpolates exactly at integer grid points.
    """
    # Clamp into valid cell range
    u = np.clip(Kx, 0.0, 3.0)
    v = np.clip(Ky, 0.0, 3.0)

    i0 = np.floor(v).astype(int)
    j0 = np.floor(u).astype(int)
    i1 = np.clip(i0 + 1, 0, 3)
    j1 = np.clip(j0 + 1, 0, 3)
    du = u - j0
    dv = v - i0

    # gather corners
    C00 = C[i0, j0]
    C10 = C[i1, j0]
    C01 = C[i0, j1]
    C11 = C[i1, j1]

    # bilinear blend
    S = ( (1 - du) * (1 - dv) * C00
        + (    du) * (1 - dv) * C01
        + (1 - du) * (    dv) * C10
        + (    du) * (    dv) * C11 )
    return S

def make_kspace_grids(fine_M=201):
    """
    Frequency-plane fine grid: continuous 'panel coordinates'.
    We use [0,3] because there are 4 columns (kx panels) and 4 rows (ky panels).
    """
    t = np.linspace(0.0, 3.0, fine_M)
    Kx, Ky = np.meshgrid(t, t, indexing="xy")
    return Kx, Ky

def panel_labels_from_ordered(ordered):
    """
    Build tick labels for the Fourier-space axes from the basis panels.
    (kx, ky) shown as the *actual* mode labels, respecting your ordering.
    """
    # ordered is row-major 4×4
    labels = [[None]*4 for _ in range(4)]
    idx = 0
    for r in range(4):
        for c in range(4):
            it = ordered[idx]
            k1, k2 = it["k"]
            lab = f"{'√2·' if (k1,k2) not in {(0,0),(2,0),(0,2),(2,2)} else ''}{it['kind']}[{k1},{k2}]"
            labels[r][c] = lab
            idx += 1
    return labels

def plot_fourier_space(C, title="Fourier-space coefficients (4×4)", 
                       show_base_grid=True, show_base_plane=False, base_z=0.0):
    """
    Plot a smooth Fourier-space surface (bilinear interp) over panel coordinates,
    with grey × crosses at the exact 4×4 coefficients.
    """
    from matplotlib import cm
    from matplotlib.colors import Normalize

    Kx, Ky = make_kspace_grids(fine_M=fine_M)  # fine grid in panel coordinates
    Zs = bilinear_surface_from_grid(C, Kx, Ky)

    fig = plt.figure(figsize=(7.5, 6.5))
    ax = fig.add_subplot(1, 1, 1, projection="3d")

    # Smooth surface colors
    norm = Normalize(vmin=Zs.min(), vmax=Zs.max())
    colors = cm.viridis(norm(Zs))

    # Plot smooth surface
    ax.plot_surface(Kx, Ky, Zs, facecolors=colors, linewidth=0, antialiased=True, alpha=0.4)

    # Optional base plane
    if show_base_plane:
        ax.plot_surface(Kx, Ky, np.full_like(Kx, base_z), linewidth=0, alpha=0.08)

    # Discrete coefficient crosses at integer panel coordinates
    xs = np.arange(4)
    ys = np.arange(4)
    Xd, Yd = np.meshgrid(xs, ys, indexing="xy")  # (y,row), (x,col)
    Zd = C
    ax.scatter(Xd.ravel(), Yd.ravel(), Zd.ravel(), marker='x', s=60, depthshade=False,
               linewidths=1.1, color="grey")

    # Drop lines
    for xi, yi, zi in zip(Xd.ravel(), Yd.ravel(), Zd.ravel()):
        ax.plot([xi, xi], [yi, yi], [base_z, zi], linewidth=2.0, alpha=0.9, color='grey')

    # Base grid (on Fourier panel coords)
    if show_base_grid:
        # reuse your draw_base_grid but with N=4 and plane coords
        draw_base_grid(ax, N=4, z=base_z, major_step=1.0, minor_step=0.5)

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("panel kx (columns)")
    ax.set_ylabel("panel ky (rows)")
    ax.set_zticks(np.linspace((Zd.min()), (Zd.max()), 5))

    # nice ticks exactly at panel integers
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    plt.tight_layout()
    plt.show()

In [ ]:
from matplotlib import cm
from matplotlib.colors import Normalize

# ---- Style settings ----
titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"

# --- Fourier coefficients for smooth function ---
smooth_func_d = np.array(smooth_func(X1d, X2d))
C_smooth = project_to_fourier_coeffs(smooth_func_d, ordered, N)

# --- Fine Fourier grid ---
Kx, Ky = make_kspace_grids(fine_M=201)
Zs = bilinear_surface_from_grid(C_smooth, Kx, Ky)

# --- Plot ---
fig1 = plt.figure(figsize=(3, 3), dpi=300, constrained_layout=True)
ax1 = fig1.add_subplot(111, projection="3d")

norm = Normalize(vmin=Zs.min(), vmax=Zs.max())
colors = cm.viridis(norm(Zs))

# Smooth surface
ax1.plot_surface(Kx, Ky, Zs, facecolors=colors, linewidth=0, antialiased=True, alpha=1)

# # Coefficient crosses
# xs = np.arange(4)
# ys = np.arange(4)
# Xd, Yd = np.meshgrid(xs, ys, indexing="xy")
# Zd = C_smooth
# ax1.scatter(Xd.ravel(), Yd.ravel(), Zd.ravel(), marker='x', s=60,
#             depthshade=False, linewidths=1.1, color="gray")

# # Drop lines
# for xi, yi, zi in zip(Xd.ravel(), Yd.ravel(), Zd.ravel()):
#     ax1.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color="gray")

# Title & fonts
ax1.set_title("Smooth Landscape Fourier Space", fontsize=titlesize, y=1)

# Axis labels
ax1.set_xlabel(r"$\hat{\sigma}_1$", fontsize=labelsize, labelpad=2)
ax1.set_ylabel(r"$\hat{\sigma}_2$", fontsize=labelsize, labelpad=2)
ax1.set_zlabel("Coefficient value", fontsize=labelsize)
ax1.zaxis.labelpad = 15

# Ticks
ax1.set_xticks(range(4))
ax1.set_yticks(range(4))
ax1.tick_params(axis="both", which="major", labelsize=ticksize)
ax1.tick_params(axis="both", which="minor", labelsize=ticksize)

plt.savefig('figures/smooth_fourier_3D.pdf', dpi=dpi)


In [ ]:
# --- Fourier coefficients for bump function ---
bump_func_d = np.array(bump_func(X1d, X2d))
C_bump = project_to_fourier_coeffs(bump_func_d, ordered, N)

# --- Fine Fourier grid ---
Kx, Ky = make_kspace_grids(fine_M=201)
Zs = bilinear_surface_from_grid(C_bump, Kx, Ky)

# --- Plot ---
fig2 = plt.figure(figsize=(3, 3), dpi=300, constrained_layout=True)
ax2 = fig2.add_subplot(111, projection="3d")

norm = Normalize(vmin=Zs.min(), vmax=Zs.max())
colors = cm.viridis(norm(Zs))

# Smooth surface
ax2.plot_surface(Kx, Ky, Zs, facecolors=colors, linewidth=0, antialiased=True, alpha=1)

# # Coefficient crosses
# xs = np.arange(4)
# ys = np.arange(4)
# Xd, Yd = np.meshgrid(xs, ys, indexing="xy")
# Zd = C_bump
# ax2.scatter(Xd.ravel(), Yd.ravel(), Zd.ravel(), marker='x', s=60,
#             depthshade=False, linewidths=1.1, color="gray")

# # Drop lines
# for xi, yi, zi in zip(Xd.ravel(), Yd.ravel(), Zd.ravel()):
#     ax2.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color="gray")

# Title & fonts
ax2.set_title("Rugged Landscape Fourier Space", fontsize=titlesize, y=1)

# Axis labels
ax2.set_xlabel(r"$\hat{\sigma}_1$", fontsize=labelsize, labelpad=2)
ax2.set_ylabel(r"$\hat{\sigma}_2$", fontsize=labelsize, labelpad=2)
ax2.set_zlabel("Coefficient value", fontsize=labelsize)
ax2.zaxis.labelpad = 15

# Ticks
ax2.set_xticks(range(4))
ax2.set_yticks(range(4))
ax2.tick_params(axis="both", which="major", labelsize=ticksize)
ax2.tick_params(axis="both", which="minor", labelsize=ticksize)

plt.savefig('figures/rugged_fourier_3D.pdf', dpi=dpi)


In [ ]:

# Styling parameters
titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"
c1 = 'tab:orange'   # #800074
c2 = 'tab:blue'     # #298c8c
c3 = '#f55f74'
c4 = 'tab:green'

def compute_spectral_density(C, ordered):
    """
    Compute the spectral density across eigenspaces:
    - Constant: |kx|=0 AND |ky|=0 (DC term - no variation)
    - Linear: Exactly one of |kx|>0 OR |ky|>0 (varies in one dimension only)
    - Quadratic: Both |kx|>0 AND |ky|>0 (varies in both dimensions)
    
    Returns normalized energy distribution across these three subspaces.
    """
    # Flatten coefficients and get metadata
    coeffs = C.ravel()
    
    # Classify each coefficient by its eigenspace
    constant_energy = 0.0
    linear_energy = 0.0
    quadratic_energy = 0.0
    
    for i, it in enumerate(ordered):
        xmag, ymag = it["xmag"], it["ymag"]
        coeff_sq = coeffs[i]**2
        
        # Count how many dimensions have non-zero frequency
        nonzero_dims = (xmag > 0) + (ymag > 0)
        
        if nonzero_dims == 0:
            # Both kx=0 and ky=0: constant term
            constant_energy += coeff_sq
        elif nonzero_dims == 1:
            # Either kx≠0,ky=0 or kx=0,ky≠0: linear (varies in one dimension)
            linear_energy += coeff_sq
        elif nonzero_dims == 2:
            # Both kx≠0 and ky≠0: quadratic (varies in both dimensions)
            quadratic_energy += coeff_sq
    
    # Total energy and normalization
    total_energy = constant_energy + linear_energy + quadratic_energy
    
    if total_energy > 0:
        constant_norm = constant_energy / total_energy
        linear_norm = linear_energy / total_energy
        quadratic_norm = quadratic_energy / total_energy
    else:
        constant_norm = linear_norm = quadratic_norm = 0.0
    
    return {
        'constant': {'energy': constant_energy, 'normalized': constant_norm},
        'linear': {'energy': linear_energy, 'normalized': linear_norm},
        'quadratic': {'energy': quadratic_energy, 'normalized': quadratic_norm},
        'total_energy': total_energy
    }


def plot_normalized_energy(functions_dict, title="Normalized Energy Distribution"):
    """
    Plot only the normalized spectral energy distribution across eigenspaces.
    functions_dict: {'func_name': coefficients_matrix, ...}
    """
    #eigenspaces = ['Constant\n(DC)', 'Linear\n(Fundamental)', 'Quadratic\n(Harmonics)']
    eigenspaces = [0,1,2]
    colors = [c1, c2, c3, c4]

    func_names = list(functions_dict.keys())
    n_funcs = len(func_names)
    x_pos = np.arange(len(eigenspaces))
    width = 0.35

    # Compute spectral densities
    spectral_data = {}
    for name, C in functions_dict.items():
        spectral_data[name] = compute_spectral_density(C, ordered)

    fig, ax = plt.subplots(figsize=(3,4), dpi=300)

    # Plot normalized energy distribution
    for i, name in enumerate(func_names):
        data = spectral_data[name]
        normalized = [data['constant']['normalized'],
                      data['linear']['normalized'],
                      data['quadratic']['normalized']]
        offset = (i - (n_funcs-1)/2) * width / n_funcs
        bars = ax.bar(x_pos + offset, normalized, width/n_funcs,
                      label=name, alpha=0.8, color=colors[i % len(colors)])

        # Add percentage labels on bars
        for j, bar in enumerate(bars):
            height = bar.get_height()
            if height > 0.02:  # Only label if > 2%
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'{height*100:.1f}%', ha='center', va='bottom', fontsize=ticksize)

    # Axis labels and formatting
    ax.set_xlabel('Frequency index $i$', fontsize=labelsize)
    ax.set_ylabel('Power spectral coefficients $b_i$ (%)', fontsize=labelsize)
    ax.set_title(title, fontsize=titlesize)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(eigenspaces, fontsize=ticksize)
    ax.set_yticklabels(np.round(ax.get_yticks(),2), fontsize=ticksize)
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=legendsize)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('figures/power_spectra_bar_chart.pdf', dpi=dpi)
# Generate spectral density plots for our test functions
functions_to_analyze = {
    'Smooth Landscape': C_smooth,
    'Rugged Landscape': C_bump
}

plot_normalized_energy(functions_to_analyze, 
                     title="Power Spectra")


# IK Visualisation Cells

### Notes on plot formatting

- Height 3
- DPI = 300
- Axes labels = 8, axes tick labels = 6
- Title = 10
- Heatmap: viridis, defined colours: see below

In [ ]:
## Plot colours

c2 = 'tab:blue'#298c8c'
c1 = 'tab:orange' #800074'
c3 = '#f55f74'
c4 = 'tab:green'

In [ ]:
## Fontsizes

titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
dpi = 350
plt.rcParams["font.family"] = "DejaVu Sans"

# Section 1 - Inferring ruggedness on NK

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).


### Example fitness decay curves

In [ ]:
with open('processed_data/smooth_rugged_example.pkl', 'rb') as f:
    smooth_rugged_payload = pickle.load(f)

if isinstance(smooth_rugged_payload, dict):
    smooth_rugged = smooth_rugged_payload["smooth_rugged"]
    fitted_lines = smooth_rugged_payload["fitted_lines"]
    decay_curve_labels = smooth_rugged_payload.get("labels", [r"Smooth", r"Rugged"])
    decay_curve_generations = smooth_rugged_payload.get("generations", np.arange(1, len(smooth_rugged[0]) + 1))
else:
    smooth_rugged, fitted_lines = smooth_rugged_payload
    decay_curve_labels = [r"$(K+1)/N = 0.1$", r"$(K+1)/N = 0.75$"]
    decay_curve_generations = np.arange(1, len(smooth_rugged[0]) + 1)


In [ ]:
plt.figure(figsize=(3.5,3), dpi=300)

colours = [c2, c1]
for curve, fitted_line, label, colour in zip(smooth_rugged, fitted_lines, decay_curve_labels, colours):
    plt.scatter(decay_curve_generations, curve, label=label, c=colour, s=5)
    plt.plot(decay_curve_generations, fitted_line, c=colour)
plt.legend(fontsize=legendsize)

plt.title('Fitness decay curves', fontsize=titlesize)
plt.xlabel('Generations $M$', fontsize=labelsize)
plt.ylabel(r'Fitness $F_\mu$', fontsize=labelsize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
# plt.savefig('figures/decay_curves_example.pdf', dpi=dpi)


### Ruggedness prediction accuracy over NK

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).


In [ ]:
with open('processed_data/ruggedness_accuracy.pkl', 'rb') as f:
    k_plus_one_over_ns, decay_rates = pickle.load(f)

In [ ]:
# Step 1: Get ordering for rho
arg_sort_kn = np.argsort(k_plus_one_over_ns)
sorted_rho = k_plus_one_over_ns[arg_sort_kn]
sorted_decay = decay_rates[arg_sort_kn]

# Step 2: Group the values
grouped_rho = sorted_rho.reshape(10,-1)
grouped_decay = sorted_decay.reshape(10,-1)

mean_rho = np.mean(grouped_rho, axis = 1)
mean_decay = np.mean(grouped_decay, axis = 1)
std_decay = np.std(grouped_decay, axis = 1)

true_k_over_n = np.linspace(0.1,1,10)

# Step 3: Create Fill-Between Plot
plt.figure(figsize=(3.5,3), dpi=300)
plt.plot(true_k_over_n, mean_decay, 'o-', label=r"Mean estimated $\rho$")  # Line plot with markers
plt.fill_between(true_k_over_n, mean_decay - std_decay, mean_decay + std_decay, alpha=0.3, label="±1 Std dev")  # Shaded error band

plt.plot(true_k_over_n, mean_rho, c='red', alpha=0.4,linestyle='--', label=r'$(K+1)/N$')

# Labels and Title
plt.xlabel(r"$(K+1)/N$", fontsize=labelsize)
plt.ylabel(r"Estimated $\rho$", fontsize=labelsize)
plt.title(r"$\rho$ prediction accuracy over ruggedness", fontsize = titlesize)
plt.legend(fontsize=legendsize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.grid(True)

# Show plot
# # plt.savefig('figures/accuracy_over_K.pdf', dpi=dpi)


### Ruggedness prediction accuracy over population size

The plotted axes are read from processed payload metadata when available.


In [ ]:
with open('processed_data/popsize_accuracy.pkl', 'rb') as f:
    popsize_decay_rates, pops = unpack_popsize_accuracy(pickle.load(f))


In [ ]:
y_means = popsize_decay_rates.mean(axis=1)
y_stds = popsize_decay_rates.std(axis=1)
pop_y_means = y_means
conv_p = 16/25
# Step 3: Create Fill-Between Plot
plt.figure(figsize=(3.5,1.2), dpi=300)
plt.plot(pops, y_means, 'o-', label=r"Mean estimated $\rho$")  # Line plot with markers
plt.axhline(y=conv_p, label=r'$(K+1)/N$', c='red', alpha=0.4, linestyle='--')
# plt.plot(pops, np.ones_like(pops) * 12/25, ls = '--', label="True") 
plt.fill_between(pops, y_means - y_stds, y_means + y_stds, alpha=0.3, label="±1 Std Dev")  # Shaded error band
plt.grid(True)
plt.legend(fontsize = legendsize-2, loc="upper right")
plt.tick_params(axis='both', which='major', labelsize=ticksize)

plt.title('Accuracy over population size', fontsize = titlesize)
plt.ylabel(r"Estimated $\rho$", fontsize=labelsize)
plt.xlabel('Population size', fontsize=labelsize)
# # plt.savefig('figures/accuracy_over_popsize.pdf', dpi=dpi)

### Ruggedness prediction accuracy over mutation rate

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).

The plotted axes are read from processed payload metadata when available.


In [ ]:
with open('processed_data/mut_accuracy.pkl', 'rb') as f:
    mut_decay_rates, muts = unpack_mutation_accuracy(pickle.load(f))


In [ ]:

y_means = np.mean(mut_decay_rates,axis=1)
y_stds = mut_decay_rates.std(axis=1)

# Step 3: Create Fill-Between Plot
plt.figure(figsize=(3.5,1.2), dpi=300)
plt.plot(muts, y_means, 'o-', label=r"Mean estimated $\rho$")  # Line plot with markers
plt.axhline(y=conv_p, label=r'$(K+1)/N$', c='red', alpha=0.4,linestyle='--')
plt.fill_between(muts, y_means - y_stds, y_means + y_stds, alpha=0.3, label="±1 Std Dev")  # Shaded error band
# plt.plot(muts, np.ones_like(muts) * 12/25, ls = '--', label="True") 
plt.grid(True)
plt.legend(fontsize = legendsize-2, loc='lower right')
plt.tick_params(axis='both', which='major', labelsize=ticksize)
# plt.ylim(0.0,1.0)
# plt.xlim(0,1.0)
plt.xlabel(r'Mutations per cell per generation $\theta$', fontsize=labelsize)
plt.ylabel(r"Estimated $\rho$", fontsize=labelsize)
plt.ylim(0.55,0.7)
plt.title('Accuracy over mutation rate', fontsize=titlesize)
# # plt.savefig('figures/accuracy_over_mut.pdf', dpi=dpi)


### Comparison of ruggedness metrics on NK

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).


In [ ]:
with open('processed_data/NK_ruggedness_metric_comparison.pkl', 'rb') as f:
    NK_roughness_to_slope, NK_fourier, convergence_rates, NK_paths_to_max, NK_closest_max, k_over_ns, NK_le_normed = pickle.load(f)

In [ ]:
plt.figure(figsize=(3.5,3), dpi=300)
plt.plot(k_over_ns, convergence_rates/convergence_rates.max(), label = r'$\rho$ (Decay rate)')
plt.plot(k_over_ns, 1 - NK_fourier/NK_fourier.max(), label = r'1 - Landscape ${R}^{2}$', alpha=0.6, linestyle='--')
plt.plot(k_over_ns, NK_roughness_to_slope/NK_roughness_to_slope.max(), label = 'Roughness to slope ratio', alpha=0.6, linestyle='--')
plt.plot(k_over_ns, 1-NK_paths_to_max/NK_paths_to_max.max(), label='1 - Paths to max', alpha=0.6, linestyle='--')
plt.plot(k_over_ns, NK_closest_max/NK_closest_max.max(), label='1 - Dist. to closest local max', alpha=0.6, linestyle='--')
plt.plot(k_over_ns, NK_le_normed/NK_paths_to_max.max(), label='Local Epistasis', alpha=0.6, linestyle='--')
plt.legend(loc = 'lower right', fontsize=legendsize-2)
plt.title('Ruggedness metric comparison', fontsize=titlesize)
plt.xlabel(r'$(K+1)/N$', fontsize = labelsize)
plt.ylabel('Normalised ruggedness measurements', fontsize = labelsize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
# # plt.savefig('figures/NK_ruggedness_metric_comparison.pdf', dpi=dpi)

### Comparing landscapes with different ruggedness metrics

In [ ]:
with open('processed_data/empirical_ruggedness_metric_comparison.pkl', 'rb') as f:
    decay_rate_measurements, roughness_to_slope_measurements, landscape_r2_measurements, local_epistasis_measurements, paths_to_max_measurements, local_max_measurements = pickle.load(f) 

In [ ]:
with open('processed_data/GB1_strategy_selection.pkl', 'rb') as f:
    _, _, gb1_decay_rate, gb1_sweep, _, _, _, _, _, _ = unpack_strategy_selection(pickle.load(f))

with open('processed_data/TrpB_strategy_selection.pkl', 'rb') as f:
    _, _, trpb_decay_rate, trpb_sweep, _, _, _, _, _, _ = unpack_strategy_selection(pickle.load(f))

with open('processed_data/TEV_strategy_selection.pkl', 'rb') as f:
    _, _, tev_decay_rate, tev_sweep, _, _, _, _, _, _ = unpack_strategy_selection(pickle.load(f))

with open('processed_data/ParD3_strategy_selection.pkl', 'rb') as f:
    _, _, pard3_decay_rate, pard3_sweep, _, _, _, _, _, _ = unpack_strategy_selection(pickle.load(f))


In [ ]:
with open('landscape_arrays/GB1_landscape_array.pkl', 'rb') as f:
    GB1 = pickle.load(f)

with open('landscape_arrays/E3_landscape_array.pkl', 'rb') as f:
    ParD3 = pickle.load(f)

with open('landscape_arrays/TEV_landscape_array.pkl', 'rb') as f:
    TEV = pickle.load(f)

with open('landscape_arrays/TrpB_landscape_array.pkl', 'rb') as f:
    TrpB = pickle.load(f)

In [ ]:
from scipy.stats import percentileofscore
#evolvability = [percentileofscore(sweep.mean(axis=(0,3)).flatten(), sweep.mean(axis=(0,3))[-1,0]) for sweep in [gb1_sweep, trpb_sweep, tev_sweep, pard3_sweep]]
#evolvability = [sweep.mean(axis=(0,3))[-1,0]/sweep.mean(axis=(0,3))[0,0] for sweep in [gb1_sweep, trpb_sweep, tev_sweep, pard3_sweep]]

def normalise_array(x):
    x = np.asarray(x, dtype=float)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)  # add epsilon to avoid div/0

evolvability = [normalise_array(sweep.mean(axis=(0,3)))[-1,0]/normalise_array(sweep.mean(axis=(0,3)))[0,-1] for sweep in [gb1_sweep, trpb_sweep, tev_sweep, pard3_sweep]]
#evolvability = [sweep.mean(axis=(0,3))[-1,0] for sweep in [gb1_sweep, trpb_sweep, tev_sweep, pard3_sweep]]
#evolvability = [i/ld.max() for i, ld in zip(evolvability, [GB1, TrpB, TEV, ParD3])]
#evolvability = [percentileofscore(ld.flatten(), i) for i, ld in zip(evolvability, [GB1, TrpB, TEV, ParD3])]


In [ ]:
decay_rate_measurements = [i[0]/2 for i in [gb1_decay_rate, trpb_decay_rate, tev_decay_rate, pard3_decay_rate]]

In [ ]:
local_epistasis_measurements = [i['simple_sign_episasis']+ i['reciprocal_sign_epistasis'] for i in local_epistasis_measurements]

In [ ]:
spectral_entropy_measurements = [get_spectral_entropy(l, remove_constant=True) for l in [GB1, TrpB, TEV, ParD3]]

In [ ]:
dirichlet_energy_measurements = [get_dirichlet_metric(l) for l in [GB1, TrpB, TEV, ParD3]]

In [ ]:
dirichlet_energy_measurements

In [ ]:
decay_rate_measurements

In [ ]:
def r_sigfig(value, sigfig=1):
    if value == 0:  
        return 0  # Special case: Zero remains zero
    
    return np.round(value, -int(np.floor(np.log10(abs(value)))) + (sigfig - 1))

colours = [c1, c2, c4, c3]
labels = ['GB1', 'TrpB', 'TEV', 'ParD3']
rank_labels = ["1st", "2nd", "3rd", "4th"]
markers = ['o', 's', '^', 'D']
reversed_axes = [2, 5, 6]  # these get lowest->"1st"
titles = [r'Decay rate $\rho$', r'Norm. Dirichlet Energy $\rho_{\Delta}$', r'Landscape $R^2$', r'Spectral Entropy $H$', r'Local epistasis $n_{\epsilon}$', r'Dist. to local max $d_{max}$', r'Paths to max $n_{max}$', r'Roughness to slope $r/s$']
arr = np.array([
    decay_rate_measurements, 
    np.array(dirichlet_energy_measurements),
    1-np.array(landscape_r2_measurements), 
    np.array(spectral_entropy_measurements), 
    local_epistasis_measurements,
    np.array(local_max_measurements),
    np.array(paths_to_max_measurements), 
    roughness_to_slope_measurements
])
norm_axes = [0,1,2,3]

fig, axes = plt.subplots(1, len(arr), figsize=(2+len(arr), 2), dpi=300)

for i, ax in enumerate(axes):

    y_positions = arr[i]                 # y-values for this metric (length 4)
    x_positions = np.full(4, 0.5)        # 4 points on x=0.5
    
    # Choose ranking order per axis
    order = np.argsort(y_positions) if i in reversed_axes else np.argsort(-y_positions)
    # Map each point j -> its rank index (0..3)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(len(y_positions))

    for j in range(4):
        jitter = np.random.uniform(-0.1, 0.1)
        jitter = 0.05 if j % 2 == 0 else -0.05
        x = x_positions[j] + jitter
        y = y_positions[j]

        ax.scatter(x, y,marker=markers[j],
                   color=colours[j], s=40, edgecolors='black', zorder=3,
                   label=labels[j] if i == 0 else None)
        # Rank label
        ax.text(x + 0.1 if j % 2 == 0 else x - 0.1, y, rank_labels[ranks[j]],
                fontsize=5, va='center', ha='left' if j % 2 == 0 else 'right', zorder=4)

    # Outline
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1)

    # Axes ticks
    ax.set_xticks([])
    if i in norm_axes:
        print(f"norm axes {titles[i]}")
        ax.set_ylim(0, 1)
        ax.set_yticks([0,1])
        ax.set_yticklabels([0,1], fontsize=5)
    else:
        ax.set_yticks([arr[i].min(), arr[i].max()])
        ax.set_yticklabels([r_sigfig(arr[i].min()), r_sigfig(arr[i].max())], fontsize=5)
    ax.set_xlim(0, 1)
    ax.margins(y=0.1)

    ax.set_title(titles[i], fontsize=titlesize-4.5)

# Legend
axes[0].legend(fontsize=legendsize, loc='upper left', bbox_to_anchor=(0.7+3+len(arr), 0.8))
#axes[1].set_yticklabels([r_sigfig(arr[1].min()), r_sigfig(arr[1].max())], fontsize=4)

# Invert y-axis where needed
for idx in reversed_axes:
    axes[idx].invert_yaxis()

# Adjust spacing
plt.subplots_adjust(wspace=0.5)

# Global label
fig.text(0.08, 0.5, r'Ruggedness (a.u.) $\rightarrow$', fontsize=8, va='center', rotation=90)

plt.savefig('figures/empirical_ruggedness_metric_comparison_IK.pdf', dpi=dpi)

In [ ]:
with open('processed_data/fourier_spectra_empirical.pkl', 'rb') as f:
    spectra = pickle.load(f)

In [ ]:
pard3_spectrum = np.asarray(spectra[3])
print(len(pard3_spectrum))
A = 20
N = 3
d = N * (A - 1)
spectrum = pard3_spectrum[1:]
indexes = np.arange(len(spectrum)) + 1
weighted_avg = np.sum(A * indexes * spectrum) / np.sum(spectrum) / d
print(weighted_avg)


In [ ]:
colours = [c1, c2, c4, c3]
labels = ['GB1', 'TrpB', 'TEV', 'ParD3']
markers = ['o', 's', '^', 'D']

plt.figure(figsize=(3.5, 3), dpi=300)
A = 20
empirical_dimensions = np.array([4, 4, 4, 3])
fourier_vals = []

for n, spectrum in enumerate(spectra):
    spectrum = np.asarray(spectrum)[1:]
    spectrum_norm = (spectrum - spectrum.min()) / (spectrum.max() - spectrum.min())
    indexes = np.arange(len(spectrum)) + 1
    d = empirical_dimensions[n] * (A - 1)
    weighted_avg = np.sum(A * indexes * spectrum) / np.sum(spectrum) / d
    frequency_centroid = weighted_avg * d / A
    plt.plot(
        indexes,
        spectrum_norm,
        label=labels[n],
        c=colours[n],
        marker=markers[n],
        markersize=4,
    )
    plt.axvline(x=frequency_centroid, c=colours[n], linestyle='--', alpha=0.5)
    fourier_vals.append(weighted_avg)
print(fourier_vals)
print(len(spectra))
plt.legend(fontsize=legendsize, loc='upper right')
plt.title('Empirical landscape fourier spectra', fontsize=titlesize)
plt.xlabel(r'Frequency index $i$', fontsize=labelsize)
plt.ylabel(r'Power spectral coefficient $b_i$', fontsize=labelsize)
plt.xticks(([1, 2, 3, 4]))
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.savefig('figures/empirical_fourier_spectra_IK.pdf', dpi=dpi)


In [ ]:
with open('processed_data/trajectory_subsampling.pkl', 'rb') as f:
    ld_results = pickle.load(f)

In [ ]:
with open('processed_data/trajectory_subsampling_IK.pkl', 'rb') as f:
    ld_results_IK = pickle.load(f)

In [ ]:
with open('processed_data/trajectory_subsampling_IK.pkl', 'rb') as f:
    ld_results_IK = pickle.load(f)

data_names = ['GB1', 'TrpB', 'TEV', 'ParD3']
colours = [c1,c2,c4,c3]
# trajectories = np.round(np.linspace(10,1000,10))
# trajectories = np.logspace(0,5,6)
trajectories = np.round(np.logspace(0,5,6))
plt.figure(figsize=(3.5,3), dpi=300)

for h in range(len(ld_results_IK)):

    means = []
    errs = []

    for t in range(len(trajectories)):
        vals = ld_results_IK[h][t]     # (n_boot,)
        means.append(np.mean(vals))
        errs.append(np.std(vals))   # or SEM: np.std(vals) / np.sqrt(len(vals))

    means = np.array(means)
    print(means)
    errs  = np.array(errs)

    plt.plot(
        trajectories,
        means,
        label=data_names[h],
        color = colours[h]
    )

    plt.fill_between(
        trajectories,
        means - errs,
        means + errs,
        alpha=0.25,
        color = colours[h],
        edgecolor=None
    )

    #plt.hlines(
    #    y=fourier_vals[h] xmin=0, xmax=1000, color=colours[h], linestyle='--', alpha=0.4)
    
    plt.hlines(
        y=fourier_vals[h], xmin=0, xmax=trajectories.max(), color=colours[h], linestyle='dotted',alpha=0.4)

plt.xlabel('Number of trajectories averaged', fontsize=labelsize)
plt.ylabel(r'Decay rate $\rho$', fontsize=labelsize)
plt.title(r'$\rho$ accuracy with increasing samples', fontsize=titlesize)
plt.legend(fontsize = legendsize)
plt.ylim(0,1.1)
plt.xscale("log")
plt.tick_params(axis='both', which='major', labelsize=ticksize)
# plt.savefig('figures/accuracy_over_sampling.pdf', dpi=dpi)

In [ ]:
data_names = ['GB1', 'TrpB', 'TEV', 'ParD3']
colours = [c1,c2,c4,c3]
trajectories = np.round(np.linspace(10,1000,10))
plt.figure(figsize=(3.5,3), dpi=300)

for h in range(len(ld_results)):

    means = []
    errs = []

    for t in range(len(trajectories)):
        vals = ld_results[h][t]     # (n_boot,)
        means.append(np.mean(vals))
        errs.append(np.std(vals))   # or SEM: np.std(vals) / np.sqrt(len(vals))

    means = np.array(means)
    errs  = np.array(errs)

    plt.plot(
        trajectories,
        means,
        label=data_names[h],
        color = colours[h]
    )

    plt.fill_between(
        trajectories,
        means - errs,
        means + errs,
        alpha=0.25,
        color = colours[h],
        edgecolor=None
    )

    #plt.hlines(
    #    y=fourier_vals[h] xmin=0, xmax=1000, color=colours[h], linestyle='--', alpha=0.4)
    
    plt.hlines(
        y=fourier_vals[h], xmin=0, xmax=1000, color=colours[h], linestyle='dotted',alpha=0.4)

plt.xlabel('Number of trajectories averaged', fontsize=labelsize)
plt.ylabel(r'Decay rate $\rho$', fontsize=labelsize)
plt.title(r'$\rho$ accuracy with increasing samples', fontsize=titlesize)
plt.legend(fontsize = legendsize)
plt.ylim(0,1.1)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
# plt.savefig('figures/accuracy_over_sampling.pdf', dpi=dpi)

In [ ]:
np.array(fourier_vals) / decay_rate_measurements


In [ ]:
plt.scatter(decay_rate_measurements, np.array(fourier_vals))
plt.ylabel(r'Analytical $\rho_2$ from Fourier spectrum')
plt.xlabel(r'Fitted $\rho_2$ from decay rate')


In [ ]:
with open('processed_data/heterogeneity_data.pkl', 'rb') as f:
    NK_rhos, empirical_rhos = pickle.load(f)

In [ ]:
len(empirical_rhos)

In [ ]:
max(empirical_rhos[2])

In [ ]:
plt.figure(figsize=(3.5, 3), dpi=300)

data = NK_rhos + empirical_rhos
var_data = [np.std(i) for i in data]

vp = plt.violinplot(
    data,
    showmeans=True,
    showextrema=True   # make sure extrema are enabled
)

# 🔧 Make the internal vertical bars thinner
vp['cbars'].set_linewidth(0.5)
vp['cmins'].set_linewidth(0.5)
vp['cmaxes'].set_linewidth(0.5)
vp['cmeans'].set_linewidth(0.5)

plt.xticks(
    [1, 2, 3, 4, 5, 6, 7, 8],
    ['0.25', '0.50', '0.75', '1.00', 'GB1', 'TrpB', 'TEV', 'PardD3'],
    fontsize=ticksize
)

plt.ylabel(r'$\rho$ Estimation', fontsize=labelsize)
plt.title('Landscape heterogeneity', fontsize=titlesize)

for i, var in enumerate(var_data):
    x = i + 1
    y = max(data[i]) * 1.05
    plt.text(
        x, y,
        f'$\sigma$={var:.2f}',
        ha='center',
        va='bottom',
        fontsize=4.5
    )

plt.axvline(x=4.5, color='gray', linestyle='--', linewidth=0.5)

plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.ylim(-0.1, 1.15)

plt.tight_layout()
# plt.savefig('figures/landscape_heterogeneity.pdf', dpi=dpi)


In [ ]:
plt.figure(figsize=(3.5, 3), dpi=300)

data = NK_rhos + empirical_rhos
var_data = [np.std(i) for i in data]

plt.violinplot(data, showmeans=True)

plt.xticks([1, 2, 3,4,5,6,7,8], ['0.25', '0.50', '0.75','1.00','GB1','TrpB','TEV', 'PardD3'], fontsize=ticksize)
plt.ylabel(r'$\rho$ Estimation', fontsize = labelsize)
plt.title('Landscape heterogeneity', fontsize=titlesize)

for i, var in enumerate(var_data):
    x = i + 1  # violin positions start at 1
    y = max(data[i]) * 1.05  # slightly above the top
    plt.text(x, y, f'$\sigma$={var:.2f}', ha='center', va='bottom', fontsize=4.5)

plt.axvline(x=4.5, color='gray', linestyle='--', linewidth=1)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.ylim(-0.1,1.15)
plt.tight_layout()
# # plt.savefig('figures/landscape_heterogeneity.pdf', dpi=dpi)

# Section 3 - Optimising directed evolution

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


### Optimal DE strategies from sweep

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
with open('processed_data/optimal_DE_strategies.pkl', 'rb') as f:
    decay_rates, optimal_splits, optimal_base_chances, optimal_strategy_params = unpack_optimal_strategies(pickle.load(f))


In [ ]:
with open('processed_data/strategy_prediction_accuracy.pkl', 'rb') as f:
    actual_k_over_ns, bc_means, bc_stds, sp_means, sp_stds = pickle.load(f)

In [ ]:
fig, ax1 = plt.subplots(figsize=(3.5,3), dpi=300)
strategy_total_popsize = optimal_strategy_params.get("popsize", 1200)

k_over_ns = np.unique(np.round(actual_k_over_ns,1))

color = c1
ax1.set_xlabel(r'True $\rho$', fontsize=8)
ax1.set_ylabel(r'Predicted base chance $b$', color=color, fontsize=8)
ax1.errorbar(k_over_ns, bc_means, yerr=bc_stds, fmt='o', capsize=5, color=c1)
ax1.plot(decay_rates, optimal_base_chances, color=color, linestyle='--', alpha=0.6, label='Optimal base chance')
ax1.tick_params(axis='y', labelcolor=color, labelsize=6)
ax1.tick_params(axis='x', labelsize=6)
ax1.grid(color='black', linestyle='--', linewidth=0.5, alpha=0.3)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = c2
ax2.set_ylabel(f'Predicted splitting (total {strategy_total_popsize})', color=color, fontsize=8) 
ax2.errorbar(k_over_ns, sp_means, yerr=sp_stds, fmt='o', capsize=5, color=c2)
ax2.plot(decay_rates, optimal_splits, color=color, linestyle='--', alpha=0.6, label='Optimal splitting')
ax2.tick_params(axis='y', labelcolor=color, labelsize=6)

ax1.set_title(r'Predicted DE strategies vs true $\rho$', fontsize=10)

# Adjust legend positions
ax1.legend(fontsize=legendsize-2, loc='upper left', bbox_to_anchor=(0.0, 1.0))
ax2.legend(fontsize=legendsize-2, loc='upper left', bbox_to_anchor=(0.0, 0.9))  # Move slightly below ax1's legend

plt.xlim(0.05, 1.05)
fig.tight_layout()
# # plt.savefig('figures/strategy_prediction.pdf', dpi=dpi)


### Directed evolution on smooth NK

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
with open('processed_data/NK_DE.pkl', 'rb') as f:
    DE_data = pickle.load(f)

with open('processed_data/NK_strategy_spaces.pkl', 'rb') as f:
    smooth_strategies, rugged_strategies, nk_strategy_space_params = unpack_strategy_spaces(pickle.load(f))


In [ ]:
plt.figure(figsize=(3,3), dpi=300)
plt.plot(DE_data[0], label = 'Baseline strategy', color=c1)
plt.plot(DE_data[2], label = 'SLIDE', color=c2)
plt.ylabel('Arbitrary fitness scale', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title('N = 45, K = 1',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
# plt.savefig('figures/N45K1_DE_fitness.pdf', dpi=dpi)

In [ ]:
strategy_base_chances = np.asarray(nk_strategy_space_params.get("base_chances", [0.0, 0.19]))
strategy_splits = list(nk_strategy_space_params.get("splits", [24, 20, 16, 12, 8, 4, 1]))
plt.figure(figsize = (1.5,1.5), dpi=300)
plt.imshow(smooth_strategies.mean(axis=1).reshape(7,7))
plt.xticks([0, len(strategy_base_chances) - 1], labels=[float(strategy_base_chances[0]), float(strategy_base_chances[-1])])
plt.yticks([0, len(strategy_splits) - 1], labels=[strategy_splits[0], strategy_splits[-1]])
plt.ylabel('No. sub populations', fontsize=8)
plt.xlabel('Base chance', fontsize=8)
plt.title('Strategy space')
plt.tick_params(axis='both', which='major', labelsize=6)
# plt.savefig('figures/N45K1_DE_strategy_space.pdf', dpi=dpi)


### Directed evolution on rough NK

Paper reference: Figure 3 (NK ruggedness inference and robustness checks).

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
plt.figure(figsize=(3,3), dpi=300)
plt.plot(DE_data[1], label = 'Baseline strategy', color=c1)
plt.plot(DE_data[3], label = 'SLIDE', color=c2)
plt.ylabel('Arbitrary fitness scale', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title('N = 45, K = 25',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
# plt.savefig('figures/N45K25_DE_fitness.pdf', dpi=dpi)

In [ ]:
strategy_base_chances = np.asarray(nk_strategy_space_params.get("base_chances", [0.0, 0.19]))
strategy_splits = list(nk_strategy_space_params.get("splits", [24, 20, 16, 12, 8, 4, 1]))
plt.figure(figsize = (1.5,1.5), dpi=300)
plt.imshow(rugged_strategies.mean(axis=1).reshape(7,7))
plt.xticks([0, len(strategy_base_chances) - 1], labels=[float(strategy_base_chances[0]), float(strategy_base_chances[-1])])
plt.yticks([0, len(strategy_splits) - 1], labels=[strategy_splits[0], strategy_splits[-1]])
plt.ylabel('No. sub populations', fontsize=8)
plt.xlabel('Base chance', fontsize=8)
plt.title('Strategy space')
plt.tick_params(axis='both', which='major', labelsize=6)
# plt.savefig('figures/N45K25_DE_strategy_space.pdf', dpi=dpi)


### GB1 directed evolution

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
def normalise_decay(y_vals, constant):
    out = y_vals - constant
    return out/out[0]

In [ ]:
with open('processed_data/GB1_strategy_selection.pkl', 'rb') as f:
    x_vals, decay_mean, decay_rate, sweep, scipy_freq_matrix, run, scatter, line, GB1_decay_multi, strategy_selection_params = unpack_strategy_selection(pickle.load(f))


In [ ]:
plt.figure(figsize=(3, 1.5), dpi=300)
plt.scatter(x_vals, scatter, label='Mean fitness', s=15)
plt.plot(x_vals, line, label='Fit')
plt.ylabel('Fitness relative to WT', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title(f'GB1 fitness decay, $\\rho$ = {np.around(decay_rate[0]/2,2)}',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
plt.ylim(0,1.1)
plt.show()

# plt.savefig('figures/GB1_decay.pdf', dpi=dpi)


In [ ]:
plt.figure(figsize=(1.5, 1.5), dpi=300)
mean_sweep = sweep.mean(axis=(0,3))
plt.imshow(mean_sweep)

# Add scatter plot with square markers proportional to frequency
max_size = 100  # Adjust as needed
dot_sizes = (scipy_freq_matrix / scipy_freq_matrix.max()) * max_size

for i in range(7):  # Loop over rows
    for j in range(7):  # Loop over columns
        if scipy_freq_matrix[i, j] > 0:  # Only plot if frequency > 0
            plt.scatter(
                j, i,
                s=dot_sizes[i, j]*0.2,
                color=c2,
                alpha=1,
                marker='o',
                label='SLIDE'
            )

# Baseline as a square
plt.scatter(0,6, s=20, color=c1, alpha=1, marker='o', label='Baseline')

# ---- Optimal strategy as a red square ----
max_idx = np.unravel_index(np.argmax(mean_sweep, axis=None), mean_sweep.shape)
optimal_i, optimal_j = max_idx
plt.scatter(
    optimal_j, optimal_i,
    s=30,              # slightly larger to stand out
    color='red',
    alpha=1,
    marker='o',
    linewidth=0.5,
    label='Optimum'
)

# Formatting
plt.xticks([0, 6], labels=[0.0, 0.19])
plt.yticks([0, 6], labels=[24, 1])
plt.ylabel('No. sub populations', fontsize=labelsize - 1)
plt.xlabel('Base chance', fontsize=labelsize - 1)
plt.title('Relative strategy performance', fontsize=titlesize - 2)
plt.tick_params(axis='both', which='major', labelsize=5)

# Legend outside, smaller
plt.legend(
    fontsize=6,
    loc='center left',
    bbox_to_anchor=(1.07, 0.5)
)

# plt.savefig('figures/GB1_strategy_space.pdf', dpi=dpi, bbox_inches='tight')
plt.show()


In [ ]:
# --- Compute mean over reps first ---
run_startmean = [
    [
        np.mean(strategy, axis=0)  # mean over reps for each strategy
        for strategy in start
    ]
    for start in run
]

n_strat = len(run_startmean[0])
n_steps = max(len(start[0]) for start in run_startmean)  # max number of steps across starts

strategy_mean = []
strategy_std  = []

# --- Compute mean/std across starts ---
for s in range(n_strat):
    # Gather the mean-over-reps trajectories for this strategy
    start_vals = [start[s] for start in run_startmean]  # list of arrays (steps)
    
    # Pad sequences to the same length with NaN (optional)
    max_len = max(len(traj) for traj in start_vals)
    start_vals_pad = [list(traj) + [np.nan]*(max_len - len(traj)) for traj in start_vals]

    # Compute mean and std at each step, ignoring NaNs
    mean_vals = [np.nanmean([traj[step] for traj in start_vals_pad]) for step in range(max_len)]
    std_vals  = [np.nanstd([traj[step] for traj in start_vals_pad], ddof=1) for step in range(max_len)]
    
    strategy_mean.append(mean_vals)
    strategy_std.append(std_vals)

# --- Plotting ---
plt.figure(figsize=(4, 1.5), dpi=300)
steps = np.arange(n_steps)
colors = [c1, c2]
labels = ['Baseline', 'SLIDE']

for s in range(n_strat):
    plt.plot(steps, strategy_mean[s], label=labels[s], c=colors[s])
    plt.fill_between(
        steps,
        [m - std for m, std in zip(strategy_mean[s], strategy_std[s])],
        [m + std for m, std in zip(strategy_mean[s], strategy_std[s])],
        alpha=0.3,
        color=colors[s],
        linewidth=0
    )

plt.ylabel('Fitness relative to WT', fontsize=labelsize)
plt.xlabel(r'Generations $M$', fontsize=labelsize)
plt.title('GB1 directed evolution, 10 start average', fontsize=titlesize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.legend(fontsize=8)
# plt.savefig('figures/GB1_DE.pdf', dpi=dpi)
plt.show()


In [ ]:
from scipy.stats import ttest_ind

baseline_final = []
slide_final = []

for start in run_startmean:
    # Strategy 0 = Baseline
    traj0 = start[0]
    if len(traj0) > 0:
        val0 = np.mean(traj0[-1]) if hasattr(traj0[-1], '__iter__') else traj0[-1]
        baseline_final.append(val0)

    # Strategy 1 = SLIDE
    traj1 = start[1]
    if len(traj1) > 0:
        val1 = np.mean(traj1[-1]) if hasattr(traj1[-1], '__iter__') else traj1[-1]
        slide_final.append(val1)

# Convert to 1D float arrays
baseline_final = np.array(baseline_final, dtype=float)
slide_final    = np.array(slide_final, dtype=float)

# Perform Welch's t-test
t_stat, p_value = ttest_ind(baseline_final, slide_final, equal_var=False)

print(f"Final generation comparison:")
print(f"Baseline mean = {baseline_final.mean():.4f}, SLIDE mean = {slide_final.mean():.4f}")
print(f"t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")



### TrpB Directed Evolution

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
with open('processed_data/TrpB_strategy_selection.pkl', 'rb') as f:
    x_vals, decay_mean, decay_rate, sweep, scipy_freq_matrix, run, scatter, line, TrpB_decay_multi, strategy_selection_params = unpack_strategy_selection(pickle.load(f))


In [ ]:
plt.figure(figsize=(3, 1.5), dpi=300)
plt.scatter(x_vals, scatter, label='Mean fitness', s=15)
plt.plot(x_vals, line, label='Fit')
plt.ylabel('Fitness relative to WT', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title(f'TrpB fitness decay, $\\rho$ = {np.around(decay_rate[0]/2,2)}',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
plt.ylim(0,1.1)
plt.show()

# plt.savefig('figures/TrpB_decay.pdf', dpi=dpi)


In [ ]:
plt.figure(figsize=(1.5, 1.5), dpi=300)
mean_sweep = sweep.mean(axis=(0,3))
plt.imshow(mean_sweep)

# Add scatter plot with square markers proportional to frequency
max_size = 100  # Adjust as needed
dot_sizes = (scipy_freq_matrix / scipy_freq_matrix.max()) * max_size

for i in range(7):  # Loop over rows
    for j in range(7):  # Loop over columns
        if scipy_freq_matrix[i, j] > 0:  # Only plot if frequency > 0
            plt.scatter(
                j, i,
                s=dot_sizes[i, j]*0.2,
                color=c2,
                alpha=1,
                marker='o',
                label='SLIDE'
            )

# Baseline as a square
plt.scatter(0,6, s=20, color=c1, alpha=1, marker='o', label='Baseline')

# ---- Optimal strategy as a red square ----
max_idx = np.unravel_index(np.argmax(mean_sweep, axis=None), mean_sweep.shape)
optimal_i, optimal_j = max_idx
plt.scatter(
    optimal_j, optimal_i,
    s=30,              # slightly larger to stand out
    color='red',
    alpha=1,
    marker='o',
    linewidth=0.5,
    label='Optimum'
)

# Formatting
plt.xticks([0, 6], labels=[0.0, 0.19])
plt.yticks([0, 6], labels=[24, 1])
plt.ylabel('No. sub populations', fontsize=labelsize - 1)
plt.xlabel('Base chance', fontsize=labelsize - 1)
plt.title('Relative strategy performance', fontsize=titlesize - 2)
plt.tick_params(axis='both', which='major', labelsize=5)

# Legend outside, smaller
plt.legend(
    fontsize=6,
    loc='center left',
    bbox_to_anchor=(1.07, 0.5)
)

# plt.savefig('figures/TrpB_strategy_space.pdf', dpi=dpi, bbox_inches='tight')
plt.show()


In [ ]:
# --- Compute mean over reps first ---
run_startmean = [
    [
        np.mean(strategy, axis=0)  # mean over reps for each strategy
        for strategy in start
    ]
    for start in run
]

n_strat = len(run_startmean[0])
n_steps = max(len(start[0]) for start in run_startmean)  # max number of steps across starts

strategy_mean = []
strategy_std  = []

# --- Compute mean/std across starts ---
for s in range(n_strat):
    # Gather the mean-over-reps trajectories for this strategy
    start_vals = [start[s] for start in run_startmean]  # list of arrays (steps)
    
    # Pad sequences to the same length with NaN (optional)
    max_len = max(len(traj) for traj in start_vals)
    start_vals_pad = [list(traj) + [np.nan]*(max_len - len(traj)) for traj in start_vals]

    # Compute mean and std at each step, ignoring NaNs
    mean_vals = [np.nanmean([traj[step] for traj in start_vals_pad]) for step in range(max_len)]
    std_vals  = [np.nanstd([traj[step] for traj in start_vals_pad], ddof=1) for step in range(max_len)]
    
    strategy_mean.append(mean_vals)
    strategy_std.append(std_vals)

# --- Plotting ---
plt.figure(figsize=(4, 1.5), dpi=300)
steps = np.arange(n_steps)
colors = [c1, c2]
labels = ['Baseline', 'SLIDE']

for s in range(n_strat):
    plt.plot(steps, strategy_mean[s], label=labels[s], c=colors[s])
    plt.fill_between(
        steps,
        [m - std for m, std in zip(strategy_mean[s], strategy_std[s])],
        [m + std for m, std in zip(strategy_mean[s], strategy_std[s])],
        alpha=0.3,
        color=colors[s],
        linewidth=0
    )

plt.ylabel('Fitness relative to WT', fontsize=labelsize)
plt.xlabel(r'Generations $M$', fontsize=labelsize)
plt.title('TrpB directed evolution, 10 start average', fontsize=titlesize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.legend(fontsize=8)
# plt.savefig('figures/TrpB_DE.pdf', dpi=dpi)
plt.show()


In [ ]:
from scipy.stats import ttest_ind

baseline_final = []
slide_final = []

for start in run_startmean:
    # Strategy 0 = Baseline
    traj0 = start[0]
    if len(traj0) > 0:
        val0 = np.mean(traj0[-1]) if hasattr(traj0[-1], '__iter__') else traj0[-1]
        baseline_final.append(val0)

    # Strategy 1 = SLIDE
    traj1 = start[1]
    if len(traj1) > 0:
        val1 = np.mean(traj1[-1]) if hasattr(traj1[-1], '__iter__') else traj1[-1]
        slide_final.append(val1)

# Convert to 1D float arrays
baseline_final = np.array(baseline_final, dtype=float)
slide_final    = np.array(slide_final, dtype=float)

# Perform Welch's t-test
t_stat, p_value = ttest_ind(baseline_final, slide_final, equal_var=False)

print(f"Final generation comparison:")
print(f"Baseline mean = {baseline_final.mean():.4f}, SLIDE mean = {slide_final.mean():.4f}")
print(f"t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

### TEV Directed Evolution

Paper reference: Figure 5 (SLIDE-informed directed-evolution strategies).


In [ ]:
with open('processed_data/TEV_strategy_selection.pkl', 'rb') as f:
    x_vals, decay_mean, decay_rate, sweep, scipy_freq_matrix, run, scatter, line, TEV_decay_multi, strategy_selection_params = unpack_strategy_selection(pickle.load(f))


In [ ]:
plt.figure(figsize=(3, 1.5), dpi=300)
plt.scatter(x_vals, scatter, label='Mean fitness', s=15)
plt.plot(x_vals, line, label='Fit')
plt.ylabel('Fitness relative to WT', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title(f'TEV fitness decay, $\\rho$ = {np.around(decay_rate[0]/2,2)}',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
plt.ylim(0,1.1)
plt.show()

# plt.savefig('figures/TEV_decay.pdf', dpi=dpi)

In [ ]:
plt.figure(figsize=(1.5, 1.5), dpi=300)
mean_sweep = sweep.mean(axis=(0,3))
plt.imshow(mean_sweep)

# Add scatter plot with square markers proportional to frequency
max_size = 100  # Adjust as needed
dot_sizes = (scipy_freq_matrix / scipy_freq_matrix.max()) * max_size

for i in range(7):  # Loop over rows
    for j in range(7):  # Loop over columns
        if scipy_freq_matrix[i, j] > 0:  # Only plot if frequency > 0
            plt.scatter(
                j, i,
                s=dot_sizes[i, j]*0.2,
                color=c2,
                alpha=1,
                marker='o',
                label='SLIDE'
            )

# Baseline as a square
plt.scatter(0,6, s=20, color=c1, alpha=1, marker='o', label='Baseline')

# ---- Optimal strategy as a red square ----
max_idx = np.unravel_index(np.argmax(mean_sweep, axis=None), mean_sweep.shape)
optimal_i, optimal_j = max_idx
plt.scatter(
    optimal_j, optimal_i,
    s=30,              # slightly larger to stand out
    color='red',
    alpha=1,
    marker='o',
    linewidth=0.5,
    label='Optimum'
)

# Formatting
plt.xticks([0, 6], labels=[0.0, 0.19])
plt.yticks([0, 6], labels=[24, 1])
plt.ylabel('No. sub populations', fontsize=labelsize - 1)
plt.xlabel('Base chance', fontsize=labelsize - 1)
plt.title('Relative strategy performance', fontsize=titlesize - 2)
plt.tick_params(axis='both', which='major', labelsize=5)

# Legend outside, smaller
plt.legend(
    fontsize=6,
    loc='center left',
    bbox_to_anchor=(1.07, 0.5)
)

# plt.savefig('figures/TEV_strategy_space.pdf', dpi=dpi, bbox_inches='tight')
plt.show()


In [ ]:
# --- Compute mean over reps first ---
run_startmean = [
    [
        np.mean(strategy, axis=0)  # mean over reps for each strategy
        for strategy in start
    ]
    for start in run
]

n_strat = len(run_startmean[0])
n_steps = max(len(start[0]) for start in run_startmean)  # max number of steps across starts

strategy_mean = []
strategy_std  = []

# --- Compute mean/std across starts ---
for s in range(n_strat):
    # Gather the mean-over-reps trajectories for this strategy
    start_vals = [start[s] for start in run_startmean]  # list of arrays (steps)
    
    # Pad sequences to the same length with NaN (optional)
    max_len = max(len(traj) for traj in start_vals)
    start_vals_pad = [list(traj) + [np.nan]*(max_len - len(traj)) for traj in start_vals]

    # Compute mean and std at each step, ignoring NaNs
    mean_vals = [np.nanmean([traj[step] for traj in start_vals_pad]) for step in range(max_len)]
    std_vals  = [np.nanstd([traj[step] for traj in start_vals_pad], ddof=1) for step in range(max_len)]
    
    strategy_mean.append(mean_vals)
    strategy_std.append(std_vals)

# --- Plotting ---
plt.figure(figsize=(4, 1.5), dpi=300)
steps = np.arange(n_steps)
colors = [c1, c2]
labels = ['Baseline', 'SLIDE']

for s in range(n_strat):
    plt.plot(steps, strategy_mean[s], label=labels[s], c=colors[s])
    plt.fill_between(
        steps,
        [m - std for m, std in zip(strategy_mean[s], strategy_std[s])],
        [m + std for m, std in zip(strategy_mean[s], strategy_std[s])],
        alpha=0.3,
        color=colors[s],
        linewidth=0
    )

plt.ylabel('Fitness relative to WT', fontsize=labelsize)
plt.xlabel(r'Generations $M$', fontsize=labelsize)
plt.title('TrpB directed evolution, 10 start average', fontsize=titlesize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.legend(fontsize=8)
# plt.savefig('figures/TEV_DE.pdf', dpi=dpi)
plt.show()


In [ ]:
from scipy.stats import ttest_ind

baseline_final = []
slide_final = []

for start in run_startmean:
    # Strategy 0 = Baseline
    traj0 = start[0]
    if len(traj0) > 0:
        val0 = np.mean(traj0[-1]) if hasattr(traj0[-1], '__iter__') else traj0[-1]
        baseline_final.append(val0)

    # Strategy 1 = SLIDE
    traj1 = start[1]
    if len(traj1) > 0:
        val1 = np.mean(traj1[-1]) if hasattr(traj1[-1], '__iter__') else traj1[-1]
        slide_final.append(val1)

# Convert to 1D float arrays
baseline_final = np.array(baseline_final, dtype=float)
slide_final    = np.array(slide_final, dtype=float)

# Perform Welch's t-test
t_stat, p_value = ttest_ind(baseline_final, slide_final, equal_var=False)

print(f"Final generation comparison:")
print(f"Baseline mean = {baseline_final.mean():.4f}, SLIDE mean = {slide_final.mean():.4f}")
print(f"t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

## ParD3

In [ ]:
with open('processed_data/ParD3_strategy_selection.pkl', 'rb') as f:
    x_vals, decay_mean, decay_rate, sweep, scipy_freq_matrix, run, scatter, line, ParD3_decay_multi, strategy_selection_params = unpack_strategy_selection(pickle.load(f))


In [ ]:
plt.figure(figsize=(3, 1.5), dpi=300)
plt.scatter(x_vals, scatter, label='Mean fitness', s=15)
plt.plot(x_vals, line, label='Fit')
plt.ylabel('Fitness relative to WT', fontsize=8)
plt.xlabel(r'Generations $M$', fontsize=8)
plt.title(f'ParD3 fitness decay, $\\rho$ = {np.around(decay_rate[0]/2,2)}',fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=6)
plt.legend(fontsize=8)
plt.ylim(0,1.1)
plt.show()

# plt.savefig('figures/ParD3_decay.pdf', dpi=dpi)

In [ ]:

plt.figure(figsize=(1.5, 1.5), dpi=300)
mean_sweep = sweep.mean(axis=(0,3))
plt.imshow(mean_sweep)

# --- Add scatter plot with square markers proportional to frequency ---
max_size = 100  # Adjust as needed
dot_sizes = (scipy_freq_matrix / scipy_freq_matrix.max()) * max_size

for i in range(5):  # rows
    for j in range(5):  # columns
        if scipy_freq_matrix[i, j] > 0:
            plt.scatter(
                j - 0.2, i,                 # offset left
                s=dot_sizes[i, j] * 0.2,
                color=c2,
                alpha=1,
                marker='o',
                label='SLIDE'  # avoid duplicate labels
            )

# Baseline marker (offset right)
plt.scatter(
    0 + 0.2, 4, s=20, color=c1, alpha=1, marker='o', label='Baseline'
)

# --- Optimal strategy marker as red circle ---
max_idx = np.unravel_index(np.argmax(mean_sweep, axis=None), mean_sweep.shape)
optimal_i, optimal_j = max_idx
plt.scatter(
    optimal_j, optimal_i,
    s=30,              # slightly larger to stand out
    color='red',
    alpha=1,
    marker='o',
    linewidth=0.5,
    label='Optimal'
)

# --- Formatting ---
plt.xticks([0, 4], labels=[0.0, 0.19])
plt.yticks([0, 4], labels=[20, 1])
plt.ylabel('No. sub populations', fontsize=labelsize - 1)
plt.xlabel('Base chance', fontsize=labelsize - 1)
plt.title('Strategy space', fontsize=titlesize - 1)
plt.tick_params(axis='both', which='major', labelsize=5)

# --- Legend outside, smaller ---
plt.legend(
    fontsize=6,
    loc='center left',
    bbox_to_anchor=(1.07, 0.5)
)

# plt.savefig('figures/ParD3_strategy_space.pdf', dpi=dpi, bbox_inches='tight')
plt.show()


In [ ]:
# --- Compute mean over reps first ---
run_startmean = [
    [
        np.mean(strategy, axis=0)  # mean over reps for each strategy
        for strategy in start
    ]
    for start in run
]

n_strat = len(run_startmean[0])
n_steps = max(len(start[0]) for start in run_startmean)  # max number of steps across starts

strategy_mean = []
strategy_std  = []

# --- Compute mean/std across starts ---
for s in range(n_strat):
    # Gather the mean-over-reps trajectories for this strategy
    start_vals = [start[s] for start in run_startmean]  # list of arrays (steps)
    
    # Pad sequences to the same length with NaN (optional)
    max_len = max(len(traj) for traj in start_vals)
    start_vals_pad = [list(traj) + [np.nan]*(max_len - len(traj)) for traj in start_vals]

    # Compute mean and std at each step, ignoring NaNs
    mean_vals = [np.nanmean([traj[step] for traj in start_vals_pad]) for step in range(max_len)]
    std_vals  = [np.nanstd([traj[step] for traj in start_vals_pad], ddof=1) for step in range(max_len)]
    
    strategy_mean.append(mean_vals)
    strategy_std.append(std_vals)

# --- Plotting ---
plt.figure(figsize=(4, 1.5), dpi=300)
steps = np.arange(n_steps)
colors = [c1, c2]
labels = ['Baseline', 'SLIDE']
adjust = [0, -0.005]

for s in range(n_strat):
    plt.plot(steps, np.array(strategy_mean[s])+adjust[s], label=labels[s], c=colors[s])
    plt.fill_between(
        steps,
        [m - std for m, std in zip(strategy_mean[s], strategy_std[s])],
        [m + std for m, std in zip(strategy_mean[s], strategy_std[s])],
        alpha=0.3,
        color=colors[s],
        linewidth=0
    )

plt.ylabel('Fitness relative to WT', fontsize=labelsize)
plt.xlabel(r'Generations $M$', fontsize=labelsize)
plt.title('TrpB directed evolution, 10 start average', fontsize=titlesize)
plt.tick_params(axis='both', which='major', labelsize=ticksize)
plt.legend(fontsize=8)
# plt.savefig('figures/TEV_DE.pdf', dpi=dpi)
plt.show()


In [ ]:
from scipy.stats import ttest_ind

baseline_final = []
slide_final = []

for start in run_startmean:
    # Strategy 0 = Baseline
    traj0 = start[0]
    if len(traj0) > 0:
        val0 = np.mean(traj0[-1]) if hasattr(traj0[-1], '__iter__') else traj0[-1]
        baseline_final.append(val0)

    # Strategy 1 = SLIDE
    traj1 = start[1]
    if len(traj1) > 0:
        val1 = np.mean(traj1[-1]) if hasattr(traj1[-1], '__iter__') else traj1[-1]
        slide_final.append(val1)

# Convert to 1D float arrays
baseline_final = np.array(baseline_final, dtype=float)
slide_final    = np.array(slide_final, dtype=float)

# Perform Welch's t-test
t_stat, p_value = ttest_ind(baseline_final, slide_final, equal_var=False)

print(f"Final generation comparison:")
print(f"Baseline mean = {baseline_final.mean():.4f}, SLIDE mean = {slide_final.mean():.4f}")
print(f"t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

### Fourier Methods Figure

Paper reference: Figure 4 (empirical landscape ruggedness and heterogeneity).


In [ ]:
with open('processed_data/fourier_analysis.pkl', 'rb') as f:
    nk_data = pickle.load(f)
    
def get_exp_matrix(
    N: int,
    A: int,
    mutations: np.array,
    is_squared: bool = True,
    fix_b0: bool = False,
) -> np.array:
    eigrange = range(N+1) if fix_b0 == False else range(1, N+1)
    eigenvalues = [A*i for i in eigrange]
    factor = 2 if is_squared is True else 1
    if fix_b0:
        return np.array([[np.exp(-mut/(N*(A-1))*l*factor)-1 for l in eigenvalues] for mut in mutations[1:]])
    else:
        return np.array([[np.exp(-mut/(N*(A-1))*l*factor) for l in eigenvalues] for mut in mutations])


## Measure decay rate function.
def get_fourier_coeffs(
    mean_fitness: np.array,
    mutations: np.array,
    N: int,
    A: int,
    is_squared: bool = False,
    fix_b0: bool = False,
    method: str = "nnls",
    alpha: str = 0.1,
) -> tuple[np.array, np.array]:
    """
    Inputs:
    mean_fitness: vector with mean fitness values
    mutations: vector with number of mutations, starting with 0
    N: length of gene
    A: number of alleles (e.g., 20 if amino acids)
    is_squared: flag to set true if mean fitness squared is provided instead.
    method: pick on of ls, ls_constrained, nnls. Results may vary. Latter to enforce positiveness of the coeffs, although positiveness is only true for the squared estimate.
    Outputs:
    (fourier_coeffs, exponentials): tuple[np.array, np.array]
    fourier_coeffs: Vector of length N+1 with Fourier coeffs from low to high frequency, with the first element corresponding to the constant.
    exponentials: len(mutations)xN+1 matrix to extract fit via exponentials*weights
    """
    exponentials = get_exp_matrix(N=N, A=A, mutations=mutations, is_squared=is_squared, fix_b0=fix_b0)
    if fix_b0:
        mean_fitness_0 = mean_fitness[0]
        mean_fitness = mean_fitness[1:] - mean_fitness_0
    if method == "ls":
        fourier_coeffs, residuals, rank, s = np.linalg.lstsq(exponentials, mean_fitness, rcond=None)
    elif method == "ls_constrained":
        res = scipy.optimize.lsq_linear(exponentials, mean_fitness, bounds=(0, np.inf))
        fourier_coeffs = res.x
    elif method == "nnls":
        fourier_coeffs, rnorm = scipy.optimize.nnls(exponentials, mean_fitness)
    elif method == "nnls_reg":
        if alpha < 0:
            raise ValueError("alpha must be >= 0")
        m, p = exponentials.shape
        A_aug = np.vstack([exponentials, np.sqrt(alpha) * np.eye(p)])
        b_aug = np.concatenate([mean_fitness, np.zeros(p)])
        fourier_coeffs, _ = scipy.optimize.nnls(A_aug, b_aug)
    else:
        raise ValueError("Method unavailable.")
    if fix_b0:
        b0 = mean_fitness_0 - np.sum(np.abs(fourier_coeffs))
        fourier_coeffs = np.concatenate((np.array([b0]), fourier_coeffs))
        exponentials = get_exp_matrix(N=N, A=A, mutations=mutations, is_squared=is_squared, fix_b0=False)
    return (fourier_coeffs, exponentials)

def model_function(x,*params):

    """
    This is the what we are fitting to (sum of exponentials).
    It assumes a decay rate of 0.5 mutations per step.

    x = steps
    params = the output of the fitting function (get_single_decay_rate).
    """

    mut = 0.5
    num_params = 1
    constant = params[-1]
    params = params[:-1]
    mut_curves = np.exp(-1.0*mut*x[:,None]*np.array(params)[None,:])
    weights = np.linspace(0.1, 0.9, num_params)
    weights = np.ones(num_params)
    weights = weights / weights.sum()
    sum_curves = np.sum(mut_curves * weights[None,:], axis = 1)
    return sum_curves * (1 - constant) + constant

## Measure decay rate function.

def get_single_decay_rate(decay_data, mut = 0.5, num_steps = 25):

    num_params = 1
    decay_data = decay_data/decay_data[0]

    if isinstance(mut, (int, float, complex)) or jnp.ndim(mut) == 0:
        steps = np.linspace(0,num_steps-1,num_steps)
    else:
        steps = mut

    if mut is None:
        mut = np.arange(len(decay_data))  # Default steps
    
    init_guess = np.linspace(0.1, 0.9, num_params)
    init_guess = np.concat([init_guess,[0.0]])
    lbounds = [0.0]*num_params + [-0.4]
    ubounds = [2.0]*num_params + [0.4]

    # From chatgpt
    #asymptote_guess = decay_data[-3:].mean() / decay_data[0]
    #lower_bound = max(0.0, asymptote_guess - 0.2)
    #upper_bound = min(1.1, asymptote_guess + 0.2)

    #init_guess = np.concatenate([np.linspace(0.1, 0.9, num_params), [asymptote_guess]])
    #lbounds = [0.0]*num_params + [lower_bound]
    #ubounds = [2.0]*num_params + [upper_bound]

    params, _ = curve_fit(model_function, steps, decay_data,p0=init_guess, maxfev= 9000, ftol = 1e-4, xtol = 1e-5, bounds = (lbounds, ubounds))

    mean_params = np.mean(params[:-1])
    fitted_constant = params[-1]  # The second returned parameter

    return mean_params, fitted_constant  # Return full params for plotting

def get_single_decay_rate_NEW(decay_data, mut = 0.1, num_steps = 25):

    num_params = 2
    decay_data = decay_data/decay_data[0]
    def model_function(x,*params):
        mut_curves = np.exp(-1.0*mut*x[:,None]*np.array(params)[None,:])
        weights = np.linspace(0.1, 0.9, num_params)
        weights = np.ones(num_params)
        weights = weights / weights.sum()
        sum_curves = np.sum(mut_curves * weights[None,:], axis = 1)
        return sum_curves
    
    steps = np.linspace(0,num_steps-1,num_steps)

    init_guess = np.linspace(0.1, 0.9, num_params)
    params, _ = curve_fit(model_function, steps, decay_data,p0=init_guess, maxfev= 5000)

    mean_params = np.mean(params)
    return mean_params

In [ ]:
N = nk_data['N_used']
K_vec = nk_data['Ks_used']
A = nk_data['A_used']
NK_landscapes = nk_data['nk_builts']
NK_spectra = []
NK_max = [(K+1)*(A-1)/A for K in K_vec]
for f in NK_landscapes:
    NK_spectra.append(get_landscape_spectrum(f, norm = True, remove_constant = False, on_gpu = True))
    
mut = 0.5
mutations = np.arange(start=0, stop=5.5, step=mut)
num_steps = len(mutations)
exponentials = get_exp_matrix(N=N, A=A, mutations=mutations, is_squared=True)
fitness_decay = []
fitness_decay_terms = []
fitted_decay = []
fitted_rho = []
fitted_rho_NEW = []
for i, K in enumerate(K_vec):
    fitness_decay.append(np.dot(exponentials, NK_spectra[i]))
    fitness_decay[i] = fitness_decay[i]# /fitness_decay[i][0]
    fitness_decay_terms_K = np.zeros(exponentials.shape)
    for j in range(exponentials.shape[1]):
        fitness_decay_terms_K[:, j] = exponentials[:, j] * NK_spectra[i][j]
    fitness_decay_terms.append(fitness_decay_terms_K)
    
    estim_data = get_single_decay_rate(fitness_decay[i], mut = mut, num_steps = num_steps)
    fitted_decay.append(np.array([np.exp(-mutations*(estim_data[0]))*(1 - estim_data[-1])]))
    fitted_rho.append(estim_data[0])
    fitted_rho_NEW.append(get_single_decay_rate_NEW(fitness_decay[i], mut = mut, num_steps = num_steps))

In [ ]:
# Set globally
rcParams['font.family'] = 'Open Sans'

N = nk_data['N_used']
K_vec = nk_data['Ks_used']
A = nk_data['A_used']
NK_landscapes = nk_data['nk_builts']
NK_spectra = []
NK_max = [(K+1)*(A-1)/A for K in K_vec]
for f in NK_landscapes:
    NK_spectra.append(get_landscape_spectrum(f, norm = True, remove_constant = False, on_gpu = True))
    
mut = 0.5
mutations = np.arange(start=0, stop=5.5, step=mut)
num_steps = len(mutations)
exponentials = get_exp_matrix(N=N, A=A, mutations=mutations, is_squared=True)
fitness_decay = []
fitness_decay_terms = []
fitted_decay = []
fitted_rho = []
for i, K in enumerate(K_vec):
    fitness_decay.append(np.dot(exponentials, NK_spectra[i]))
    tmp = 1 # fitness_decay[i][0]
    fitness_decay[i] = fitness_decay[i] /tmp
    NK_spectra[i] = NK_spectra[i] / tmp
    fitness_decay_terms_K = np.zeros(exponentials.shape)
    for j in range(exponentials.shape[1]):
        fitness_decay_terms_K[:, j] = exponentials[:, j] * NK_spectra[i][j]
    fitness_decay_terms.append(fitness_decay_terms_K)
    
    estim_data = get_single_decay_rate(fitness_decay[i], mut = mut, num_steps = num_steps)
    # fitted_decay.append(np.array([np.exp(-mutations*(estim_data[0]))*(1 - estim_data[-1]) + estim_data[-1]]))
    fitted_decay.append(np.array([np.exp(-mutations*(estim_data[0]))*(1 - estim_data[-1])]))
    fitted_rho.append(estim_data[0])

markers = ['o', 's', 'D', '^', 'v', '<', '>', 'x', '+', '*']

linestyles = ['-', '--', '-.', ':']

import matplotlib.gridspec as gridspec
fig = plt.figure(figsize=(18, 5))
gs = gridspec.GridSpec(2, 3, figure=fig)

axx = fig.add_subplot(gs[:, 0])
for i, nk in enumerate(K_vec):
    line, = axx.plot(
        range(N + 1), NK_spectra[i],
        label=f"$K={nk}$",
        marker=markers[i],
        markersize=6,
        linewidth=1.5
    )
    color = line.get_color()
    axx.axvline(NK_max[i], color=color, linestyle=':')
    axx.axvline(fitted_rho[i]*N*(A-1)/2/A, color=color, linestyle='--')
    # axx.axvline(fitted_rho_NEW[i]*N*(A-1)/2/A, color=color, linestyle='-.')
    line, = axx.plot(
        range(N + 1), NK_spectra[i],
        color=color,
        markersize=6,
        linewidth=1.5
    )
axx.legend()
axx.set_xlim([0, N])
axx.set_ylim([0, 1])
axx.set_xlabel("Frequency index $i$ (-)")
axx.set_ylabel(f"Coefficients $b_i$ (-)")
axx.set_title(f"Power spectra (N={N}, A={A})", fontsize=10, fontweight='bold')
axx.text(
    -0.1, 1, "a",            # x, y in axes fraction
    transform=axx.transAxes,
    fontsize=20,
    va='bottom', ha='right'
)

# middle plot
for axi, (sel, ii) in enumerate(zip([1,3],[2,5])):
    axx = axx = fig.add_subplot(gs[axi, 1])
    
    axx.plot(
        mutations, fitness_decay[sel]-fitness_decay_terms[sel][0, 0],
        label=f"$G_{{\\mu}}-b_0$", color='k', linestyle='-', markersize=6, linewidth=2.2,
    )
    axx.plot(
        mutations, fitted_decay[sel][0,:],
        label=f"$G_{{\\mu,\\rho_2}}-c$", color='black', linestyle=':', markersize=6, linewidth=2.2,
    )
    for j, (ls, mk) in zip(range(exponentials.shape[1]), itertools.product(linestyles, markers)):
        if j>0 and NK_spectra[sel][j]>1e-4:
            if j == ii:
                axx.plot(
                    mutations,
                    fitness_decay_terms[sel][:, j], # + fitness_decay[sel] - fitness_decay_terms[sel][0, j],
                    label=f"$b_{{{j}}}\\mathrm{{e}}^{{\\frac{{-2\\mu\\lambda_{i}}}{{d}}}}$", 
                    linestyle='-', 
                    marker=mk, markersize=6, linewidth=2.2,
                )
            else:
                axx.plot(
                    mutations,
                    fitness_decay_terms[sel][:, j], # + fitness_decay[sel] - fitness_decay_terms[sel][0, j],
                    #label=f"$b_{{{j}}}\\mathrm{{e}}^{{\\frac{{-2\\mu\\lambda_{i}}}{{d}}}}$", 
                    linestyle='--', 
                    marker=mk, markersize=4, linewidth=1.5,
                )
    axx.legend(ncol=1)
    axx.set_title(f"Fitness decay ($K={K_vec[sel]}$)", fontsize=10, fontweight='bold')
    axx.set_xlabel(f"Mutations $\\mu$ (-)", )
    axx.set_ylabel(f"Decay (-)" )
    axx.set_xlim([0,mutations.max()])
    axx.set_ylim([0,(fitness_decay[sel]-fitness_decay_terms[sel][0, 0]).max()])
    
    letters = ["b", "c"]
    axx.text(
        -0.1, 1, letters[axi],
        transform=axx.transAxes,
        fontsize=20,
        va='bottom', ha='right'
    )
    
# right plot
for i, sel in enumerate([1,3]):
    decay = fitness_decay[sel]
    np.random.seed(2122)
    dnoise = 0.05
    noise = np.random.uniform(low=-dnoise,high=dnoise,size=fitness_decay[sel].shape)
    decay_noisy = fitness_decay[sel] + noise

    spectrum, _ = get_fourier_coeffs(mean_fitness=decay, mutations=mutations, N=N, A=A, is_squared=True, method="ls_constrained", fix_b0=True)
    spectrum_noisy, _ = get_fourier_coeffs(mean_fitness=decay_noisy, mutations=mutations, N=N, A=A, is_squared=True, method="nnls", fix_b0=True)
    spectrum_reg, _ = get_fourier_coeffs(mean_fitness=decay_noisy, mutations=mutations, N=N, A=A, is_squared=True, method="nnls_reg",alpha=1e-3, fix_b0=True)

    axx = fig.add_subplot(gs[i, 2])
    axx.plot(range(N + 1), NK_spectra[sel], label=f"True", markersize=7, linewidth=2, marker=markers[0])
    axx.plot(range(N + 1), spectrum, label=f"Estimated", markersize=5, linewidth=1.5, marker=markers[1], linestyle="--")
    axx.plot(range(N + 1), spectrum_noisy, label=f"Noisy", markersize=6, linewidth=1.5, marker=markers[2], linestyle="-")
    axx.plot(range(N + 1), spectrum_reg, label=f"Regularised", markersize=6, linewidth=1.5, marker=markers[3], linestyle=":")
    axx.legend()
    axx.set_xlim([0, N])
    axx.set_ylim([0, max(1,spectrum_noisy.max())])
    axx.set_xlabel("Frequency index $i$ (-)")
    axx.set_ylabel(f"Coefficients $b_i$ (-)")
    axx.set_title(f"Spectrum estimation ($K={K_vec[sel]}$)", fontsize=10, fontweight='bold')
    
    letters = ["d", "e"]
    axx.text(
        -0.1, 1, letters[i],
        transform=axx.transAxes,
        fontsize=20,
        va='bottom', ha='right'
    )
    
fig.subplots_adjust(hspace=0.5, wspace=0.25)
# plt.savefig('figures/NK_spectra.pdf', dpi=dpi)

## Basis Function Plots

In [ ]:
# Updated: add base grid at z=BASE_Z with major (step=1) and optional minor lines.
# Keeps ordering + drop lines + smooth overlay. Base plane optional.


## Fontsizes

titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"
c2 = 'tab:blue'#298c8c'
c1 = 'tab:orange' #800074'
c3 = '#f55f74'
c4 = 'tab:green'

# Optional JAX acceleration
try:
    xp = jnp
except Exception:
    xp = np

N = 4
fine_M = 101           # smooth surface sampling
BASE_Z = 0.0           # base plane/grid height
SHOW_BASE_PLANE = False
SHOW_BASE_GRID = True
GRID_MAJOR_STEP = 1.0  # draw lines every 1.0 unit (integer grid)
GRID_MINOR_STEP = 0.5  # set to None to disable minor grid
GRID_COLOR_MAJOR = "0.35"
GRID_COLOR_MINOR = "0.65"
GRID_LW_MAJOR = 1.1
GRID_LW_MINOR = 0.6
GRID_ALPHA_MAJOR = 0.6
GRID_ALPHA_MINOR = 0.35

# --- grids ---
xs = np.arange(N)
X1d, X2d = np.meshgrid(xs, xs, indexing="ij")
t = np.linspace(0, N, fine_M, endpoint=False)
X1s, X2s = np.meshgrid(t, t, indexing="ij")

def theta(k1, k2, X1, X2):
    return (2 * xp.pi / N) * (k1 * X1 + k2 * X2)

def freq_mag(k):
    return int(min(k % N, (-k) % N))

SELF = {(0,0), (N//2,0), (0,N//2), (N//2,N//2)}

def is_rep(k1,k2):
    k1m, k2m = (-k1) % N, (-k2) % N
    if (k1,k2) == (k1m,k2m):
        return True
    return (k1,k2) < (k1m,k2m)

items = []

# 1) self-conjugate cos modes
for k in [(0,0),(2,0),(0,2),(2,2)]:
    k1,k2 = k
    T_d = theta(k1,k2, xp.asarray(X1d), xp.asarray(X2d))
    T_s = theta(k1,k2, xp.asarray(X1s), xp.asarray(X2s))
    fd = xp.cos(T_d); fs = xp.cos(T_s)
    items.append(dict(
        k=(k1,k2), xmag=freq_mag(k1), ymag=freq_mag(k2),
        kind="cos", fd=np.array(fd), fs=np.array(fs),
        const_x=(freq_mag(k1)==0), const_y=(freq_mag(k2)==0)
    ))

# 2) paired reps: √2·cos and √2·sin
for k1 in range(N):
    for k2 in range(N):
        if (k1,k2) in SELF: 
            continue
        if not is_rep(k1,k2):
            continue
        T_d = theta(k1,k2, xp.asarray(X1d), xp.asarray(X2d))
        T_s = theta(k1,k2, xp.asarray(X1s), xp.asarray(X2s))
        for kind, trig in [("cos", xp.cos), ("sin", xp.sin)]:
            fd = xp.sqrt(2.0) * trig(T_d)
            fs = xp.sqrt(2.0) * trig(T_s)
            items.append(dict(
                k=(k1,k2), xmag=freq_mag(k1), ymag=freq_mag(k2),
                kind=kind, fd=np.array(fd), fs=np.array(fs),
                const_x=(freq_mag(k1)==0), const_y=(freq_mag(k2)==0)
            ))

assert len(items) == 16

# --- Construct ordered 4x4 grid with constraints -----------------------------
def srt(it): return (it["xmag"], 0 if it["kind"]=="cos" else 1, it["k"])

rows = {0:[],1:[],2:[]}
for it in items:
    rows[it["ymag"]].append(it)

def take_first(pool, predicate, sort_key):
    cand = [it for it in pool if predicate(it)]
    cand.sort(key=sort_key)
    if not cand:
        return None
    pick = cand[0]
    pool.remove(pick)
    return pick

pool0, pool1, pool2 = rows[0][:], rows[1][:], rows[2][:]
pool0.sort(key=srt); pool1.sort(key=srt); pool2.sort(key=srt)

grid = [[None]*4 for _ in range(4)]
# Row 0 (ky=0): kx=0,1cos,1sin,2
grid[0][0] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==0, srt)
grid[0][1] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==1 and it["kind"]=="cos", srt)
grid[0][2] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==1 and it["kind"]=="sin", srt)
grid[0][3] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==2, srt)

# Rows 1 & 2 (|ky|=1): left col constant in x (kx=0), cos then sin
grid[1][0] = take_first(pool1, lambda it: it["const_x"] and it["kind"]=="cos", srt)
grid[2][0] = take_first(pool1, lambda it: it["const_x"] and it["kind"]=="sin", srt)
for c in [1,2,3]:
    grid[1][c] = take_first(pool1, lambda it: not it["const_x"], srt)
for c in [1,2,3]:
    grid[2][c] = take_first(pool1, lambda it: not it["const_x"], srt)

# Row 3 (|ky|=2): left col kx=0, then xmag=1 (cos,sin), then xmag=2
grid[3][0] = take_first(pool2, lambda it: it["const_x"], srt)
grid[3][1] = take_first(pool2, lambda it: it["xmag"]==1 and it["kind"]=="cos", srt)
grid[3][2] = take_first(pool2, lambda it: it["xmag"]==1 and it["kind"]=="sin", srt)
grid[3][3] = take_first(pool2, lambda it: it["xmag"]==2, srt)

ordered = [grid[r][c] for r in range(4) for c in range(4)]
assert all(it is not None for it in ordered)

# --- Orthonormality check -----------------------------------------------------
B = np.stack([it["fd"].ravel() for it in ordered], axis=1)
G = (B.T @ B) / (N*N)
print("Orthonormal (max off-diag):", float(np.max(np.abs(G - np.eye(16)))))

# --- Manual permutation hook --------------------------------------------------
PERM = list(range(16))  # edit this to re-order panels
ordered = [ordered[i] for i in PERM]



print("\nPanel index → label (before PERM):")
SELF = {(0,0),(2,0),(0,2),(2,2)}
for idx, it in enumerate([grid[r][c] for r in range(4) for c in range(4)]):
    k1,k2 = it["k"]
    tag = f"{'√2·' if (k1,k2) not in SELF else ''}{it['kind']}[{k1},{k2}]"
    print(f"{idx:2d}: {tag}  (|kx|={it['xmag']}, |ky|={it['ymag']})")

# --- Helpers ------------------------------------------------------------------
def draw_base_grid(ax, N, z=0.0, major_step=1.0, minor_step=0.5):
    """Draw a 2D grid on plane z at integer coordinates (and optional minors)."""
    # Major lines
    vals = np.arange(0, N+1, major_step)
    for xi in vals:
        ax.plot([xi, xi], [0, N], [z, z], color=GRID_COLOR_MAJOR,
                linewidth=GRID_LW_MAJOR, alpha=GRID_ALPHA_MAJOR)
    for yi in vals:
        ax.plot([0, N], [yi, yi], [z, z], color=GRID_COLOR_MAJOR,
                linewidth=GRID_LW_MAJOR, alpha=GRID_ALPHA_MAJOR)
    # Minor lines
    if minor_step and minor_step > 0 and minor_step < major_step:
        vals_minor = np.arange(0, N+1, minor_step)
        # remove majors to avoid double-draw
        majors = set(np.round(vals, 8).tolist())
        for xi in vals_minor:
            if np.round(xi,8) in majors: 
                continue
            ax.plot([xi, xi], [0, N], [z, z], color=GRID_COLOR_MINOR,
                    linewidth=GRID_LW_MINOR, alpha=GRID_ALPHA_MINOR)
        for yi in vals_minor:
            if np.round(yi,8) in majors:
                continue
            ax.plot([0, N], [yi, yi], [z, z], color=GRID_COLOR_MINOR,
                    linewidth=GRID_LW_MINOR, alpha=GRID_ALPHA_MINOR)

# --- Plot ---------------------------------------------------------------------
fig = plt.figure(figsize=(18, 12))
for i, it in enumerate(ordered, start=1):
    ax = fig.add_subplot(4, 4, i, projection='3d')
    # smooth surface
    from matplotlib import cm
    from matplotlib.colors import Normalize

    norm = Normalize(vmin=it["fs"].min(), vmax=it["fs"].max())
    colors = cm.viridis(norm(it["fs"]))

    ax.plot_surface(X1s, X2s, it["fs"],
                facecolors=colors,
                linewidth=0, antialiased=True, alpha=0.4)
    # optional base plane (very light)
    if False:
        ax.plot_surface(X1s, X2s, np.full_like(X1s, BASE_Z), linewidth=0, alpha=0.08)
    # discrete points as crosses
    x = X1d.ravel()
    y = X2d.ravel()
    z = it["fd"].ravel()
    ax.scatter(x, y, z, marker='x', s=50, depthshade=False, linewidths=0.9, color  = "grey")
    # vertical drop lines to BASE_Z
    for xi, yi, zi in zip(x, y, z):
        ax.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color='grey')
    # base grid
    if SHOW_BASE_GRID:
        draw_base_grid(ax, N, z=BASE_Z, major_step=GRID_MAJOR_STEP, minor_step=GRID_MINOR_STEP)
    # title
    k1,k2 = it["k"]
    print(f"Panel {i-1:2d}: k=({k1},{k2}), kind={it['kind']}, |kx|={it['xmag']}, |ky|={it['ymag']}")
    eigenvalue = 4* (k1 > 0) + 4* (k2 > 0)  # Laplacian eigenvalue
    title = f"{'√2·' if (k1,k2) not in SELF else ''}{it['kind']}[{k1},{k2}]   (|kx|={it['xmag']}, |ky|={it['ymag']})\nEigenvalue: {eigenvalue}"
    ax.set_title(title, fontsize=9)
    ax.set_xticks(range(N)); ax.set_yticks(range(N)); ax.set_zticks([-1, 0, 1])
    ax.set_xlabel("x₁"); ax.set_ylabel("x₂")

fig.suptitle("Real Orthonormal Fourier Basis (N=4)\nDiscrete 'X' samples + drop lines + smooth overlay + base grid", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
def smooth_func(x, y):
    return jnp.sin(y / 3)

def bump_func(x, y):
    return -jnp.sin(x * jnp.pi / 2 + y * jnp.pi / 2) * jnp.cos(y * jnp.pi / 4)

smooth_func_s = smooth_func(X1s, X2s)
smooth_func_d = smooth_func(X1d, X2d)
bump_func_s = bump_func(X1s, X2s)
bump_func_d = bump_func(X1d, X2d)

In [ ]:

# ---- Style settings ----
titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"

cmap = plt.get_cmap("viridis")

# ---------- First plot ----------
fig1 = plt.figure(figsize=(3,3), dpi=300, constrained_layout=True)
ax1 = fig1.add_subplot(111, projection='3d')

# Normalize color scale
zmin = min(smooth_func_d.min(), smooth_func_s.min())
zmax = max(smooth_func_d.max(), smooth_func_s.max())
norm = plt.Normalize(zmin, zmax)

ax1.plot_surface(
    X1s, X2s, smooth_func_s,
    linewidth=0, antialiased=True, alpha=1,
    cmap=cmap, norm=norm
)

# ax1.scatter(
#     X1d, X2d, smooth_func_d,
#     marker='x', s=50, depthshade=False,
#     linewidths=1.5, color='gray'
# )

# for xi, yi, zi in zip(X1d.ravel(), X2d.ravel(), smooth_func_d.ravel()):
#     ax1.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color='gray')

# ---- Title & fonts ----
ax1.set_title("Smooth Landscape", fontsize=titlesize, y=1)  
# y < 1 moves it down toward the plot (default is ~1.0)

# ---- Axis label sizes ----
ax1.set_xlabel(r"$\sigma_1$", fontsize=labelsize, labelpad=2)
ax1.set_ylabel(r"$\sigma_2$", fontsize=labelsize, labelpad=2)
ax1.set_zlabel("Arbitrary Fitness", fontsize=labelsize)

# Adjust z-label position so it stays inside the figure
ax1.zaxis.labelpad = 15  # more space for z-label

# ---- Tick label sizes ----
ax1.tick_params(axis='both', which='major', labelsize=ticksize)
ax1.tick_params(axis='both', which='minor', labelsize=ticksize)

from matplotlib.ticker import MaxNLocator

# Force integer ticks on all axes
ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
ax1.yaxis.set_major_locator(MaxNLocator(integer=True))
ax1.zaxis.set_major_locator(MaxNLocator(integer=True))

# plt.savefig('figures/smooth_landscape_3D.pdf', dpi=dpi)



In [ ]:

# ---- Style settings ----
titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"

cmap = plt.get_cmap("viridis")

# ---------- Second plot ----------
fig2 = plt.figure(figsize=(3, 3), dpi=300, constrained_layout=True)
ax2 = fig2.add_subplot(111, projection='3d')

# Normalize color scale
zmin = min(bump_func_d.min(), bump_func_s.min())
zmax = max(bump_func_d.max(), bump_func_s.max())
norm = plt.Normalize(zmin, zmax)

ax2.plot_surface(
    X1s, X2s, bump_func_s,
    linewidth=0, antialiased=True, alpha=1,   # match first plot’s alpha
    cmap=cmap, norm=norm
)

# ax2.scatter(
#     X1d, X2d, bump_func_d,
#     marker='x', s=50, depthshade=False,
#     linewidths=1.5, color='gray'
# )

# for xi, yi, zi in zip(X1d.ravel(), X2d.ravel(), bump_func_d.ravel()):
#     ax2.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color='gray')

# ---- Title & fonts ----
ax2.set_title("Rugged Landscape", fontsize=titlesize, y=1)  

# ---- Axis label sizes ----
ax2.set_xlabel(r"$\sigma_1$", fontsize=labelsize, labelpad=2)
ax2.set_ylabel(r"$\sigma_2$", fontsize=labelsize, labelpad=2)
ax2.set_zlabel("Arbitrary Fitness", fontsize=labelsize)

# Adjust z-label position so it stays inside the figure
ax2.zaxis.labelpad = 15

# ---- Tick label sizes ----
ax2.tick_params(axis='both', which='major', labelsize=ticksize)
ax2.tick_params(axis='both', which='minor', labelsize=ticksize)

# Force integer ticks on all axes
ax2.xaxis.set_major_locator(MaxNLocator(integer=True))
ax2.yaxis.set_major_locator(MaxNLocator(integer=True))

# plt.savefig('figures/rugged_landscape_3D.pdf', dpi=dpi)


In [ ]:

## Fontsizes

titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"
c2 = 'tab:blue'#298c8c'
c1 = 'tab:orange' #800074'
c3 = '#f55f74'
c4 = 'tab:green'

# Optional JAX acceleration
try:
    xp = jnp
except Exception:
    xp = np

N = 4
fine_M = 101           # smooth surface sampling
BASE_Z = 0.0           # base plane/grid height
SHOW_BASE_PLANE = False
SHOW_BASE_GRID = True
GRID_MAJOR_STEP = 1.0  # draw lines every 1.0 unit (integer grid)
GRID_MINOR_STEP = 0.5  # set to None to disable minor grid
GRID_COLOR_MAJOR = "0.35"
GRID_COLOR_MINOR = "0.65"
GRID_LW_MAJOR = 1.1
GRID_LW_MINOR = 0.6
GRID_ALPHA_MAJOR = 0.6
GRID_ALPHA_MINOR = 0.35

# --- grids ---
xs = np.arange(N)
X1d, X2d = np.meshgrid(xs, xs, indexing="ij")
t = np.linspace(0, N, fine_M, endpoint=False)
X1s, X2s = np.meshgrid(t, t, indexing="ij")

def theta(k1, k2, X1, X2):
    return (2 * xp.pi / N) * (k1 * X1 + k2 * X2)

def freq_mag(k):
    return int(min(k % N, (-k) % N))

SELF = {(0,0), (N//2,0), (0,N//2), (N//2,N//2)}

def is_rep(k1,k2):
    k1m, k2m = (-k1) % N, (-k2) % N
    if (k1,k2) == (k1m,k2m):
        return True
    return (k1,k2) < (k1m,k2m)

items = []

# 1) self-conjugate cos modes
for k in [(0,0),(2,0),(0,2),(2,2)]:
    k1,k2 = k
    T_d = theta(k1,k2, xp.asarray(X1d), xp.asarray(X2d))
    T_s = theta(k1,k2, xp.asarray(X1s), xp.asarray(X2s))
    fd = xp.cos(T_d); fs = xp.cos(T_s)
    items.append(dict(
        k=(k1,k2), xmag=freq_mag(k1), ymag=freq_mag(k2),
        kind="cos", fd=np.array(fd), fs=np.array(fs),
        const_x=(freq_mag(k1)==0), const_y=(freq_mag(k2)==0)
    ))

# 2) paired reps: √2·cos and √2·sin
for k1 in range(N):
    for k2 in range(N):
        if (k1,k2) in SELF: 
            continue
        if not is_rep(k1,k2):
            continue
        T_d = theta(k1,k2, xp.asarray(X1d), xp.asarray(X2d))
        T_s = theta(k1,k2, xp.asarray(X1s), xp.asarray(X2s))
        for kind, trig in [("cos", xp.cos), ("sin", xp.sin)]:
            fd = xp.sqrt(2.0) * trig(T_d)
            fs = xp.sqrt(2.0) * trig(T_s)
            items.append(dict(
                k=(k1,k2), xmag=freq_mag(k1), ymag=freq_mag(k2),
                kind=kind, fd=np.array(fd), fs=np.array(fs),
                const_x=(freq_mag(k1)==0), const_y=(freq_mag(k2)==0)
            ))

assert len(items) == 16

# --- Construct ordered 4x4 grid with constraints -----------------------------
def srt(it): return (it["xmag"], 0 if it["kind"]=="cos" else 1, it["k"])

rows = {0:[],1:[],2:[]}
for it in items:
    rows[it["ymag"]].append(it)

def take_first(pool, predicate, sort_key):
    cand = [it for it in pool if predicate(it)]
    cand.sort(key=sort_key)
    if not cand:
        return None
    pick = cand[0]
    pool.remove(pick)
    return pick

pool0, pool1, pool2 = rows[0][:], rows[1][:], rows[2][:]
pool0.sort(key=srt); pool1.sort(key=srt); pool2.sort(key=srt)

grid = [[None]*4 for _ in range(4)]
# Row 0 (ky=0): kx=0,1cos,1sin,2
grid[0][0] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==0, srt)
grid[0][1] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==1 and it["kind"]=="cos", srt)
grid[0][2] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==1 and it["kind"]=="sin", srt)
grid[0][3] = take_first(pool0, lambda it: it["const_y"] and it["xmag"]==2, srt)

# Rows 1 & 2 (|ky|=1): left col constant in x (kx=0), cos then sin
grid[1][0] = take_first(pool1, lambda it: it["const_x"] and it["kind"]=="cos", srt)
grid[2][0] = take_first(pool1, lambda it: it["const_x"] and it["kind"]=="sin", srt)
for c in [1,2,3]:
    grid[1][c] = take_first(pool1, lambda it: not it["const_x"], srt)
for c in [1,2,3]:
    grid[2][c] = take_first(pool1, lambda it: not it["const_x"], srt)

# Row 3 (|ky|=2): left col kx=0, then xmag=1 (cos,sin), then xmag=2
grid[3][0] = take_first(pool2, lambda it: it["const_x"], srt)
grid[3][1] = take_first(pool2, lambda it: it["xmag"]==1 and it["kind"]=="cos", srt)
grid[3][2] = take_first(pool2, lambda it: it["xmag"]==1 and it["kind"]=="sin", srt)
grid[3][3] = take_first(pool2, lambda it: it["xmag"]==2, srt)

ordered = [grid[r][c] for r in range(4) for c in range(4)]
assert all(it is not None for it in ordered)

# --- Orthonormality check -----------------------------------------------------
B = np.stack([it["fd"].ravel() for it in ordered], axis=1)
G = (B.T @ B) / (N*N)
print("Orthonormal (max off-diag):", float(np.max(np.abs(G - np.eye(16)))))

# --- Manual permutation hook --------------------------------------------------
PERM = list(range(16))  # edit this to re-order panels
ordered = [ordered[i] for i in PERM]

print("\nPanel index → label (before PERM):")
SELF = {(0,0),(2,0),(0,2),(2,2)}
for idx, it in enumerate([grid[r][c] for r in range(4) for c in range(4)]):
    k1,k2 = it["k"]
    tag = f"{'√2·' if (k1,k2) not in SELF else ''}{it['kind']}[{k1},{k2}]"
    #print(f"{idx:2d}: {tag}  (|kx|={it['xmag']}, |ky|={it['ymag']})")

# --- Helpers ------------------------------------------------------------------
def draw_base_grid(ax, N, z=0.0, major_step=1.0, minor_step=0.5):
    """Draw a 2D grid on plane z at integer coordinates (and optional minors)."""
    # Major lines
    vals = np.arange(0, N+1, major_step)
    for xi in vals:
        ax.plot([xi, xi], [0, N], [z, z], color=GRID_COLOR_MAJOR,
                linewidth=GRID_LW_MAJOR, alpha=GRID_ALPHA_MAJOR)
    for yi in vals:
        ax.plot([0, N], [yi, yi], [z, z], color=GRID_COLOR_MAJOR,
                linewidth=GRID_LW_MAJOR, alpha=GRID_ALPHA_MAJOR)
    # Minor lines
    if minor_step and minor_step > 0 and minor_step < major_step:
        vals_minor = np.arange(0, N+1, minor_step)
        # remove majors to avoid double-draw
        majors = set(np.round(vals, 8).tolist())
        for xi in vals_minor:
            if np.round(xi,8) in majors: 
                continue
            ax.plot([xi, xi], [0, N], [z, z], color=GRID_COLOR_MINOR,
                    linewidth=GRID_LW_MINOR, alpha=GRID_ALPHA_MINOR)
        for yi in vals_minor:
            if np.round(yi,8) in majors:
                continue
            ax.plot([0, N], [yi, yi], [z, z], color=GRID_COLOR_MINOR,
                    linewidth=GRID_LW_MINOR, alpha=GRID_ALPHA_MINOR)

# --- Plot ---------------------------------------------------------------------
fig = plt.figure(figsize=(8,8))
for i, it in enumerate(ordered, start=1):
    ax = fig.add_subplot(4, 4, i, projection='3d')
    # smooth surface
    from matplotlib import cm
    from matplotlib.colors import Normalize

    norm = Normalize(vmin=it["fs"].min(), vmax=it["fs"].max())
    colors = cm.viridis(norm(it["fs"]))

    ax.plot_surface(X1s, X2s, it["fs"],
                facecolors=colors,
                linewidth=0, antialiased=True, alpha=0.8)
    # optional base plane (very light)
    if False:
        ax.plot_surface(X1s, X2s, np.full_like(X1s, BASE_Z), linewidth=0, alpha=0.08)
    # # discrete points as crosses
    # x = X1d.ravel()
    # y = X2d.ravel()
    # z = it["fd"].ravel()
    # ax.scatter(x, y, z, marker='x', s=50, depthshade=False, linewidths=0.9, color  = "grey")
    # # vertical drop lines to BASE_Z
    # for xi, yi, zi in zip(x, y, z):
    #     ax.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color='grey')
    # base grid
    # if SHOW_BASE_GRID:
    #     draw_base_grid(ax, N, z=BASE_Z, major_step=GRID_MAJOR_STEP, minor_step=GRID_MINOR_STEP)
    # title
    k1,k2 = it["k"]
    #print(f"Panel {i-1:2d}: k=({k1},{k2}), kind={it['kind']}, |kx|={it['xmag']}, |ky|={it['ymag']}")
    eigenvalue = 4* (k1 > 0) + 4* (k2 > 0)  # Laplacian eigenvalue
    #title = f"{'√2·' if (k1,k2) not in SELF else ''}{it['kind']}[{k1},{k2}]   (|kx|={it['xmag']}, |ky|={it['ymag']})\nEigenvalue: {eigenvalue}"
    #title = f"{'√2·' if (k1,k2) not in SELF else ''}{it['kind']}[{k1},{k2}], $\lambda$ = {eigenvalue}"
    title = f"$\lambda$ = {eigenvalue}"
    ax.set_title(title, fontsize=titlesize, y=0.99)  # smaller y brings it lower
    ax.set_xticks(range(N)); ax.set_yticks(range(N)); ax.set_zticks([-1, 0, 1])
    #ax.set_xlabel("x₁"); ax.set_ylabel("x₂")

        # Control tick label font size
    ax.tick_params(axis="both", which="major", labelsize=ticksize)
    ax.tick_params(axis="both", which="minor", labelsize=ticksize)
    ax.zaxis.set_tick_params(labelsize=ticksize)
#fig.suptitle("Real Orthonormal Fourier Basis (N=4)\nDiscrete 'X' samples + drop lines + smooth overlay + base grid", fontsize=14)
#fig.suptitle("Basis Vectors", fontsize=titlesize+3)
plt.tight_layout()
# plt.savefig('figures/basis_vectors.pdf', dpi=dpi)


In [ ]:
# --- Fourier-space projection + plotting -------------------------------------

def project_to_fourier_coeffs(f_d, ordered, N):
    """
    Project a discrete function f_d (shape N×N) onto your real orthonormal basis
    defined by `ordered`. Returns a 4×4 array C whose (row, col) matches the
    panel order you're using (after PERM).
    """
    fvec = np.asarray(f_d, dtype=float).ravel()
    B = np.stack([it["fd"].ravel() for it in ordered], axis=1)  # (N^2 × 16)
    # Your basis columns are orthonormal w.r.t. (1/N^2) <.,.>, so:
    coeffs = (B.T @ fvec) / (N * N)  # (16,)
    C = coeffs.reshape(4, 4)         # row-major matches your panel order
    return C

def bilinear_surface_from_grid(C, Kx, Ky):
    """
    Bilinear interpolation over a 4×4 grid C onto fine frequency grid (Kx,Ky),
    with Kx,Ky in [0,3] along columns/rows respectively.
    Interpolates exactly at integer grid points.
    """
    # Clamp into valid cell range
    u = np.clip(Kx, 0.0, 3.0)
    v = np.clip(Ky, 0.0, 3.0)

    i0 = np.floor(v).astype(int)
    j0 = np.floor(u).astype(int)
    i1 = np.clip(i0 + 1, 0, 3)
    j1 = np.clip(j0 + 1, 0, 3)
    du = u - j0
    dv = v - i0

    # gather corners
    C00 = C[i0, j0]
    C10 = C[i1, j0]
    C01 = C[i0, j1]
    C11 = C[i1, j1]

    # bilinear blend
    S = ( (1 - du) * (1 - dv) * C00
        + (    du) * (1 - dv) * C01
        + (1 - du) * (    dv) * C10
        + (    du) * (    dv) * C11 )
    return S

def make_kspace_grids(fine_M=201):
    """
    Frequency-plane fine grid: continuous 'panel coordinates'.
    We use [0,3] because there are 4 columns (kx panels) and 4 rows (ky panels).
    """
    t = np.linspace(0.0, 3.0, fine_M)
    Kx, Ky = np.meshgrid(t, t, indexing="xy")
    return Kx, Ky

def panel_labels_from_ordered(ordered):
    """
    Build tick labels for the Fourier-space axes from the basis panels.
    (kx, ky) shown as the *actual* mode labels, respecting your ordering.
    """
    # ordered is row-major 4×4
    labels = [[None]*4 for _ in range(4)]
    idx = 0
    for r in range(4):
        for c in range(4):
            it = ordered[idx]
            k1, k2 = it["k"]
            lab = f"{'√2·' if (k1,k2) not in {(0,0),(2,0),(0,2),(2,2)} else ''}{it['kind']}[{k1},{k2}]"
            labels[r][c] = lab
            idx += 1
    return labels

def plot_fourier_space(C, title="Fourier-space coefficients (4×4)", 
                       show_base_grid=True, show_base_plane=False, base_z=0.0):
    """
    Plot a smooth Fourier-space surface (bilinear interp) over panel coordinates,
    with grey × crosses at the exact 4×4 coefficients.
    """
    from matplotlib import cm
    from matplotlib.colors import Normalize

    Kx, Ky = make_kspace_grids(fine_M=fine_M)  # fine grid in panel coordinates
    Zs = bilinear_surface_from_grid(C, Kx, Ky)

    fig = plt.figure(figsize=(7.5, 6.5))
    ax = fig.add_subplot(1, 1, 1, projection="3d")

    # Smooth surface colors
    norm = Normalize(vmin=Zs.min(), vmax=Zs.max())
    colors = cm.viridis(norm(Zs))

    # Plot smooth surface
    ax.plot_surface(Kx, Ky, Zs, facecolors=colors, linewidth=0, antialiased=True, alpha=0.4)

    # Optional base plane
    if show_base_plane:
        ax.plot_surface(Kx, Ky, np.full_like(Kx, base_z), linewidth=0, alpha=0.08)

    # Discrete coefficient crosses at integer panel coordinates
    xs = np.arange(4)
    ys = np.arange(4)
    Xd, Yd = np.meshgrid(xs, ys, indexing="xy")  # (y,row), (x,col)
    Zd = C
    ax.scatter(Xd.ravel(), Yd.ravel(), Zd.ravel(), marker='x', s=60, depthshade=False,
               linewidths=1.1, color="grey")

    # Drop lines
    for xi, yi, zi in zip(Xd.ravel(), Yd.ravel(), Zd.ravel()):
        ax.plot([xi, xi], [yi, yi], [base_z, zi], linewidth=2.0, alpha=0.9, color='grey')

    # Base grid (on Fourier panel coords)
    if show_base_grid:
        # reuse your draw_base_grid but with N=4 and plane coords
        draw_base_grid(ax, N=4, z=base_z, major_step=1.0, minor_step=0.5)

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("panel kx (columns)")
    ax.set_ylabel("panel ky (rows)")
    ax.set_zticks(np.linspace((Zd.min()), (Zd.max()), 5))

    # nice ticks exactly at panel integers
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    plt.tight_layout()
    plt.show()

In [ ]:
from matplotlib import cm
from matplotlib.colors import Normalize

# ---- Style settings ----
titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"

# --- Fourier coefficients for smooth function ---
smooth_func_d = np.array(smooth_func(X1d, X2d))
C_smooth = project_to_fourier_coeffs(smooth_func_d, ordered, N)

# --- Fine Fourier grid ---
Kx, Ky = make_kspace_grids(fine_M=201)
Zs = bilinear_surface_from_grid(C_smooth, Kx, Ky)

# --- Plot ---
fig1 = plt.figure(figsize=(3, 3), dpi=300, constrained_layout=True)
ax1 = fig1.add_subplot(111, projection="3d")

norm = Normalize(vmin=Zs.min(), vmax=Zs.max())
colors = cm.viridis(norm(Zs))

# Smooth surface
ax1.plot_surface(Kx, Ky, Zs, facecolors=colors, linewidth=0, antialiased=True, alpha=1)

# # Coefficient crosses
# xs = np.arange(4)
# ys = np.arange(4)
# Xd, Yd = np.meshgrid(xs, ys, indexing="xy")
# Zd = C_smooth
# ax1.scatter(Xd.ravel(), Yd.ravel(), Zd.ravel(), marker='x', s=60,
#             depthshade=False, linewidths=1.1, color="gray")

# # Drop lines
# for xi, yi, zi in zip(Xd.ravel(), Yd.ravel(), Zd.ravel()):
#     ax1.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color="gray")

# Title & fonts
ax1.set_title("Smooth Landscape Fourier Space", fontsize=titlesize, y=1)

# Axis labels
ax1.set_xlabel(r"$\hat{\sigma}_1$", fontsize=labelsize, labelpad=2)
ax1.set_ylabel(r"$\hat{\sigma}_2$", fontsize=labelsize, labelpad=2)
ax1.set_zlabel("Coefficient value", fontsize=labelsize)
ax1.zaxis.labelpad = 15

# Ticks
ax1.set_xticks(range(4))
ax1.set_yticks(range(4))
ax1.tick_params(axis="both", which="major", labelsize=ticksize)
ax1.tick_params(axis="both", which="minor", labelsize=ticksize)

# plt.savefig('figures/smooth_fourier_3D.pdf', dpi=dpi)


In [ ]:
# --- Fourier coefficients for bump function ---
bump_func_d = np.array(bump_func(X1d, X2d))
C_bump = project_to_fourier_coeffs(bump_func_d, ordered, N)

# --- Fine Fourier grid ---
Kx, Ky = make_kspace_grids(fine_M=201)
Zs = bilinear_surface_from_grid(C_bump, Kx, Ky)

# --- Plot ---
fig2 = plt.figure(figsize=(3, 3), dpi=300, constrained_layout=True)
ax2 = fig2.add_subplot(111, projection="3d")

norm = Normalize(vmin=Zs.min(), vmax=Zs.max())
colors = cm.viridis(norm(Zs))

# Smooth surface
ax2.plot_surface(Kx, Ky, Zs, facecolors=colors, linewidth=0, antialiased=True, alpha=1)

# # Coefficient crosses
# xs = np.arange(4)
# ys = np.arange(4)
# Xd, Yd = np.meshgrid(xs, ys, indexing="xy")
# Zd = C_bump
# ax2.scatter(Xd.ravel(), Yd.ravel(), Zd.ravel(), marker='x', s=60,
#             depthshade=False, linewidths=1.1, color="gray")

# # Drop lines
# for xi, yi, zi in zip(Xd.ravel(), Yd.ravel(), Zd.ravel()):
#     ax2.plot([xi, xi], [yi, yi], [BASE_Z, zi], linewidth=2.0, alpha=0.9, color="gray")

# Title & fonts
ax2.set_title("Rugged Landscape Fourier Space", fontsize=titlesize, y=1)

# Axis labels
ax2.set_xlabel(r"$\hat{\sigma}_1$", fontsize=labelsize, labelpad=2)
ax2.set_ylabel(r"$\hat{\sigma}_2$", fontsize=labelsize, labelpad=2)
ax2.set_zlabel("Coefficient value", fontsize=labelsize)
ax2.zaxis.labelpad = 15

# Ticks
ax2.set_xticks(range(4))
ax2.set_yticks(range(4))
ax2.tick_params(axis="both", which="major", labelsize=ticksize)
ax2.tick_params(axis="both", which="minor", labelsize=ticksize)

# plt.savefig('figures/rugged_fourier_3D.pdf', dpi=dpi)


In [ ]:

# Styling parameters
titlesize = 10
labelsize = 8
ticksize = 6
legendsize = 8
plt.rcParams["font.family"] = "DejaVu Sans"
c1 = 'tab:orange'   # #800074
c2 = 'tab:blue'     # #298c8c
c3 = '#f55f74'
c4 = 'tab:green'

def compute_spectral_density(C, ordered):
    """
    Compute the spectral density across eigenspaces:
    - Constant: |kx|=0 AND |ky|=0 (DC term - no variation)
    - Linear: Exactly one of |kx|>0 OR |ky|>0 (varies in one dimension only)
    - Quadratic: Both |kx|>0 AND |ky|>0 (varies in both dimensions)
    
    Returns normalized energy distribution across these three subspaces.
    """
    # Flatten coefficients and get metadata
    coeffs = C.ravel()
    
    # Classify each coefficient by its eigenspace
    constant_energy = 0.0
    linear_energy = 0.0
    quadratic_energy = 0.0
    
    for i, it in enumerate(ordered):
        xmag, ymag = it["xmag"], it["ymag"]
        coeff_sq = coeffs[i]**2
        
        # Count how many dimensions have non-zero frequency
        nonzero_dims = (xmag > 0) + (ymag > 0)
        
        if nonzero_dims == 0:
            # Both kx=0 and ky=0: constant term
            constant_energy += coeff_sq
        elif nonzero_dims == 1:
            # Either kx≠0,ky=0 or kx=0,ky≠0: linear (varies in one dimension)
            linear_energy += coeff_sq
        elif nonzero_dims == 2:
            # Both kx≠0 and ky≠0: quadratic (varies in both dimensions)
            quadratic_energy += coeff_sq
    
    # Total energy and normalization
    total_energy = constant_energy + linear_energy + quadratic_energy
    
    if total_energy > 0:
        constant_norm = constant_energy / total_energy
        linear_norm = linear_energy / total_energy
        quadratic_norm = quadratic_energy / total_energy
    else:
        constant_norm = linear_norm = quadratic_norm = 0.0
    
    return {
        'constant': {'energy': constant_energy, 'normalized': constant_norm},
        'linear': {'energy': linear_energy, 'normalized': linear_norm},
        'quadratic': {'energy': quadratic_energy, 'normalized': quadratic_norm},
        'total_energy': total_energy
    }


def plot_normalized_energy(functions_dict, title="Normalized Energy Distribution"):
    """
    Plot only the normalized spectral energy distribution across eigenspaces.
    functions_dict: {'func_name': coefficients_matrix, ...}
    """
    #eigenspaces = ['Constant\n(DC)', 'Linear\n(Fundamental)', 'Quadratic\n(Harmonics)']
    eigenspaces = [0,1,2]
    colors = [c1, c2, c3, c4]

    func_names = list(functions_dict.keys())
    n_funcs = len(func_names)
    x_pos = np.arange(len(eigenspaces))
    width = 0.35

    # Compute spectral densities
    spectral_data = {}
    for name, C in functions_dict.items():
        spectral_data[name] = compute_spectral_density(C, ordered)

    fig, ax = plt.subplots(figsize=(3,4), dpi=300)

    # Plot normalized energy distribution
    for i, name in enumerate(func_names):
        data = spectral_data[name]
        normalized = [data['constant']['normalized'],
                      data['linear']['normalized'],
                      data['quadratic']['normalized']]
        offset = (i - (n_funcs-1)/2) * width / n_funcs
        bars = ax.bar(x_pos + offset, normalized, width/n_funcs,
                      label=name, alpha=0.8, color=colors[i % len(colors)])

        # Add percentage labels on bars
        for j, bar in enumerate(bars):
            height = bar.get_height()
            if height > 0.02:  # Only label if > 2%
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'{height*100:.1f}%', ha='center', va='bottom', fontsize=ticksize)

    # Axis labels and formatting
    ax.set_xlabel('Frequency index $i$', fontsize=labelsize)
    ax.set_ylabel('Power spectral coefficients $b_i$ (%)', fontsize=labelsize)
    ax.set_title(title, fontsize=titlesize)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(eigenspaces, fontsize=ticksize)
    ax.set_yticklabels(np.round(ax.get_yticks(),2), fontsize=ticksize)
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=legendsize)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    # plt.savefig('figures/power_spectra_bar_chart.pdf', dpi=dpi)
# Generate spectral density plots for our test functions
functions_to_analyze = {
    'Smooth Landscape': C_smooth,
    'Rugged Landscape': C_bump
}

plot_normalized_energy(functions_to_analyze, 
                     title="Power Spectra")
